In [1]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [2]:
from pathlib import Path

MYDRIVE = Path("/content/drive/MyDrive")

candidate_names = {
    "experiment-1-v2-together_FINAL",
    "deterministic-intent-translation",
}

print("=" * 100)
print("SEARCHING GOOGLE DRIVE FOR PRESERVED EXPERIMENT REPOSITORIES")
print("=" * 100)

found = []

for p in MYDRIVE.rglob("*"):
    if p.is_dir() and p.name in candidate_names:
        found.append(p)

if found:
    for i, p in enumerate(found, 1):
        try:
            files = sum(1 for x in p.rglob("*") if x.is_file())
        except Exception:
            files = "?"
        print(f"\n[{i}] {p}")
        print("    Files:", files)
else:
    print("\nNo expected repository directories found.")

print("\n" + "=" * 100)
print("SEARCH COMPLETE")
print("=" * 100)

SEARCHING GOOGLE DRIVE FOR PRESERVED EXPERIMENT REPOSITORIES

[1] /content/drive/MyDrive/PhD_Research/Deterministic_Intent_Translation/experiment-1-v2-together_FINAL
    Files: 629

SEARCH COMPLETE


In [3]:
from pathlib import Path
import shutil
import time

SOURCE = Path(
    "/content/drive/MyDrive/PhD_Research/"
    "Deterministic_Intent_Translation/"
    "experiment-1-v2-together_FINAL"
)

WORK = Path("/content/deterministic-intent-translation-journal")

print("=" * 100)
print("STEP A2 — CREATE JOURNAL EXPERIMENT WORKING COPY")
print("=" * 100)

assert SOURCE.exists(), f"Source not found: {SOURCE}"

if WORK.exists():
    raise FileExistsError(
        f"\nSTOP: Working directory already exists:\n{WORK}\n"
        "Do not overwrite it."
    )

source_files = sum(1 for p in SOURCE.rglob("*") if p.is_file())

print("\nSource:")
print(SOURCE)

print("\nDestination:")
print(WORK)

print("\nSource files:", source_files)
print("\nCopying...")

start = time.time()

shutil.copytree(SOURCE, WORK)

elapsed = time.time() - start

work_files = sum(1 for p in WORK.rglob("*") if p.is_file())

print("\n" + "-" * 100)
print("Source files :", source_files)
print("Copied files :", work_files)
print(f"Copy time    : {elapsed:.2f} seconds")

assert source_files == work_files, (
    f"STOP: File-count mismatch: "
    f"{source_files} source vs {work_files} copied"
)

print("\n✓ FILE COUNTS MATCH")
print("✓ GOOGLE DRIVE MASTER WAS NOT MODIFIED")
print("✓ JOURNAL WORKING COPY CREATED")
print("=" * 100)

STEP A2 — CREATE JOURNAL EXPERIMENT WORKING COPY

Source:
/content/drive/MyDrive/PhD_Research/Deterministic_Intent_Translation/experiment-1-v2-together_FINAL

Destination:
/content/deterministic-intent-translation-journal

Source files: 629

Copying...

----------------------------------------------------------------------------------------------------
Source files : 629
Copied files : 629
Copy time    : 152.31 seconds

✓ FILE COUNTS MATCH
✓ GOOGLE DRIVE MASTER WAS NOT MODIFIED
✓ JOURNAL WORKING COPY CREATED


In [4]:
from pathlib import Path
import hashlib

ROOT = Path("/content/deterministic-intent-translation-journal")

RUNNER = ROOT / (
    "experiment-2b/implementation/e3_v3/"
    "run_experiment2b_e3v3_r7_dev_v1.py"
)

DATASET = ROOT / (
    "experiment-2b/reproducibility/"
    "e3v3_heldout_benchmark_v1.0/"
    "e3v3_heldout_benchmark_120_v1.csv"
)

EXPECTED_RUNNER_SHA = (
    "40c035cb864978c718a72fcfdfbb164d262529960b0ae78242d5194147cd8257"
)

EXPECTED_DATASET_SHA = (
    "84576e7d99a2721e61ded5a14bdafeb96707b068397243bbed32aa735496d4de"
)

def sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(1024 * 1024), b""):
            h.update(block)
    return h.hexdigest()


print("=" * 100)
print("STEP A3 — VERIFY FROZEN E3-v3 ASSETS")
print("=" * 100)

for label, path in [
    ("Canonical runner", RUNNER),
    ("Frozen benchmark", DATASET),
]:
    print(f"\n{label}:")
    print(path)
    print("Exists:", path.exists())


assert RUNNER.exists(), f"STOP: Runner missing: {RUNNER}"
assert DATASET.exists(), f"STOP: Dataset missing: {DATASET}"

runner_sha = sha256(RUNNER)
dataset_sha = sha256(DATASET)

print("\n" + "-" * 100)
print("RUNNER")
print("Actual   :", runner_sha)
print("Expected :", EXPECTED_RUNNER_SHA)
print("Match    :", runner_sha == EXPECTED_RUNNER_SHA)

print("\nDATASET")
print("Actual   :", dataset_sha)
print("Expected :", EXPECTED_DATASET_SHA)
print("Match    :", dataset_sha == EXPECTED_DATASET_SHA)

assert runner_sha == EXPECTED_RUNNER_SHA, (
    "STOP: Canonical E3-v3 runner hash mismatch."
)

assert dataset_sha == EXPECTED_DATASET_SHA, (
    "STOP: Frozen E3-v3 benchmark hash mismatch."
)

print("\n" + "=" * 100)
print("✓ CANONICAL RUNNER VERIFIED")
print("✓ FROZEN 120-CASE BENCHMARK VERIFIED")
print("✓ NO EXPERIMENT EXECUTED")
print("✓ NO MODEL/API CALL MADE")
print("=" * 100)

STEP A3 — VERIFY FROZEN E3-v3 ASSETS

Canonical runner:
/content/deterministic-intent-translation-journal/experiment-2b/implementation/e3_v3/run_experiment2b_e3v3_r7_dev_v1.py
Exists: True

Frozen benchmark:
/content/deterministic-intent-translation-journal/experiment-2b/reproducibility/e3v3_heldout_benchmark_v1.0/e3v3_heldout_benchmark_120_v1.csv
Exists: True

----------------------------------------------------------------------------------------------------
RUNNER
Actual   : 40c035cb864978c718a72fcfdfbb164d262529960b0ae78242d5194147cd8257
Expected : 40c035cb864978c718a72fcfdfbb164d262529960b0ae78242d5194147cd8257
Match    : True

DATASET
Actual   : 84576e7d99a2721e61ded5a14bdafeb96707b068397243bbed32aa735496d4de
Expected : 84576e7d99a2721e61ded5a14bdafeb96707b068397243bbed32aa735496d4de
Match    : True

✓ CANONICAL RUNNER VERIFIED
✓ FROZEN 120-CASE BENCHMARK VERIFIED
✓ NO EXPERIMENT EXECUTED
✓ NO MODEL/API CALL MADE


In [5]:
from pathlib import Path
import hashlib
import pandas as pd
import json

ROOT = Path("/content/deterministic-intent-translation-journal")

# ---------------------------------------------------------------------
# Known frozen official E3-v3 result
# ---------------------------------------------------------------------

RESULT = ROOT / (
    "experiment-2b/heldout/results/"
    "e3v3_heldout_benchmark_run_v1.csv"
)

RESULT_MANIFEST = ROOT / (
    "experiment-2b/heldout/results/"
    "e3v3_heldout_benchmark_run_v1_manifest.json"
)

EXPECTED_RESULT_SHA = (
    "12c3a82bd8cde2d07de352155e9ee14913c821fb745d401c7ffe2f207e850e5e"
)

EXPECTED_MANIFEST_SHA = (
    "7d3ed06d31ce61cb5535a3acafcb572c6ed99d27e5cc2bb58fc433b2742704af"
)

def sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(1024 * 1024), b""):
            h.update(block)
    return h.hexdigest()


print("=" * 100)
print("STEP A4 — VERIFY ORIGINAL OFFICIAL E3-v3 RESULT")
print("=" * 100)

print("\nOfficial result:")
print(RESULT)
print("Exists:", RESULT.exists())

print("\nOfficial manifest:")
print(RESULT_MANIFEST)
print("Exists:", RESULT_MANIFEST.exists())

assert RESULT.exists(), f"STOP: Official result missing: {RESULT}"
assert RESULT_MANIFEST.exists(), (
    f"STOP: Official result manifest missing: {RESULT_MANIFEST}"
)

result_sha = sha256(RESULT)
manifest_sha = sha256(RESULT_MANIFEST)

print("\n" + "-" * 100)
print("OFFICIAL RESULT SHA-256")
print("Actual   :", result_sha)
print("Expected :", EXPECTED_RESULT_SHA)
print("Match    :", result_sha == EXPECTED_RESULT_SHA)

print("\nOFFICIAL MANIFEST SHA-256")
print("Actual   :", manifest_sha)
print("Expected :", EXPECTED_MANIFEST_SHA)
print("Match    :", manifest_sha == EXPECTED_MANIFEST_SHA)

assert result_sha == EXPECTED_RESULT_SHA, (
    "STOP: Original official result hash mismatch."
)

assert manifest_sha == EXPECTED_MANIFEST_SHA, (
    "STOP: Original official result manifest hash mismatch."
)

# ---------------------------------------------------------------------
# Inspect result structure
# ---------------------------------------------------------------------

df = pd.read_csv(RESULT)

print("\n" + "-" * 100)
print("RESULT STRUCTURE")
print("-" * 100)

print("Rows   :", len(df))
print("Columns:", len(df.columns))

print("\nColumn names:")
for i, col in enumerate(df.columns, 1):
    print(f"{i:02d}. {col}")

assert len(df) == 120, (
    f"STOP: Expected 120 rows, found {len(df)}."
)

# ---------------------------------------------------------------------
# Inspect frozen manifest
# ---------------------------------------------------------------------

with open(RESULT_MANIFEST, "r", encoding="utf-8") as f:
    manifest = json.load(f)

print("\n" + "-" * 100)
print("OFFICIAL RUN MANIFEST")
print("-" * 100)

print(json.dumps(manifest, indent=2))

print("\n" + "=" * 100)
print("✓ ORIGINAL OFFICIAL RESULT VERIFIED")
print("✓ ORIGINAL RESULT MANIFEST VERIFIED")
print("✓ 120 RESULT ROWS PRESENT")
print("✓ ORIGINAL FILES WERE READ ONLY")
print("✓ NO MODEL/API CALL MADE")
print("=" * 100)

STEP A4 — VERIFY ORIGINAL OFFICIAL E3-v3 RESULT

Official result:
/content/deterministic-intent-translation-journal/experiment-2b/heldout/results/e3v3_heldout_benchmark_run_v1.csv
Exists: True

Official manifest:
/content/deterministic-intent-translation-journal/experiment-2b/heldout/results/e3v3_heldout_benchmark_run_v1_manifest.json
Exists: True

----------------------------------------------------------------------------------------------------
OFFICIAL RESULT SHA-256
Actual   : 12c3a82bd8cde2d07de352155e9ee14913c821fb745d401c7ffe2f207e850e5e
Expected : 12c3a82bd8cde2d07de352155e9ee14913c821fb745d401c7ffe2f207e850e5e
Match    : True

OFFICIAL MANIFEST SHA-256
Actual   : 7d3ed06d31ce61cb5535a3acafcb572c6ed99d27e5cc2bb58fc433b2742704af
Expected : 7d3ed06d31ce61cb5535a3acafcb572c6ed99d27e5cc2bb58fc433b2742704af
Match    : True

----------------------------------------------------------------------------------------------------
RESULT STRUCTURE
------------------------------------------

In [6]:
from pathlib import Path
import pandas as pd
import json

ROOT = Path("/content/deterministic-intent-translation-journal")

RESULT = ROOT / (
    "experiment-2b/heldout/results/"
    "e3v3_heldout_benchmark_run_v1.csv"
)

df = pd.read_csv(RESULT)

print("=" * 100)
print("STEP A5 — RECONSTRUCT OFFICIAL E3-v3 BASELINE METRICS")
print("=" * 100)

# ------------------------------------------------------------------
# Normalize boolean columns safely
# ------------------------------------------------------------------

def to_bool(series):
    if series.dtype == bool:
        return series

    return (
        series.astype(str)
        .str.strip()
        .str.lower()
        .map({
            "true": True,
            "false": False,
            "1": True,
            "0": False
        })
    )

expected_valid = to_bool(df["expected_valid"])
accepted = to_bool(df["accepted"])
decision_correct = to_bool(df["decision_correct"])
llm_invoked = to_bool(df["llm_invoked"])
parse_ok = to_bool(df["parse_ok"])

assert expected_valid.notna().all()
assert accepted.notna().all()
assert decision_correct.notna().all()
assert llm_invoked.notna().all()

# ------------------------------------------------------------------
# Basic counts
# ------------------------------------------------------------------

n = len(df)

valid_mask = expected_valid == True
invalid_mask = expected_valid == False

valid_n = int(valid_mask.sum())
invalid_n = int(invalid_mask.sum())

correct = int(decision_correct.sum())

valid_accepted = int((valid_mask & accepted).sum())
invalid_rejected = int((invalid_mask & ~accepted).sum())

# Invalid accepted = false acceptance
invalid_accepted = int((invalid_mask & accepted).sum())

# Valid rejected = false rejection
valid_rejected = int((valid_mask & ~accepted).sum())

# ------------------------------------------------------------------
# Metrics
# ------------------------------------------------------------------

accuracy = correct / n
var = valid_accepted / valid_n
irr = invalid_rejected / invalid_n

# Rejection precision/recall/F1
# "positive" = should reject invalid request

tp = invalid_rejected
fp = valid_rejected
fn = invalid_accepted
tn = valid_accepted

precision = tp / (tp + fp)
recall = tp / (tp + fn)
f1 = 2 * precision * recall / (precision + recall)

# ------------------------------------------------------------------
# LLM information
# ------------------------------------------------------------------

llm_calls = int(llm_invoked.sum())
parse_success = int((llm_invoked & parse_ok.fillna(False)).sum())

# ------------------------------------------------------------------
# Descriptor information
# ------------------------------------------------------------------

def descriptor_present(value):
    if pd.isna(value):
        return False

    s = str(value).strip()

    if s in {"", "{}", "null", "None", "nan"}:
        return False

    try:
        obj = json.loads(s)
        return obj not in ({}, None, [])
    except Exception:
        return True

descriptor_emitted = df["descriptor_json"].apply(descriptor_present)

descriptor_count = int(descriptor_emitted.sum())

# ------------------------------------------------------------------
# Print results
# ------------------------------------------------------------------

print("\nDATASET")
print("-" * 100)
print(f"Total cases                : {n}")
print(f"Expected valid             : {valid_n}")
print(f"Expected invalid           : {invalid_n}")

print("\nDECISION COUNTS")
print("-" * 100)
print(f"Correct                    : {correct}")
print(f"Incorrect                  : {n - correct}")
print(f"Valid accepted             : {valid_accepted}")
print(f"Valid rejected             : {valid_rejected}")
print(f"Invalid rejected           : {invalid_rejected}")
print(f"Invalid accepted           : {invalid_accepted}")

print("\nCONFUSION MATRIX — REJECTION AS POSITIVE")
print("-" * 100)
print(f"TP invalid rejected        : {tp}")
print(f"FP valid rejected          : {fp}")
print(f"FN invalid accepted        : {fn}")
print(f"TN valid accepted          : {tn}")

print("\nPRIMARY METRICS")
print("-" * 100)
print(f"Decision accuracy          : {correct}/{n} = {accuracy:.6f} ({accuracy*100:.2f}%)")
print(f"VAR (valid acceptance)     : {valid_accepted}/{valid_n} = {var:.6f} ({var*100:.2f}%)")
print(f"IRR (invalid rejection)    : {invalid_rejected}/{invalid_n} = {irr:.6f} ({irr*100:.2f}%)")
print(f"Rejection precision        : {precision:.6f} ({precision*100:.2f}%)")
print(f"Rejection recall           : {recall:.6f} ({recall*100:.2f}%)")
print(f"Rejection F1               : {f1:.6f} ({f1*100:.2f}%)")

print("\nEXECUTION")
print("-" * 100)
print(f"LLM invoked                : {llm_calls}/{n}")
print(f"Parse success (LLM rows)   : {parse_success}/{llm_calls}")
print(f"Descriptors emitted        : {descriptor_count}/{n}")

# ------------------------------------------------------------------
# Hard verification against frozen headline numbers
# ------------------------------------------------------------------

assert n == 120
assert valid_n == 58
assert invalid_n == 62

assert correct == 101
assert valid_accepted == 51
assert invalid_rejected == 50

assert tp == 50
assert fp == 7
assert fn == 12
assert tn == 51

assert llm_calls == 77

print("\n" + "=" * 100)
print("✓ FROZEN BASELINE RECONSTRUCTED DIRECTLY FROM RAW RESULT")
print("✓ ACCURACY = 101/120 = 84.17%")
print("✓ VAR      = 51/58   = 87.93%")
print("✓ IRR      = 50/62   = 80.65%")
print("✓ CONFUSION MATRIX = TP 50 / FP 7 / FN 12 / TN 51")
print("✓ NO MODEL/API CALL MADE")
print("=" * 100)

STEP A5 — RECONSTRUCT OFFICIAL E3-v3 BASELINE METRICS

DATASET
----------------------------------------------------------------------------------------------------
Total cases                : 120
Expected valid             : 58
Expected invalid           : 62

DECISION COUNTS
----------------------------------------------------------------------------------------------------
Correct                    : 101
Incorrect                  : 19
Valid accepted             : 51
Valid rejected             : 7
Invalid rejected           : 50
Invalid accepted           : 12

CONFUSION MATRIX — REJECTION AS POSITIVE
----------------------------------------------------------------------------------------------------
TP invalid rejected        : 50
FP valid rejected          : 7
FN invalid accepted        : 12
TN valid accepted          : 51

PRIMARY METRICS
----------------------------------------------------------------------------------------------------
Decision accuracy          : 101/120 = 0.

/tmp/ipykernel_2438/1693551863.py:97: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  parse_success = int((llm_invoked & parse_ok.fillna(False)).sum())


In [7]:
from pathlib import Path
from datetime import datetime, timezone
import hashlib
import json

ROOT = Path("/content/deterministic-intent-translation-journal")

EXP_A = ROOT / "journal_experiments" / "experiment_A_model_sweep"

DIRS = [
    EXP_A,
    EXP_A / "preregistration",
    EXP_A / "results" / "raw",
    EXP_A / "results" / "metrics",
    EXP_A / "results" / "analysis",
    EXP_A / "manifests",
    EXP_A / "logs",
]

print("=" * 100)
print("STEP A6 — INITIALIZE JOURNAL EXPERIMENT A")
print("=" * 100)

# ------------------------------------------------------------------
# Safety: do not overwrite an existing Experiment A
# ------------------------------------------------------------------

if EXP_A.exists():
    existing = list(EXP_A.rglob("*"))
    if existing:
        raise FileExistsError(
            f"\nSTOP: Experiment A directory already contains files:\n{EXP_A}\n"
            "Do not overwrite an existing experiment."
        )

for d in DIRS:
    d.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------------
# Frozen source artifacts
# ------------------------------------------------------------------

RUNNER = ROOT / (
    "experiment-2b/implementation/e3_v3/"
    "run_experiment2b_e3v3_r7_dev_v1.py"
)

DATASET = ROOT / (
    "experiment-2b/reproducibility/"
    "e3v3_heldout_benchmark_v1.0/"
    "e3v3_heldout_benchmark_120_v1.csv"
)

OLD_RESULT = ROOT / (
    "experiment-2b/heldout/results/"
    "e3v3_heldout_benchmark_run_v1.csv"
)

OLD_MANIFEST = ROOT / (
    "experiment-2b/heldout/results/"
    "e3v3_heldout_benchmark_run_v1_manifest.json"
)

def sha256(path):
    h = hashlib.sha256()

    with open(path, "rb") as f:
        for block in iter(lambda: f.read(1024 * 1024), b""):
            h.update(block)

    return h.hexdigest()

# ------------------------------------------------------------------
# Verify once more before recording provenance
# ------------------------------------------------------------------

EXPECTED = {
    "runner":
        "40c035cb864978c718a72fcfdfbb164d262529960b0ae78242d5194147cd8257",

    "dataset":
        "84576e7d99a2721e61ded5a14bdafeb96707b068397243bbed32aa735496d4de",

    "official_result":
        "12c3a82bd8cde2d07de352155e9ee14913c821fb745d401c7ffe2f207e850e5e",

    "official_manifest":
        "7d3ed06d31ce61cb5535a3acafcb572c6ed99d27e5cc2bb58fc433b2742704af",
}

actual = {
    "runner": sha256(RUNNER),
    "dataset": sha256(DATASET),
    "official_result": sha256(OLD_RESULT),
    "official_manifest": sha256(OLD_MANIFEST),
}

for key in EXPECTED:
    assert actual[key] == EXPECTED[key], (
        f"STOP: Frozen artifact changed: {key}"
    )

# ------------------------------------------------------------------
# Create Experiment A provenance record
# ------------------------------------------------------------------

provenance = {
    "experiment_id": "journal-experiment-A-model-sweep-v1.0",

    "purpose": (
        "Evaluate model sensitivity of the frozen E3-v3 deterministic "
        "intent-translation cascade while holding the benchmark and "
        "deterministic decision procedure fixed."
    ),

    "created_utc": datetime.now(timezone.utc).isoformat(),

    "frozen_inputs": {
        "runner": str(RUNNER.relative_to(ROOT)),
        "runner_sha256": actual["runner"],

        "dataset": str(DATASET.relative_to(ROOT)),
        "dataset_sha256": actual["dataset"],

        "original_official_result": str(
            OLD_RESULT.relative_to(ROOT)
        ),
        "original_official_result_sha256":
            actual["official_result"],

        "original_official_manifest": str(
            OLD_MANIFEST.relative_to(ROOT)
        ),
        "original_official_manifest_sha256":
            actual["official_manifest"],
    },

    "frozen_baseline": {
        "model": "Qwen/Qwen3.5-9B",

        "cases": 120,
        "valid_cases": 58,
        "invalid_cases": 62,

        "correct": 101,
        "accuracy": 101 / 120,

        "valid_accepted": 51,
        "valid_acceptance_rate": 51 / 58,

        "invalid_rejected": 50,
        "invalid_rejection_rate": 50 / 62,

        "rejection_f1": 0.840336,

        "llm_invocations": 77,
        "parse_successes": 77,

        "descriptors_emitted": 63,
    },

    "experimental_policy": {
        "vary_only_model": True,
        "change_runner": False,
        "change_dataset": False,
        "change_gate_logic": False,
        "change_prompt": False,
        "change_thresholds": False,
        "change_labels": False,
        "case_removal_allowed": False,
        "result_driven_rerun_allowed": False,
        "all_preregistered_arms_must_be_reported": True,
    },

    "status": "initialized_before_model_sweep",
}

PROVENANCE = EXP_A / "manifests" / "experiment_A_provenance_v1.json"

PROVENANCE.write_text(
    json.dumps(provenance, indent=2, sort_keys=True) + "\n",
    encoding="utf-8"
)

provenance_sha = sha256(PROVENANCE)

print("\nExperiment A workspace:")
print(EXP_A)

print("\nCreated directories:")
for d in DIRS:
    print(" ✓", d.relative_to(ROOT))

print("\nFrozen source verification:")
for key in actual:
    print(f" ✓ {key:20s} {actual[key]}")

print("\nProvenance:")
print(PROVENANCE)
print("SHA-256:", provenance_sha)

print("\n" + "=" * 100)
print("✓ EXPERIMENT A WORKSPACE INITIALIZED")
print("✓ ORIGINAL E3-v3 FILES REMAIN UNCHANGED")
print("✓ FROZEN BASELINE RECORDED")
print("✓ EXPERIMENT POLICY RECORDED")
print("✓ NO MODEL/API CALL MADE")
print("=" * 100)

STEP A6 — INITIALIZE JOURNAL EXPERIMENT A

Experiment A workspace:
/content/deterministic-intent-translation-journal/journal_experiments/experiment_A_model_sweep

Created directories:
 ✓ journal_experiments/experiment_A_model_sweep
 ✓ journal_experiments/experiment_A_model_sweep/preregistration
 ✓ journal_experiments/experiment_A_model_sweep/results/raw
 ✓ journal_experiments/experiment_A_model_sweep/results/metrics
 ✓ journal_experiments/experiment_A_model_sweep/results/analysis
 ✓ journal_experiments/experiment_A_model_sweep/manifests
 ✓ journal_experiments/experiment_A_model_sweep/logs

Frozen source verification:
 ✓ runner               40c035cb864978c718a72fcfdfbb164d262529960b0ae78242d5194147cd8257
 ✓ dataset              84576e7d99a2721e61ded5a14bdafeb96707b068397243bbed32aa735496d4de
 ✓ official_result      12c3a82bd8cde2d07de352155e9ee14913c821fb745d401c7ffe2f207e850e5e
 ✓ official_manifest    7d3ed06d31ce61cb5535a3acafcb572c6ed99d27e5cc2bb58fc433b2742704af

Provenance:
/conte

In [8]:
!pip -q install -U together

print("✓ Together SDK installed/updated")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 890.5/890.5 kB 18.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 242.7/242.7 kB 14.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 310.5/310.5 kB 17.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 279.3/279.3 kB 14.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.49.0 requires rich<14,>=12.4.4, but you have rich 14.3.4 which is incompatible.
✓ Together SDK installed/updated


In [9]:
import os
from getpass import getpass

print("=" * 100)
print("STEP A7 — TOGETHER API AUTHENTICATION")
print("=" * 100)

if "TOGETHER_API_KEY" not in os.environ:
    os.environ["TOGETHER_API_KEY"] = getpass(
        "Enter TOGETHER_API_KEY (input hidden): "
    )

key = os.environ.get("TOGETHER_API_KEY", "")

assert key.strip(), "STOP: TOGETHER_API_KEY is empty."

print("\n✓ TOGETHER_API_KEY is present in environment")
print("✓ API key value was NOT printed")
print("✓ NO EXPERIMENTAL MODEL CALL MADE")

STEP A7 — TOGETHER API AUTHENTICATION
Enter TOGETHER_API_KEY (input hidden): ··········

✓ TOGETHER_API_KEY is present in environment
✓ API key value was NOT printed
✓ NO EXPERIMENTAL MODEL CALL MADE


In [10]:
from together import Together

client = Together(
    api_key=os.environ["TOGETHER_API_KEY"]
)

print("=" * 100)
print("STEP A7B — RETRIEVE CURRENT TOGETHER MODEL CATALOG")
print("=" * 100)

models = client.models.list()

print("Model catalogue retrieved successfully.")
print("Returned models:", len(models))

print("\n" + "-" * 100)
print("RELEVANT LANGUAGE / CHAT MODEL CANDIDATES")
print("-" * 100)

keywords = [
    "qwen",
    "llama",
    "mistral",
    "deepseek",
    "gemma",
    "gpt-oss",
]

candidate_ids = []

for model in models:
    model_id = getattr(model, "id", str(model))

    if any(k in model_id.lower() for k in keywords):
        candidate_ids.append(model_id)

for model_id in sorted(set(candidate_ids)):
    print(model_id)

print("\nCandidate count:", len(set(candidate_ids)))

print("\n" + "=" * 100)
print("✓ TOGETHER MODEL CATALOG RETRIEVED")
print("✓ NO BENCHMARK CASE EXECUTED")
print("✓ NO EXPERIMENTAL RESULT GENERATED")
print("=" * 100)

STEP A7B — RETRIEVE CURRENT TOGETHER MODEL CATALOG
Model catalogue retrieved successfully.
Returned models: 264

----------------------------------------------------------------------------------------------------
RELEVANT LANGUAGE / CHAT MODEL CANDIDATES
----------------------------------------------------------------------------------------------------
Qwen/QwQ-32B
Qwen/Qwen-Image
Qwen/Qwen-Image-2.0
Qwen/Qwen-Image-2.0-Pro
Qwen/Qwen2-1.5B
Qwen/Qwen2-1.5B-Instruct
Qwen/Qwen2-72B
Qwen/Qwen2-72B-Instruct
Qwen/Qwen2-7B
Qwen/Qwen2-VL-72B-Instruct
Qwen/Qwen2.5-1.5B
Qwen/Qwen2.5-1.5B-Instruct
Qwen/Qwen2.5-14B
Qwen/Qwen2.5-14B-Instruct
Qwen/Qwen2.5-32B
Qwen/Qwen2.5-32B-Instruct
Qwen/Qwen2.5-3B-Instruct
Qwen/Qwen2.5-72B
Qwen/Qwen2.5-72B-Instruct
Qwen/Qwen2.5-72B-Instruct-Turbo
Qwen/Qwen2.5-7B
Qwen/Qwen2.5-7B-Instruct
Qwen/Qwen2.5-7B-Instruct-Turbo
Qwen/Qwen2.5-Coder-32B-Instruct
Qwen/Qwen2.5-VL-72B-Instruct
Qwen/Qwen3-0.6B
Qwen/Qwen3-0.6B-Base
Qwen/Qwen3-1.7B
Qwen/Qwen3-1.7B-Base
Qwen/Qwen3-

In [11]:
from pathlib import Path
from datetime import datetime, timezone
import hashlib
import json

ROOT = Path("/content/deterministic-intent-translation-journal")

EXP_A = (
    ROOT /
    "journal_experiments" /
    "experiment_A_model_sweep"
)

PREREG_DIR = EXP_A / "preregistration"

RUNNER = ROOT / (
    "experiment-2b/implementation/e3_v3/"
    "run_experiment2b_e3v3_r7_dev_v1.py"
)

DATASET = ROOT / (
    "experiment-2b/reproducibility/"
    "e3v3_heldout_benchmark_v1.0/"
    "e3v3_heldout_benchmark_120_v1.csv"
)

PROVENANCE = (
    EXP_A /
    "manifests" /
    "experiment_A_provenance_v1.json"
)

EXPECTED_RUNNER_SHA = (
    "40c035cb864978c718a72fcfdfbb164d262529960b0ae78242d5194147cd8257"
)

EXPECTED_DATASET_SHA = (
    "84576e7d99a2721e61ded5a14bdafeb96707b068397243bbed32aa735496d4de"
)


def sha256(path):
    h = hashlib.sha256()

    with open(path, "rb") as f:
        for block in iter(lambda: f.read(1024 * 1024), b""):
            h.update(block)

    return h.hexdigest()


print("=" * 100)
print("STEP A8 — SIX-MODEL EXPERIMENT A PRE-REGISTRATION")
print("=" * 100)

# ------------------------------------------------------------------
# 1. Verify frozen experimental assets
# ------------------------------------------------------------------

assert RUNNER.exists(), f"STOP: Missing runner: {RUNNER}"
assert DATASET.exists(), f"STOP: Missing dataset: {DATASET}"
assert PROVENANCE.exists(), f"STOP: Missing provenance: {PROVENANCE}"

runner_sha = sha256(RUNNER)
dataset_sha = sha256(DATASET)

assert runner_sha == EXPECTED_RUNNER_SHA, (
    "STOP: Frozen runner hash mismatch."
)

assert dataset_sha == EXPECTED_DATASET_SHA, (
    "STOP: Frozen benchmark hash mismatch."
)

print("\n✓ Frozen runner verified")
print("✓ Frozen benchmark verified")

# ------------------------------------------------------------------
# 2. Six pre-registered model arms
# ------------------------------------------------------------------

arms = [
    {
        "arm": "A1",
        "model": "Qwen/Qwen3.5-9B",
        "role": (
            "Frozen E3-v3 baseline model and mandatory "
            "reproduction gate"
        ),
    },
    {
        "arm": "A2",
        "model": "meta-llama/Llama-3.3-70B-Instruct-Turbo",
        "role": (
            "Large general-purpose model from a different "
            "model family"
        ),
    },
    {
        "arm": "A3",
        "model": "Qwen/Qwen2.5-7B-Instruct-Turbo",
        "role": (
            "Smaller model arm for model-scale sensitivity"
        ),
    },
    {
        "arm": "A4",
        "model": "mistralai/Mistral-Small-24B-Instruct-2501",
        "role": (
            "Alternative Mistral model family"
        ),
    },
    {
        "arm": "A5",
        "model": "deepseek-ai/DeepSeek-V3.1",
        "role": (
            "Alternative DeepSeek model family"
        ),
    },
    {
        "arm": "A6",
        "model": "openai/gpt-oss-20b",
        "role": (
            "Independent open-weight model family"
        ),
    },
]

# ------------------------------------------------------------------
# 3. Protect against accidental second pre-registration
# ------------------------------------------------------------------

PREREG_JSON = (
    PREREG_DIR /
    "experiment_A_model_sweep_prereg_v1.json"
)

PREREG_MD = (
    PREREG_DIR /
    "experiment_A_model_sweep_prereg_v1.md"
)

if PREREG_JSON.exists() or PREREG_MD.exists():
    raise FileExistsError(
        "\nSTOP: Experiment A has already been pre-registered.\n"
        "Do NOT silently replace the model list.\n"
        f"JSON: {PREREG_JSON}\n"
        f"MD:   {PREREG_MD}"
    )

# ------------------------------------------------------------------
# 4. Create formal pre-registration
# ------------------------------------------------------------------

created_utc = datetime.now(timezone.utc).isoformat()

prereg = {
    "preregistration_id":
        "journal-experiment-A-model-sweep-prereg-v1.0",

    "created_utc": created_utc,

    "experiment":
        "Experiment A — Model-Independence Sweep",

    "research_question": (
        "When the deterministic E3-v3 cascade and benchmark are "
        "held fixed, how sensitive are decision performance and "
        "deterministic containment to the choice of LLM semantic parser?"
    ),

    "prediction": {
        "decision_performance":
            "Decision accuracy, VAR and IRR may vary across model arms.",

        "containment":
            "Requests failing deterministic pre-actuation gates "
            "must not produce executable descriptors."
    },

    "model_arms": arms,

    "number_of_preregistered_arms": len(arms),

    "telecom_specialized_arm": {
        "included": False,
        "reason": (
            "No clearly telecom-specialized text-generation model "
            "was identified in the Together catalogue retrieved "
            "before pre-registration. No post-result substitute "
            "will be introduced."
        )
    },

    "frozen_runner": {
        "path": str(RUNNER.relative_to(ROOT)),
        "sha256": runner_sha,
    },

    "frozen_dataset": {
        "path": str(DATASET.relative_to(ROOT)),
        "sha256": dataset_sha,
        "cases": 120,
        "valid_cases": 58,
        "invalid_cases": 62,
    },

    "frozen_original_baseline": {
        "model": "Qwen/Qwen3.5-9B",

        "correct": 101,
        "cases": 120,
        "accuracy": 101 / 120,

        "valid_accepted": 51,
        "valid_cases": 58,
        "VAR": 51 / 58,

        "invalid_rejected": 50,
        "invalid_cases": 62,
        "IRR": 50 / 62,

        "rejection_precision": 50 / 57,
        "rejection_recall": 50 / 62,
        "rejection_f1": 0.840336,

        "llm_invocations": 77,
        "parse_successes": 77,
        "descriptors_emitted": 63,
    },

    "primary_metrics": [
        "decision_accuracy",
        "valid_acceptance_rate_VAR",
        "invalid_rejection_rate_IRR",
        "containment_violation_count",
    ],

    "secondary_metrics": [
        "rejection_precision",
        "rejection_recall",
        "rejection_f1",
        "llm_invocation_count",
        "parse_success_count",
        "descriptor_count",
        "end_to_end_latency",
    ],

    "confidence_intervals": {
        "method": "Clopper-Pearson exact",
        "confidence_level": 0.95,
        "required_for": [
            "decision_accuracy",
            "VAR",
            "IRR",
        ],
    },

    "containment_definition": (
        "A containment violation occurs when an executable descriptor "
        "is emitted for a request that failed a deterministic "
        "pre-actuation gate."
    ),

    "execution_order": [
        "A1",
        "A2",
        "A3",
        "A4",
        "A5",
        "A6",
    ],

    "execution_policy": {
        "A1_is_reproduction_gate": True,

        "if_A1_does_not_reproduce": (
            "STOP. Do not execute A2-A6. Investigate environment, "
            "API, runner, dependency, or serving drift."
        ),

        "all_preregistered_arms_must_be_reported": True,

        "failed_or_unavailable_arm_policy": (
            "Preserve and report the failure/unavailability. "
            "Do not silently substitute another model."
        ),

        "no_result_driven_arm_removal": True,
        "no_result_driven_rerun": True,
        "no_case_removal": True,
        "no_case_relabeling": True,

        "no_prompt_change_between_arms": True,
        "no_gate_change_between_arms": True,
        "no_threshold_change_between_arms": True,
        "no_dataset_change_between_arms": True,

        "vary_only_model": True,
    },

    "interpretation_boundaries": [
        (
            "This experiment evaluates model sensitivity within "
            "one controlled benchmark."
        ),
        (
            "It does not establish universal model independence."
        ),
        (
            "It does not establish hallucination-free operation."
        ),
        (
            "It does not establish external real-world generalization."
        ),
        (
            "It does not establish formal 3GPP or O-RAN conformance."
        ),
    ],
}

PREREG_JSON.write_text(
    json.dumps(prereg, indent=2) + "\n",
    encoding="utf-8"
)

# ------------------------------------------------------------------
# 5. Human-readable pre-registration
# ------------------------------------------------------------------

lines = [
    "# Experiment A — Model-Independence Sweep",
    "",
    "## Pre-registration",
    "",
    f"Created UTC: `{created_utc}`",
    "",
    "This document was created before any Experiment A benchmark arm "
    "was executed.",
    "",
    "## Research question",
    "",
    (
        "When the deterministic E3-v3 cascade and benchmark are held "
        "fixed, how sensitive are decision performance and deterministic "
        "containment to the choice of LLM semantic parser?"
    ),
    "",
    "## Frozen runner",
    "",
    f"`{RUNNER.relative_to(ROOT)}`",
    "",
    f"SHA-256: `{runner_sha}`",
    "",
    "## Frozen benchmark",
    "",
    f"`{DATASET.relative_to(ROOT)}`",
    "",
    f"SHA-256: `{dataset_sha}`",
    "",
    "Cases: **120** — 58 valid and 62 invalid.",
    "",
    "## Pre-registered model arms",
    "",
    "| Arm | Model | Experimental role |",
    "|---|---|---|",
]

for arm in arms:
    lines.append(
        f"| {arm['arm']} | `{arm['model']}` | {arm['role']} |"
    )

lines += [
    "",
    "## Pre-registered prediction",
    "",
    (
        "Decision performance may vary across models. "
        "Deterministic containment is expected to remain invariant: "
        "requests failing deterministic pre-actuation gates must not "
        "produce executable descriptors."
    ),
    "",
    "## Primary metrics",
    "",
    "- Decision accuracy",
    "- Valid acceptance rate (VAR)",
    "- Invalid rejection rate (IRR)",
    "- Containment violation count",
    "",
    (
        "Decision accuracy, VAR and IRR will be reported with "
        "95% exact Clopper–Pearson confidence intervals."
    ),
    "",
    "## Execution policy",
    "",
    (
        "A1 (`Qwen/Qwen3.5-9B`) is the mandatory reproduction gate. "
        "If A1 does not reproduce the frozen E3-v3 baseline, execution "
        "will stop before A2–A6."
    ),
    "",
    (
        "All six pre-registered arms will be reported, including "
        "failed or poorly performing arms. No result-driven reruns, "
        "model substitutions, case removal, relabeling, prompt changes, "
        "gate changes, threshold changes, or benchmark changes are permitted."
    ),
    "",
    "## Telecom-specialized model",
    "",
    (
        "No telecom-specialized arm was pre-registered because no clearly "
        "telecom-specialized text-generation model was identified in the "
        "Together catalogue retrieved before execution. No model will be "
        "substituted after observing results."
    ),
    "",
    "## Interpretation boundaries",
    "",
    "- Controlled model-sensitivity experiment, not universal model independence.",
    "- Hallucination-resistant does not mean hallucination-free.",
    "- The benchmark does not establish external real-world generalization.",
    "- No formal 3GPP or O-RAN conformance claim is made.",
]

PREREG_MD.write_text(
    "\n".join(lines) + "\n",
    encoding="utf-8"
)

# ------------------------------------------------------------------
# 6. Hash/freeze the pre-registration
# ------------------------------------------------------------------

json_sha = sha256(PREREG_JSON)
md_sha = sha256(PREREG_MD)

PREREG_MANIFEST = (
    EXP_A /
    "manifests" /
    "experiment_A_preregistration_manifest_v1.json"
)

manifest = {
    "preregistration_id":
        "journal-experiment-A-model-sweep-prereg-v1.0",

    "created_utc": created_utc,

    "model_arm_count": 6,

    "json": str(PREREG_JSON.relative_to(ROOT)),
    "json_sha256": json_sha,

    "markdown": str(PREREG_MD.relative_to(ROOT)),
    "markdown_sha256": md_sha,

    "runner_sha256": runner_sha,
    "dataset_sha256": dataset_sha,

    "status": "frozen_before_experiment_A_model_calls",
}

PREREG_MANIFEST.write_text(
    json.dumps(manifest, indent=2) + "\n",
    encoding="utf-8"
)

manifest_sha = sha256(PREREG_MANIFEST)

# ------------------------------------------------------------------
# 7. Final output
# ------------------------------------------------------------------

print("\nPRE-REGISTERED MODEL ARMS")
print("-" * 100)

for arm in arms:
    print(f"{arm['arm']}  {arm['model']}")
    print(f"    Role: {arm['role']}")

print("\n" + "-" * 100)

print("Pre-registration JSON:")
print(PREREG_JSON)
print("SHA-256:", json_sha)

print("\nPre-registration Markdown:")
print(PREREG_MD)
print("SHA-256:", md_sha)

print("\nPre-registration manifest:")
print(PREREG_MANIFEST)
print("SHA-256:", manifest_sha)

print("\nFrozen runner SHA-256:")
print(runner_sha)

print("\nFrozen dataset SHA-256:")
print(dataset_sha)

print("\n" + "=" * 100)
print("✓ SIX MODEL ARMS PRE-REGISTERED")
print("✓ MODEL LIST IS NOW FROZEN")
print("✓ PRE-REGISTRATION HASHED")
print("✓ RUNNER REMAINS FROZEN")
print("✓ BENCHMARK REMAINS FROZEN")
print("✓ NO EXPERIMENT A BENCHMARK MODEL CALL MADE")
print("=" * 100)

STEP A8 — SIX-MODEL EXPERIMENT A PRE-REGISTRATION

✓ Frozen runner verified
✓ Frozen benchmark verified

PRE-REGISTERED MODEL ARMS
----------------------------------------------------------------------------------------------------
A1  Qwen/Qwen3.5-9B
    Role: Frozen E3-v3 baseline model and mandatory reproduction gate
A2  meta-llama/Llama-3.3-70B-Instruct-Turbo
    Role: Large general-purpose model from a different model family
A3  Qwen/Qwen2.5-7B-Instruct-Turbo
    Role: Smaller model arm for model-scale sensitivity
A4  mistralai/Mistral-Small-24B-Instruct-2501
    Role: Alternative Mistral model family
A5  deepseek-ai/DeepSeek-V3.1
    Role: Alternative DeepSeek model family
A6  openai/gpt-oss-20b
    Role: Independent open-weight model family

----------------------------------------------------------------------------------------------------
Pre-registration JSON:
/content/deterministic-intent-translation-journal/journal_experiments/experiment_A_model_sweep/preregistration/experi

In [12]:
from pathlib import Path
import ast
import hashlib
import re

ROOT = Path("/content/deterministic-intent-translation-journal")

RUNNER = ROOT / (
    "experiment-2b/implementation/e3_v3/"
    "run_experiment2b_e3v3_r7_dev_v1.py"
)

EXPECTED_RUNNER_SHA = (
    "40c035cb864978c718a72fcfdfbb164d262529960b0ae78242d5194147cd8257"
)

def sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(1024 * 1024), b""):
            h.update(block)
    return h.hexdigest()


print("=" * 100)
print("STEP A9 — INSPECT CANONICAL RUNNER")
print("=" * 100)

assert RUNNER.exists()

actual_sha = sha256(RUNNER)

print("\nRunner:")
print(RUNNER)

print("\nSHA-256:")
print(actual_sha)

assert actual_sha == EXPECTED_RUNNER_SHA, (
    "STOP: runner hash changed."
)

source = RUNNER.read_text(encoding="utf-8")

# ------------------------------------------------------------------
# Syntax validation only — does not execute the runner
# ------------------------------------------------------------------

ast.parse(source)

print("\n✓ Python syntax parsed successfully")

# ------------------------------------------------------------------
# Extract CLI arguments
# ------------------------------------------------------------------

arg_patterns = re.findall(
    r'add_argument\(\s*[\'"]([^\'"]+)[\'"]',
    source
)

print("\nCLI ARGUMENTS FOUND")
print("-" * 100)

for arg in arg_patterns:
    print(arg)

# ------------------------------------------------------------------
# Show lines relevant to CLI/model/output/API behavior
# ------------------------------------------------------------------

keywords = [
    "DEFAULT_MODEL",
    "add_argument",
    "TOGETHER_API_KEY",
    "Together",
    "--dataset",
    "--out",
    "--model",
    "--limit",
]

print("\nRELEVANT RUNNER LINES")
print("-" * 100)

for line_no, line in enumerate(source.splitlines(), start=1):
    if any(k in line for k in keywords):
        print(f"{line_no:04d}: {line}")

# ------------------------------------------------------------------
# Detect obvious write operations
# ------------------------------------------------------------------

write_terms = [
    "to_csv(",
    "write_text(",
    "write_bytes(",
    "open(",
    "json.dump(",
]

print("\nPOTENTIAL FILE-WRITE LINES")
print("-" * 100)

found_write = False

for line_no, line in enumerate(source.splitlines(), start=1):
    if any(term in line for term in write_terms):
        print(f"{line_no:04d}: {line}")
        found_write = True

if not found_write:
    print("No obvious file-write statements found.")

print("\n" + "=" * 100)
print("✓ RUNNER INSPECTED")
print("✓ RUNNER NOT MODIFIED")
print("✓ RUNNER NOT EXECUTED")
print("✓ NO MODEL/API CALL MADE")
print("=" * 100)

STEP A9 — INSPECT CANONICAL RUNNER

Runner:
/content/deterministic-intent-translation-journal/experiment-2b/implementation/e3_v3/run_experiment2b_e3v3_r7_dev_v1.py

SHA-256:
40c035cb864978c718a72fcfdfbb164d262529960b0ae78242d5194147cd8257

✓ Python syntax parsed successfully

CLI ARGUMENTS FOUND
----------------------------------------------------------------------------------------------------
--dataset
--out
--model
--limit

RELEVANT RUNNER LINES
----------------------------------------------------------------------------------------------------
0103: DEFAULT_MODEL = "Qwen/Qwen3.5-9B"
0805:     model=DEFAULT_MODEL,
1117:     parser.add_argument(
1118:         "--dataset",
1122:     parser.add_argument(
1123:         "--out",
1127:     parser.add_argument(
1128:         "--model",
1129:         default=DEFAULT_MODEL,
1132:     parser.add_argument(
1133:         "--limit",

POTENTIAL FILE-WRITE LINES
--------------------------------------------------------------------------------------

In [13]:
from pathlib import Path
import ast
import importlib.util

ROOT = Path("/content/deterministic-intent-translation-journal")

RUNNER = ROOT / (
    "experiment-2b/implementation/e3_v3/"
    "run_experiment2b_e3v3_r7_dev_v1.py"
)

source = RUNNER.read_text(encoding="utf-8")
tree = ast.parse(source)

imports = set()

for node in ast.walk(tree):
    if isinstance(node, ast.Import):
        for alias in node.names:
            imports.add(alias.name.split(".")[0])

    elif isinstance(node, ast.ImportFrom):
        if node.module:
            imports.add(node.module.split(".")[0])

print("=" * 100)
print("STEP A9B — RUNNER IMPORT PREFLIGHT")
print("=" * 100)

print("\nTop-level imports:")
print("-" * 100)

for name in sorted(imports):
    print(name)

print("\nImport availability:")
print("-" * 100)

missing = []

for name in sorted(imports):

    # Local project modules may not be discoverable until
    # the runner establishes its own sys.path.
    try:
        spec = importlib.util.find_spec(name)
        available = spec is not None
    except Exception:
        available = False

    print(f"{name:30s} {'FOUND' if available else 'NOT FOUND / MAY BE LOCAL'}")

    if not available:
        missing.append(name)

print("\nTOGETHER_API_KEY present:",
      bool(__import__("os").environ.get("TOGETHER_API_KEY")))

print("\n" + "=" * 100)
print("✓ IMPORT PREFLIGHT COMPLETED")
print("✓ NO BENCHMARK CASE EXECUTED")
print("✓ NO MODEL/API CALL MADE")
print("=" * 100)

STEP A9B — RUNNER IMPORT PREFLIGHT

Top-level imports:
----------------------------------------------------------------------------------------------------
argparse
cir_reconciliation
e3v2_guard
importlib
json
pandas
pathlib
qualitative_post_preservation_r7
qualitative_source_integration_r7
reliability_normalization_r5_final
service_type_integration_r6
src
sys
time

Import availability:
----------------------------------------------------------------------------------------------------
argparse                       FOUND
cir_reconciliation             NOT FOUND / MAY BE LOCAL
e3v2_guard                     NOT FOUND / MAY BE LOCAL
importlib                      FOUND
json                           FOUND
pandas                         FOUND
pathlib                        FOUND
qualitative_post_preservation_r7 NOT FOUND / MAY BE LOCAL
qualitative_source_integration_r7 NOT FOUND / MAY BE LOCAL
reliability_normalization_r5_final NOT FOUND / MAY BE LOCAL
service_type_integration_r6    NOT 

In [14]:
from pathlib import Path
import hashlib
import re

ROOT = Path("/content/deterministic-intent-translation-journal")

RUNNER = ROOT / (
    "experiment-2b/implementation/e3_v3/"
    "run_experiment2b_e3v3_r7_dev_v1.py"
)

DATASET = ROOT / (
    "experiment-2b/reproducibility/"
    "e3v3_heldout_benchmark_v1.0/"
    "e3v3_heldout_benchmark_120_v1.csv"
)

EXP_A = (
    ROOT /
    "journal_experiments" /
    "experiment_A_model_sweep"
)

A1_OUT = (
    EXP_A /
    "results" /
    "raw" /
    "A1_Qwen3.5-9B_results.csv"
)

EXPECTED_RUNNER_SHA = (
    "40c035cb864978c718a72fcfdfbb164d262529960b0ae78242d5194147cd8257"
)

EXPECTED_DATASET_SHA = (
    "84576e7d99a2721e61ded5a14bdafeb96707b068397243bbed32aa735496d4de"
)

def sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(1024 * 1024), b""):
            h.update(block)
    return h.hexdigest()


print("=" * 100)
print("STEP A10 — FINAL A1 PRE-EXECUTION VERIFICATION")
print("=" * 100)

# ------------------------------------------------------------------
# Frozen assets
# ------------------------------------------------------------------

assert RUNNER.exists()
assert DATASET.exists()

runner_sha = sha256(RUNNER)
dataset_sha = sha256(DATASET)

assert runner_sha == EXPECTED_RUNNER_SHA, "STOP: runner changed."
assert dataset_sha == EXPECTED_DATASET_SHA, "STOP: dataset changed."

print("\n✓ Runner SHA verified")
print("✓ Dataset SHA verified")

# ------------------------------------------------------------------
# Pre-registration must already exist
# ------------------------------------------------------------------

PREREG = (
    EXP_A /
    "preregistration" /
    "experiment_A_model_sweep_prereg_v1.json"
)

PREREG_MANIFEST = (
    EXP_A /
    "manifests" /
    "experiment_A_preregistration_manifest_v1.json"
)

assert PREREG.exists(), "STOP: preregistration missing."
assert PREREG_MANIFEST.exists(), "STOP: preregistration manifest missing."

print("✓ Pre-registration present")
print("✓ Pre-registration manifest present")

# ------------------------------------------------------------------
# A1 output must NOT already exist
# ------------------------------------------------------------------

assert not A1_OUT.exists(), (
    f"STOP: A1 output already exists:\n{A1_OUT}\n"
    "Do not overwrite or rerun without investigation."
)

print("✓ A1 output path is clean")

# ------------------------------------------------------------------
# API key
# ------------------------------------------------------------------

import os

assert os.environ.get("TOGETHER_API_KEY"), (
    "STOP: TOGETHER_API_KEY missing."
)

print("✓ TOGETHER_API_KEY present")

# ------------------------------------------------------------------
# Inspect frozen runner for model propagation
# ------------------------------------------------------------------

source = RUNNER.read_text(encoding="utf-8")

print("\nMODEL-RELATED LINES")
print("-" * 100)

for no, line in enumerate(source.splitlines(), 1):
    lower = line.lower()

    if (
        "model=" in lower
        or "--model" in lower
        or "default_model" in lower
    ):
        print(f"{no:04d}: {line}")

# ------------------------------------------------------------------
# Print exact command — but DO NOT execute
# ------------------------------------------------------------------

command = (
    f'python "{RUNNER}" '
    f'--dataset "{DATASET}" '
    f'--out "{A1_OUT}" '
    f'--model "Qwen/Qwen3.5-9B"'
)

print("\n" + "-" * 100)
print("REGISTERED A1 COMMAND")
print("-" * 100)
print(command)

print("\nA1 output will be:")
print(A1_OUT)

print("\n" + "=" * 100)
print("✓ A1 PRE-EXECUTION CHECK PASSED")
print("✓ NO --limit WILL BE USED")
print("✓ NO RESULT FILE WILL BE OVERWRITTEN")
print("✓ NO MODEL/API CALL MADE")
print("=" * 100)

STEP A10 — FINAL A1 PRE-EXECUTION VERIFICATION

✓ Runner SHA verified
✓ Dataset SHA verified
✓ Pre-registration present
✓ Pre-registration manifest present
✓ A1 output path is clean
✓ TOGETHER_API_KEY present

MODEL-RELATED LINES
----------------------------------------------------------------------------------------------------
0103: DEFAULT_MODEL = "Qwen/Qwen3.5-9B"
0302:         model=model,
0805:     model=DEFAULT_MODEL,
1128:         "--model",
1129:         default=DEFAULT_MODEL,
1143:         model=args.model,

----------------------------------------------------------------------------------------------------
REGISTERED A1 COMMAND
----------------------------------------------------------------------------------------------------
python "/content/deterministic-intent-translation-journal/experiment-2b/implementation/e3_v3/run_experiment2b_e3v3_r7_dev_v1.py" --dataset "/content/deterministic-intent-translation-journal/experiment-2b/reproducibility/e3v3_heldout_benchmark_v1.0/e3v3

In [15]:
import subprocess
from pathlib import Path
from datetime import datetime, timezone
import hashlib
import json
import time

ROOT = Path("/content/deterministic-intent-translation-journal")

RUNNER = ROOT / (
    "experiment-2b/implementation/e3_v3/"
    "run_experiment2b_e3v3_r7_dev_v1.py"
)

DATASET = ROOT / (
    "experiment-2b/reproducibility/"
    "e3v3_heldout_benchmark_v1.0/"
    "e3v3_heldout_benchmark_120_v1.csv"
)

EXP_A = (
    ROOT /
    "journal_experiments" /
    "experiment_A_model_sweep"
)

A1_OUT = (
    EXP_A /
    "results" /
    "raw" /
    "A1_Qwen3.5-9B_results.csv"
)

A1_LOG = (
    EXP_A /
    "logs" /
    "A1_Qwen3.5-9B_execution.log"
)

MODEL = "Qwen/Qwen3.5-9B"

def sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(1024 * 1024), b""):
            h.update(block)
    return h.hexdigest()


print("=" * 100)
print("STEP A11 — EXECUTE A1 BASELINE REPRODUCTION GATE")
print("=" * 100)

assert not A1_OUT.exists(), (
    "STOP: A1 result already exists. Refusing to overwrite."
)

assert not A1_LOG.exists(), (
    "STOP: A1 execution log already exists. Refusing to overwrite."
)

cmd = [
    "python",
    str(RUNNER),
    "--dataset", str(DATASET),
    "--out", str(A1_OUT),
    "--model", MODEL,
]

print("\nModel   :", MODEL)
print("Dataset :", DATASET)
print("Output  :", A1_OUT)

print("\nExecuting frozen 120-case benchmark ONCE...")
print("Do not interrupt the cell unless an actual error occurs.\n")

started = datetime.now(timezone.utc)
t0 = time.time()

process = subprocess.run(
    cmd,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    env=__import__("os").environ.copy(),
)

finished = datetime.now(timezone.utc)
elapsed = time.time() - t0

# Preserve complete console output whether success or failure
A1_LOG.write_text(
    process.stdout or "",
    encoding="utf-8"
)

print(process.stdout)

print("\n" + "-" * 100)
print("PROCESS RETURN CODE :", process.returncode)
print(f"WALL-CLOCK SECONDS  : {elapsed:.2f}")
print("LOG                  :", A1_LOG)

if process.returncode != 0:
    print("\n✗ A1 EXECUTION FAILED")
    print("✗ DO NOT RERUN")
    print("✗ DO NOT START A2")
    print("Send this complete output to ChatGPT for diagnosis.")
    raise RuntimeError(
        f"A1 runner exited with code {process.returncode}"
    )

assert A1_OUT.exists(), (
    "STOP: Runner returned success but A1 output was not created."
)

result_sha = sha256(A1_OUT)
log_sha = sha256(A1_LOG)

print("\nA1 RESULT CREATED")
print("SHA-256:", result_sha)

print("\nA1 EXECUTION LOG")
print("SHA-256:", log_sha)

print("\n" + "=" * 100)
print("✓ A1 EXECUTION COMPLETED")
print("✓ A1 RESULT PRESERVED")
print("✓ EXECUTION LOG PRESERVED")
print("✓ DO NOT RUN A2 YET")
print("=" * 100)

STEP A11 — EXECUTE A1 BASELINE REPRODUCTION GATE

Model   : Qwen/Qwen3.5-9B
Dataset : /content/deterministic-intent-translation-journal/experiment-2b/reproducibility/e3v3_heldout_benchmark_v1.0/e3v3_heldout_benchmark_120_v1.csv
Output  : /content/deterministic-intent-translation-journal/journal_experiments/experiment_A_model_sweep/results/raw/A1_Qwen3.5-9B_results.csv

Executing frozen 120-case benchmark ONCE...
Do not interrupt the cell unless an actual error occurs.

[1/120] HO001
[2/120] HO002
[3/120] HO003
[4/120] HO004
[5/120] HO005
[6/120] HO006
[7/120] HO007
[8/120] HO008
[9/120] HO009
[10/120] HO010
[11/120] HO011
[12/120] HO012
[13/120] HO013
[14/120] HO014
[15/120] HO015
[16/120] HO016
[17/120] HO017
[18/120] HO018
[19/120] HO019
[20/120] HO020
[21/120] HO021
[22/120] HO022
[23/120] HO023
[24/120] HO024
[25/120] HO025
[26/120] HO026
[27/120] HO027
[28/120] HO028
[29/120] HO029
[30/120] HO030
[31/120] HO031
[32/120] HO032
[33/120] HO033
[34/120] HO034
[35/120] HO035
[36/120] H

In [16]:
from pathlib import Path
import pandas as pd
import hashlib
import json

ROOT = Path("/content/deterministic-intent-translation-journal")

EXP_A = (
    ROOT /
    "journal_experiments" /
    "experiment_A_model_sweep"
)

A1 = (
    EXP_A /
    "results" /
    "raw" /
    "A1_Qwen3.5-9B_results.csv"
)

ORIGINAL = (
    ROOT /
    "experiment-2b" /
    "heldout" /
    "results" /
    "e3v3_heldout_benchmark_run_v1.csv"
)

METRICS_DIR = EXP_A / "results" / "metrics"
ANALYSIS_DIR = EXP_A / "results" / "analysis"

METRICS_DIR.mkdir(parents=True, exist_ok=True)
ANALYSIS_DIR.mkdir(parents=True, exist_ok=True)

EXPECTED_ORIGINAL_SHA = (
    "12c3a82bd8cde2d07de352155e9ee14913c821fb745d401c7ffe2f207e850e5e"
)

EXPECTED_A1_SHA = (
    "c22618b2750de7f5ef1824fa4036a901ef54f611ad162fcd797b20d79e2b4e06"
)


def sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(1024 * 1024), b""):
            h.update(block)
    return h.hexdigest()


def as_bool(series):
    """
    Robust conversion for CSV boolean columns.
    """
    if pd.api.types.is_bool_dtype(series):
        return series.fillna(False)

    return (
        series.astype(str)
        .str.strip()
        .str.lower()
        .map({
            "true": True,
            "false": False,
            "1": True,
            "0": False,
        })
        .fillna(False)
        .astype(bool)
    )


def metrics(df):
    expected_valid = as_bool(df["expected_valid"])
    accepted = as_bool(df["accepted"])
    decision_correct = as_bool(df["decision_correct"])
    llm_invoked = as_bool(df["llm_invoked"])
    parse_ok = as_bool(df["parse_ok"])

    valid = expected_valid
    invalid = ~expected_valid

    valid_accepted = int((valid & accepted).sum())
    valid_rejected = int((valid & ~accepted).sum())

    invalid_rejected = int((invalid & ~accepted).sum())
    invalid_accepted = int((invalid & accepted).sum())

    total = len(df)
    valid_n = int(valid.sum())
    invalid_n = int(invalid.sum())
    correct = int(decision_correct.sum())

    # invalid/reject is the positive class
    TP = invalid_rejected
    FP = valid_rejected
    FN = invalid_accepted
    TN = valid_accepted

    precision = TP / (TP + FP) if (TP + FP) else 0.0
    recall = TP / (TP + FN) if (TP + FN) else 0.0

    f1 = (
        2 * precision * recall / (precision + recall)
        if (precision + recall)
        else 0.0
    )

    descriptor_count = int(
        df["descriptor_json"].fillna("").astype(str)
        .str.strip()
        .ne("")
        .sum()
    )

    return {
        "total": total,
        "valid": valid_n,
        "invalid": invalid_n,

        "correct": correct,
        "incorrect": total - correct,

        "valid_accepted": valid_accepted,
        "valid_rejected": valid_rejected,

        "invalid_rejected": invalid_rejected,
        "invalid_accepted": invalid_accepted,

        "TP": TP,
        "FP": FP,
        "FN": FN,
        "TN": TN,

        "accuracy": correct / total,
        "VAR": valid_accepted / valid_n,
        "IRR": invalid_rejected / invalid_n,

        "rejection_precision": precision,
        "rejection_recall": recall,
        "rejection_f1": f1,

        "llm_invoked": int(llm_invoked.sum()),
        "parse_success": int((llm_invoked & parse_ok).sum()),

        "descriptor_count": descriptor_count,
    }


print("=" * 110)
print("STEP A12 — A1 FROZEN-BASELINE REPRODUCTION ANALYSIS")
print("=" * 110)

# ------------------------------------------------------------------
# Integrity
# ------------------------------------------------------------------

assert A1.exists(), "STOP: A1 result missing."
assert ORIGINAL.exists(), "STOP: frozen official result missing."

a1_sha = sha256(A1)
original_sha = sha256(ORIGINAL)

assert a1_sha == EXPECTED_A1_SHA, (
    "STOP: A1 result changed after execution."
)

assert original_sha == EXPECTED_ORIGINAL_SHA, (
    "STOP: original frozen result hash mismatch."
)

print("\n✓ A1 result hash verified")
print("✓ Original frozen result hash verified")

# ------------------------------------------------------------------
# Load
# ------------------------------------------------------------------

new = pd.read_csv(A1)
old = pd.read_csv(ORIGINAL)

assert len(new) == 120
assert len(old) == 120

assert new["id"].is_unique
assert old["id"].is_unique

assert set(new["id"]) == set(old["id"]), (
    "STOP: case IDs differ."
)

print("✓ Both files contain exactly 120 unique matching case IDs")

# ------------------------------------------------------------------
# Metrics
# ------------------------------------------------------------------

old_m = metrics(old)
new_m = metrics(new)

print("\n" + "-" * 110)
print("FROZEN BASELINE vs NEW A1")
print("-" * 110)

print(
    f"{'Metric':30s}"
    f"{'Frozen':>16s}"
    f"{'New A1':>16s}"
    f"{'Delta':>16s}"
)

print("-" * 110)

for key in ["accuracy", "VAR", "IRR",
            "rejection_precision",
            "rejection_recall",
            "rejection_f1"]:

    old_v = old_m[key]
    new_v = new_m[key]

    print(
        f"{key:30s}"
        f"{old_v:16.6f}"
        f"{new_v:16.6f}"
        f"{new_v-old_v:+16.6f}"
    )

print("\nCOUNTS")
print("-" * 110)

count_keys = [
    "correct",
    "incorrect",
    "valid_accepted",
    "valid_rejected",
    "invalid_rejected",
    "invalid_accepted",
    "TP", "FP", "FN", "TN",
    "llm_invoked",
    "parse_success",
    "descriptor_count",
]

for key in count_keys:
    print(
        f"{key:30s}"
        f"{old_m[key]:16d}"
        f"{new_m[key]:16d}"
        f"{new_m[key]-old_m[key]:+16d}"
    )

# ------------------------------------------------------------------
# Case-level comparison
# ------------------------------------------------------------------

comparison = old[
    [
        "id",
        "expected_valid",
        "accepted",
        "decision_correct",
        "llm_invoked",
        "parse_ok",
        "rejection_stage",
        "error_codes_json",
    ]
].copy()

comparison = comparison.rename(
    columns={
        "accepted": "old_accepted",
        "decision_correct": "old_correct",
        "llm_invoked": "old_llm_invoked",
        "parse_ok": "old_parse_ok",
        "rejection_stage": "old_rejection_stage",
        "error_codes_json": "old_error_codes",
    }
)

new_part = new[
    [
        "id",
        "accepted",
        "decision_correct",
        "llm_invoked",
        "parse_ok",
        "rejection_stage",
        "error_codes_json",
    ]
].copy()

new_part = new_part.rename(
    columns={
        "accepted": "new_accepted",
        "decision_correct": "new_correct",
        "llm_invoked": "new_llm_invoked",
        "parse_ok": "new_parse_ok",
        "rejection_stage": "new_rejection_stage",
        "error_codes_json": "new_error_codes",
    }
)

comparison = comparison.merge(
    new_part,
    on="id",
    how="inner",
    validate="one_to_one",
)

for col in [
    "old_accepted",
    "new_accepted",
    "old_correct",
    "new_correct",
    "old_llm_invoked",
    "new_llm_invoked",
    "old_parse_ok",
    "new_parse_ok",
]:
    comparison[col] = as_bool(comparison[col])

comparison["decision_changed"] = (
    comparison["old_accepted"] != comparison["new_accepted"]
)

comparison["correctness_changed"] = (
    comparison["old_correct"] != comparison["new_correct"]
)

comparison["llm_path_changed"] = (
    comparison["old_llm_invoked"] !=
    comparison["new_llm_invoked"]
)

decision_changes = int(comparison["decision_changed"].sum())
correctness_changes = int(comparison["correctness_changed"].sum())
llm_path_changes = int(comparison["llm_path_changed"].sum())

print("\nCASE-LEVEL DRIFT")
print("-" * 110)

print("Accepted/rejected decisions changed :", decision_changes)
print("Decision correctness changed        :", correctness_changes)
print("LLM invocation path changed         :", llm_path_changes)

changed = comparison[
    comparison["decision_changed"] |
    comparison["correctness_changed"] |
    comparison["llm_path_changed"]
].copy()

if len(changed):
    print("\nCASES WITH DECISION/PATH DRIFT")
    print("-" * 110)

    show_cols = [
        "id",
        "expected_valid",
        "old_accepted",
        "new_accepted",
        "old_correct",
        "new_correct",
        "old_llm_invoked",
        "new_llm_invoked",
        "old_rejection_stage",
        "new_rejection_stage",
    ]

    print(changed[show_cols].to_string(index=False))

else:
    print("\n✓ No case-level decision/path drift detected")

# ------------------------------------------------------------------
# Reproduction gate
#
# Pre-registration requires published baseline reproduction.
# Exact raw CSV SHA equality is NOT required because latency/raw
# hosted-model fields may differ.
# ------------------------------------------------------------------

metric_exact = (
    new_m["correct"] == 101
    and new_m["valid_accepted"] == 51
    and new_m["invalid_rejected"] == 50
)

decision_exact = (decision_changes == 0)

path_exact = (llm_path_changes == 0)

gate_pass = (
    metric_exact
    and decision_exact
    and path_exact
)

print("\n" + "=" * 110)
print("A1 REPRODUCTION GATE")
print("=" * 110)

print("Expected accuracy : 101/120 = 84.17%")
print("Observed accuracy :", f"{new_m['correct']}/120 = {100*new_m['accuracy']:.2f}%")

print("\nExpected VAR      : 51/58  = 87.93%")
print("Observed VAR      :", f"{new_m['valid_accepted']}/58 = {100*new_m['VAR']:.2f}%")

print("\nExpected IRR      : 50/62  = 80.65%")
print("Observed IRR      :", f"{new_m['invalid_rejected']}/62 = {100*new_m['IRR']:.2f}%")

print("\nMetric reproduction exact :", metric_exact)
print("Decision reproduction exact:", decision_exact)
print("LLM path reproduction exact :", path_exact)

if gate_pass:
    print("\n✓ A1 REPRODUCTION GATE PASSED")
    print("✓ Eligible to proceed to A2 after recording A1 metrics")
else:
    print("\n✗ A1 REPRODUCTION GATE DID NOT PASS")
    print("✗ DO NOT RUN A2")
    print("✗ Preserve this run and diagnose the discrepancy")

# ------------------------------------------------------------------
# Save analysis
# ------------------------------------------------------------------

comparison_path = (
    ANALYSIS_DIR /
    "A1_vs_frozen_case_comparison.csv"
)

metrics_path = (
    METRICS_DIR /
    "A1_reproduction_metrics.json"
)

assert not comparison_path.exists(), (
    "STOP: comparison output already exists."
)

assert not metrics_path.exists(), (
    "STOP: metrics output already exists."
)

comparison.to_csv(
    comparison_path,
    index=False
)

record = {
    "arm": "A1",
    "model": "Qwen/Qwen3.5-9B",

    "a1_result_sha256": a1_sha,
    "frozen_result_sha256": original_sha,

    "frozen_metrics": old_m,
    "a1_metrics": new_m,

    "decision_changes": decision_changes,
    "correctness_changes": correctness_changes,
    "llm_path_changes": llm_path_changes,

    "metric_reproduction_exact": metric_exact,
    "decision_reproduction_exact": decision_exact,
    "llm_path_reproduction_exact": path_exact,

    "reproduction_gate_passed": gate_pass,
}

metrics_path.write_text(
    json.dumps(record, indent=2) + "\n",
    encoding="utf-8"
)

print("\nArtifacts:")
print(comparison_path)
print(metrics_path)

print("\nComparison SHA-256:")
print(sha256(comparison_path))

print("\nMetrics SHA-256:")
print(sha256(metrics_path))

print("\n" + "=" * 110)

if gate_pass:
    print("✓ STEP A12 COMPLETE — A1 GATE PASSED")
else:
    print("✗ STEP A12 COMPLETE — A1 GATE FAILED; STOP HERE")

print("=" * 110)

STEP A12 — A1 FROZEN-BASELINE REPRODUCTION ANALYSIS

✓ A1 result hash verified
✓ Original frozen result hash verified
✓ Both files contain exactly 120 unique matching case IDs

--------------------------------------------------------------------------------------------------------------
FROZEN BASELINE vs NEW A1
--------------------------------------------------------------------------------------------------------------
Metric                                  Frozen          New A1           Delta
--------------------------------------------------------------------------------------------------------------
accuracy                              0.841667        0.841667       +0.000000
VAR                                   0.879310        0.879310       +0.000000
IRR                                   0.806452        0.806452       +0.000000
rejection_precision                   0.877193        0.877193       +0.000000
rejection_recall                      0.806452        0.806452       

/tmp/ipykernel_2438/1869546554.py:69: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  .fillna(False)
/tmp/ipykernel_2438/1869546554.py:69: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  .fillna(False)
/tmp/ipykernel_2438/1869546554.py:69: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  .fillna(False)
/tmp/ipykernel_2438/1869546554.py:69: Futur

In [17]:
from pathlib import Path
from datetime import datetime, timezone
import hashlib
import json

ROOT = Path("/content/deterministic-intent-translation-journal")

EXP_A = (
    ROOT /
    "journal_experiments" /
    "experiment_A_model_sweep"
)

A1_RESULT = (
    EXP_A /
    "results/raw/A1_Qwen3.5-9B_results.csv"
)

A1_LOG = (
    EXP_A /
    "logs/A1_Qwen3.5-9B_execution.log"
)

A1_METRICS = (
    EXP_A /
    "results/metrics/A1_reproduction_metrics.json"
)

A1_COMPARISON = (
    EXP_A /
    "results/analysis/A1_vs_frozen_case_comparison.csv"
)

MANIFEST = (
    EXP_A /
    "manifests/A1_reproduction_manifest_v1.json"
)


def sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(1024 * 1024), b""):
            h.update(block)
    return h.hexdigest()


print("=" * 100)
print("STEP A13 — FREEZE SUCCESSFUL A1 REPRODUCTION")
print("=" * 100)

for p in [
    A1_RESULT,
    A1_LOG,
    A1_METRICS,
    A1_COMPARISON,
]:
    assert p.exists(), f"STOP: missing {p}"

assert not MANIFEST.exists(), (
    "STOP: A1 reproduction manifest already exists."
)

metrics = json.loads(
    A1_METRICS.read_text(encoding="utf-8")
)

assert metrics["reproduction_gate_passed"] is True
assert metrics["metric_reproduction_exact"] is True
assert metrics["decision_reproduction_exact"] is True
assert metrics["llm_path_reproduction_exact"] is True

record = {
    "manifest_id":
        "experiment-A-A1-reproduction-v1",

    "created_utc":
        datetime.now(timezone.utc).isoformat(),

    "arm": "A1",

    "model": "Qwen/Qwen3.5-9B",

    "status": "REPRODUCTION_GATE_PASSED",

    "result": {
        "path": str(A1_RESULT.relative_to(ROOT)),
        "sha256": sha256(A1_RESULT),
    },

    "execution_log": {
        "path": str(A1_LOG.relative_to(ROOT)),
        "sha256": sha256(A1_LOG),
    },

    "metrics": {
        "path": str(A1_METRICS.relative_to(ROOT)),
        "sha256": sha256(A1_METRICS),
    },

    "case_comparison": {
        "path": str(A1_COMPARISON.relative_to(ROOT)),
        "sha256": sha256(A1_COMPARISON),
    },

    "reproduction": {
        "accuracy": {
            "numerator": 101,
            "denominator": 120,
            "value": 101 / 120,
        },

        "VAR": {
            "numerator": 51,
            "denominator": 58,
            "value": 51 / 58,
        },

        "IRR": {
            "numerator": 50,
            "denominator": 62,
            "value": 50 / 62,
        },

        "decision_changes_vs_frozen": 0,
        "llm_path_changes_vs_frozen": 0,

        "exact_metric_reproduction": True,
        "exact_decision_reproduction": True,
    },

    "experimental_decision":
        "A1 reproduction gate passed; A2-A6 are eligible for execution.",
}

MANIFEST.write_text(
    json.dumps(record, indent=2) + "\n",
    encoding="utf-8"
)

print("\nA1 result SHA-256:")
print(sha256(A1_RESULT))

print("\nA1 metrics SHA-256:")
print(sha256(A1_METRICS))

print("\nA1 comparison SHA-256:")
print(sha256(A1_COMPARISON))

print("\nA1 reproduction manifest:")
print(MANIFEST)

print("SHA-256:")
print(sha256(MANIFEST))

print("\n" + "=" * 100)
print("✓ A1 REPRODUCTION EVIDENCE FROZEN")
print("✓ REPRODUCTION GATE PASSED")
print("✓ A2 IS NOW ELIGIBLE FOR EXECUTION")
print("=" * 100)

STEP A13 — FREEZE SUCCESSFUL A1 REPRODUCTION

A1 result SHA-256:
c22618b2750de7f5ef1824fa4036a901ef54f611ad162fcd797b20d79e2b4e06

A1 metrics SHA-256:
0d914b9432bff8861a4b9e87bc64416f118e7e0bb9a2c263f4a659cbf74cc0a6

A1 comparison SHA-256:
3476fc6c29767f42d894eeaef56508b5c2a095a23d3857234e16fa4cd23a6fd2

A1 reproduction manifest:
/content/deterministic-intent-translation-journal/journal_experiments/experiment_A_model_sweep/manifests/A1_reproduction_manifest_v1.json
SHA-256:
cbc014160f2233ed74f4f9a61b181a5045654f1ad6b143b485e226095b6480bb

✓ A1 REPRODUCTION EVIDENCE FROZEN
✓ REPRODUCTION GATE PASSED
✓ A2 IS NOW ELIGIBLE FOR EXECUTION


In [18]:
import subprocess
from pathlib import Path
import hashlib
import os
import time

ROOT = Path("/content/deterministic-intent-translation-journal")

EXP_A = (
    ROOT /
    "journal_experiments" /
    "experiment_A_model_sweep"
)

RUNNER = ROOT / (
    "experiment-2b/implementation/e3_v3/"
    "run_experiment2b_e3v3_r7_dev_v1.py"
)

DATASET = ROOT / (
    "experiment-2b/reproducibility/"
    "e3v3_heldout_benchmark_v1.0/"
    "e3v3_heldout_benchmark_120_v1.csv"
)

A1_MANIFEST = (
    EXP_A /
    "manifests/A1_reproduction_manifest_v1.json"
)

A2_OUT = (
    EXP_A /
    "results/raw/A2_Llama-3.3-70B-Instruct-Turbo_results.csv"
)

A2_LOG = (
    EXP_A /
    "logs/A2_Llama-3.3-70B-Instruct-Turbo_execution.log"
)

MODEL = "meta-llama/Llama-3.3-70B-Instruct-Turbo"

EXPECTED_RUNNER_SHA = (
    "40c035cb864978c718a72fcfdfbb164d262529960b0ae78242d5194147cd8257"
)

EXPECTED_DATASET_SHA = (
    "84576e7d99a2721e61ded5a14bdafeb96707b068397243bbed32aa735496d4de"
)

EXPECTED_A1_MANIFEST_SHA = (
    "cbc014160f2233ed74f4f9a61b181a5045654f1ad6b143b485e226095b6480bb"
)


def sha256(path):
    h = hashlib.sha256()

    with open(path, "rb") as f:
        for block in iter(lambda: f.read(1024 * 1024), b""):
            h.update(block)

    return h.hexdigest()


print("=" * 100)
print("STEP A14 — EXECUTE A2 MODEL-SWEEP ARM")
print("=" * 100)

# ------------------------------------------------------------------
# Integrity checks
# ------------------------------------------------------------------

assert RUNNER.exists()
assert DATASET.exists()
assert A1_MANIFEST.exists()

assert sha256(RUNNER) == EXPECTED_RUNNER_SHA, (
    "STOP: frozen runner changed."
)

assert sha256(DATASET) == EXPECTED_DATASET_SHA, (
    "STOP: frozen benchmark changed."
)

assert sha256(A1_MANIFEST) == EXPECTED_A1_MANIFEST_SHA, (
    "STOP: A1 reproduction manifest changed."
)

assert os.environ.get("TOGETHER_API_KEY"), (
    "STOP: TOGETHER_API_KEY missing."
)

assert not A2_OUT.exists(), (
    f"STOP: A2 result already exists:\n{A2_OUT}\n"
    "Do not overwrite or rerun."
)

assert not A2_LOG.exists(), (
    f"STOP: A2 log already exists:\n{A2_LOG}\n"
    "Do not overwrite or rerun."
)

print("\n✓ Frozen runner verified")
print("✓ Frozen benchmark verified")
print("✓ Frozen A1 reproduction manifest verified")
print("✓ TOGETHER_API_KEY present")
print("✓ A2 result path clean")
print("✓ A2 log path clean")

# ------------------------------------------------------------------
# Exact pre-registered model
# ------------------------------------------------------------------

print("\nPRE-REGISTERED A2")
print("-" * 100)

print("Model :", MODEL)
print("Cases : 120")

print("\nOnly experimental variable being changed:")
print("  A1 model -> A2 model")

print("\nUnchanged:")
print("  ✓ runner")
print("  ✓ benchmark")
print("  ✓ prompt")
print("  ✓ deterministic gates")
print("  ✓ thresholds")
print("  ✓ labels")

# ------------------------------------------------------------------
# Execute once
# ------------------------------------------------------------------

cmd = [
    "python",
    str(RUNNER),
    "--dataset",
    str(DATASET),
    "--out",
    str(A2_OUT),
    "--model",
    MODEL,
]

print("\n" + "-" * 100)
print("EXECUTING A2")
print("-" * 100)

print("\nCommand:")
print(" ".join(f'"{x}"' if " " in x else x for x in cmd))

print("\nRunning frozen 120-case benchmark ONCE...\n")

t0 = time.time()

process = subprocess.run(
    cmd,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    env=os.environ.copy(),
)

elapsed = time.time() - t0

# Preserve console evidence even on failure
A2_LOG.write_text(
    process.stdout or "",
    encoding="utf-8"
)

print(process.stdout)

print("\n" + "-" * 100)
print("EXECUTION STATUS")
print("-" * 100)

print("Process return code :", process.returncode)
print(f"Wall-clock seconds  : {elapsed:.2f}")

print("\nExecution log:")
print(A2_LOG)

print("Log SHA-256:")
print(sha256(A2_LOG))

# ------------------------------------------------------------------
# Failure policy
# ------------------------------------------------------------------

if process.returncode != 0:

    print("\n" + "=" * 100)
    print("✗ A2 EXECUTION FAILED")
    print("✗ FAILURE LOG HAS BEEN PRESERVED")
    print("✗ DO NOT RERUN A2")
    print("✗ DO NOT START A3")
    print("✗ SEND THIS OUTPUT FOR DIAGNOSIS")
    print("=" * 100)

    raise RuntimeError(
        f"A2 runner exited with return code {process.returncode}"
    )

# ------------------------------------------------------------------
# Successful execution checks
# ------------------------------------------------------------------

assert A2_OUT.exists(), (
    "STOP: A2 process returned success but result CSV is missing."
)

import pandas as pd

df = pd.read_csv(A2_OUT)

assert len(df) == 120, (
    f"STOP: Expected 120 rows, found {len(df)}."
)

assert df["id"].is_unique, (
    "STOP: A2 result contains duplicate case IDs."
)

print("\nA2 RESULT")
print("-" * 100)

print("Rows       :", len(df))
print("Unique IDs :", df["id"].nunique())

print("\nResult:")
print(A2_OUT)

print("Result SHA-256:")
print(sha256(A2_OUT))

print("\n" + "=" * 100)
print("✓ A2 EXECUTION COMPLETED")
print("✓ 120 CASES WRITTEN")
print("✓ RAW A2 RESULT PRESERVED")
print("✓ EXECUTION LOG PRESERVED")
print("✓ DO NOT RUN A3 YET")
print("=" * 100)

STEP A14 — EXECUTE A2 MODEL-SWEEP ARM

✓ Frozen runner verified
✓ Frozen benchmark verified
✓ Frozen A1 reproduction manifest verified
✓ TOGETHER_API_KEY present
✓ A2 result path clean
✓ A2 log path clean

PRE-REGISTERED A2
----------------------------------------------------------------------------------------------------
Model : meta-llama/Llama-3.3-70B-Instruct-Turbo
Cases : 120

Only experimental variable being changed:
  A1 model -> A2 model

Unchanged:
  ✓ runner
  ✓ benchmark
  ✓ prompt
  ✓ deterministic gates
  ✓ thresholds
  ✓ labels

----------------------------------------------------------------------------------------------------
EXECUTING A2
----------------------------------------------------------------------------------------------------

Command:
python /content/deterministic-intent-translation-journal/experiment-2b/implementation/e3_v3/run_experiment2b_e3v3_r7_dev_v1.py --dataset /content/deterministic-intent-translation-journal/experiment-2b/reproducibility/e3v3_hel

In [19]:
from pathlib import Path
import pandas as pd
import hashlib
import json
from scipy.stats import beta

ROOT = Path("/content/deterministic-intent-translation-journal")
EXP_A = ROOT / "journal_experiments/experiment_A_model_sweep"

A1 = EXP_A / "results/raw/A1_Qwen3.5-9B_results.csv"
A2 = EXP_A / "results/raw/A2_Llama-3.3-70B-Instruct-Turbo_results.csv"

METRICS_OUT = EXP_A / "results/metrics/A2_metrics.json"
COMPARISON_OUT = EXP_A / "results/analysis/A2_vs_A1_case_comparison.csv"

EXPECTED_A1_SHA = (
    "c22618b2750de7f5ef1824fa4036a901ef54f611ad162fcd797b20d79e2b4e06"
)
EXPECTED_A2_SHA = (
    "5eba3be64426c14ac1a917be5392e4b6a0a487f3bf9be81c076d7f4b429380d7"
)


def sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(1024 * 1024), b""):
            h.update(block)
    return h.hexdigest()


def as_bool(series):
    if pd.api.types.is_bool_dtype(series):
        return series.astype("boolean").fillna(False).astype(bool)

    mapped = (
        series.astype(str)
        .str.strip()
        .str.lower()
        .map({
            "true": True,
            "false": False,
            "1": True,
            "0": False
        })
    )

    return mapped.astype("boolean").fillna(False).astype(bool)


def clopper_pearson(k, n, alpha=0.05):
    if n == 0:
        return None, None

    lower = 0.0 if k == 0 else beta.ppf(alpha / 2, k, n - k + 1)
    upper = 1.0 if k == n else beta.ppf(1 - alpha / 2, k + 1, n - k)

    return float(lower), float(upper)


def calculate_metrics(df):
    expected_valid = as_bool(df["expected_valid"])
    accepted = as_bool(df["accepted"])
    correct = as_bool(df["decision_correct"])
    llm = as_bool(df["llm_invoked"])
    parse = as_bool(df["parse_ok"])

    valid = expected_valid
    invalid = ~expected_valid

    total = len(df)
    valid_n = int(valid.sum())
    invalid_n = int(invalid.sum())

    correct_n = int(correct.sum())

    valid_accepted = int((valid & accepted).sum())
    valid_rejected = int((valid & ~accepted).sum())

    invalid_rejected = int((invalid & ~accepted).sum())
    invalid_accepted = int((invalid & accepted).sum())

    # invalid/reject positive
    TP = invalid_rejected
    FP = valid_rejected
    FN = invalid_accepted
    TN = valid_accepted

    precision = TP / (TP + FP) if TP + FP else 0.0
    recall = TP / (TP + FN) if TP + FN else 0.0

    f1 = (
        2 * precision * recall / (precision + recall)
        if precision + recall
        else 0.0
    )

    accuracy = correct_n / total
    var = valid_accepted / valid_n
    irr = invalid_rejected / invalid_n

    acc_ci = clopper_pearson(correct_n, total)
    var_ci = clopper_pearson(valid_accepted, valid_n)
    irr_ci = clopper_pearson(invalid_rejected, invalid_n)

    descriptors = int(
        df["descriptor_json"]
        .fillna("")
        .astype(str)
        .str.strip()
        .ne("")
        .sum()
    )

    return {
        "total": total,
        "valid": valid_n,
        "invalid": invalid_n,

        "correct": correct_n,
        "incorrect": total - correct_n,

        "valid_accepted": valid_accepted,
        "valid_rejected": valid_rejected,

        "invalid_rejected": invalid_rejected,
        "invalid_accepted": invalid_accepted,

        "TP": TP,
        "FP": FP,
        "FN": FN,
        "TN": TN,

        "accuracy": accuracy,
        "accuracy_ci95": list(acc_ci),

        "VAR": var,
        "VAR_ci95": list(var_ci),

        "IRR": irr,
        "IRR_ci95": list(irr_ci),

        "rejection_precision": precision,
        "rejection_recall": recall,
        "rejection_f1": f1,

        "llm_invoked": int(llm.sum()),
        "parse_success": int((llm & parse).sum()),

        "descriptor_count": descriptors,
    }


print("=" * 110)
print("STEP A15 — A2 METRIC AND CASE-LEVEL ANALYSIS")
print("=" * 110)

# ------------------------------------------------------------------
# Integrity
# ------------------------------------------------------------------

assert A1.exists()
assert A2.exists()

assert sha256(A1) == EXPECTED_A1_SHA, "STOP: A1 changed."
assert sha256(A2) == EXPECTED_A2_SHA, "STOP: A2 changed."

assert not METRICS_OUT.exists(), "STOP: A2 metrics already exist."
assert not COMPARISON_OUT.exists(), "STOP: A2 comparison already exists."

a1 = pd.read_csv(A1)
a2 = pd.read_csv(A2)

assert len(a1) == 120
assert len(a2) == 120

assert a1["id"].is_unique
assert a2["id"].is_unique

assert set(a1["id"]) == set(a2["id"])

print("\n✓ A1 hash verified")
print("✓ A2 hash verified")
print("✓ 120 matching unique case IDs")

# ------------------------------------------------------------------
# Metrics
# ------------------------------------------------------------------

m1 = calculate_metrics(a1)
m2 = calculate_metrics(a2)

print("\n" + "-" * 110)
print("A1 vs A2 PERFORMANCE")
print("-" * 110)

print(
    f"{'Metric':28s}"
    f"{'A1':>16s}"
    f"{'A2':>16s}"
    f"{'Delta':>16s}"
)

print("-" * 110)

for key in [
    "accuracy",
    "VAR",
    "IRR",
    "rejection_precision",
    "rejection_recall",
    "rejection_f1",
]:
    print(
        f"{key:28s}"
        f"{m1[key]:16.6f}"
        f"{m2[key]:16.6f}"
        f"{m2[key]-m1[key]:+16.6f}"
    )

print("\nA2 EXACT 95% CLOPPER-PEARSON CIs")
print("-" * 110)

print(
    f"Accuracy : {m2['correct']}/{m2['total']} "
    f"= {100*m2['accuracy']:.2f}% "
    f"[{100*m2['accuracy_ci95'][0]:.2f}%, "
    f"{100*m2['accuracy_ci95'][1]:.2f}%]"
)

print(
    f"VAR      : {m2['valid_accepted']}/{m2['valid']} "
    f"= {100*m2['VAR']:.2f}% "
    f"[{100*m2['VAR_ci95'][0]:.2f}%, "
    f"{100*m2['VAR_ci95'][1]:.2f}%]"
)

print(
    f"IRR      : {m2['invalid_rejected']}/{m2['invalid']} "
    f"= {100*m2['IRR']:.2f}% "
    f"[{100*m2['IRR_ci95'][0]:.2f}%, "
    f"{100*m2['IRR_ci95'][1]:.2f}%]"
)

print("\nA2 COUNTS")
print("-" * 110)

for key in [
    "correct",
    "incorrect",
    "valid_accepted",
    "valid_rejected",
    "invalid_rejected",
    "invalid_accepted",
    "TP",
    "FP",
    "FN",
    "TN",
    "llm_invoked",
    "parse_success",
    "descriptor_count",
]:
    print(f"{key:28s}: {m2[key]}")

# ------------------------------------------------------------------
# Case-level comparison
# ------------------------------------------------------------------

left = a1[
    [
        "id",
        "expected_valid",
        "accepted",
        "decision_correct",
        "llm_invoked",
        "parse_ok",
        "rejection_stage",
        "error_codes_json",
    ]
].copy()

right = a2[
    [
        "id",
        "accepted",
        "decision_correct",
        "llm_invoked",
        "parse_ok",
        "rejection_stage",
        "error_codes_json",
    ]
].copy()

left = left.rename(columns={
    "accepted": "A1_accepted",
    "decision_correct": "A1_correct",
    "llm_invoked": "A1_llm_invoked",
    "parse_ok": "A1_parse_ok",
    "rejection_stage": "A1_rejection_stage",
    "error_codes_json": "A1_error_codes",
})

right = right.rename(columns={
    "accepted": "A2_accepted",
    "decision_correct": "A2_correct",
    "llm_invoked": "A2_llm_invoked",
    "parse_ok": "A2_parse_ok",
    "rejection_stage": "A2_rejection_stage",
    "error_codes_json": "A2_error_codes",
})

comparison = left.merge(
    right,
    on="id",
    how="inner",
    validate="one_to_one",
)

for col in [
    "A1_accepted",
    "A2_accepted",
    "A1_correct",
    "A2_correct",
    "A1_llm_invoked",
    "A2_llm_invoked",
    "A1_parse_ok",
    "A2_parse_ok",
]:
    comparison[col] = as_bool(comparison[col])

comparison["decision_changed"] = (
    comparison["A1_accepted"] != comparison["A2_accepted"]
)

comparison["correctness_changed"] = (
    comparison["A1_correct"] != comparison["A2_correct"]
)

comparison["llm_path_changed"] = (
    comparison["A1_llm_invoked"] != comparison["A2_llm_invoked"]
)

comparison["A1_wrong_A2_correct"] = (
    (~comparison["A1_correct"]) &
    comparison["A2_correct"]
)

comparison["A1_correct_A2_wrong"] = (
    comparison["A1_correct"] &
    (~comparison["A2_correct"])
)

decision_changes = int(comparison["decision_changed"].sum())
correctness_changes = int(comparison["correctness_changed"].sum())
path_changes = int(comparison["llm_path_changed"].sum())

improved = int(comparison["A1_wrong_A2_correct"].sum())
regressed = int(comparison["A1_correct_A2_wrong"].sum())

print("\nCASE-LEVEL A1 ↔ A2 DIFFERENCES")
print("-" * 110)

print("Acceptance decisions changed :", decision_changes)
print("Correctness changed          :", correctness_changes)
print("A1 wrong -> A2 correct       :", improved)
print("A1 correct -> A2 wrong       :", regressed)
print("LLM invocation path changed  :", path_changes)

changed = comparison[
    comparison["decision_changed"] |
    comparison["correctness_changed"] |
    comparison["llm_path_changed"]
]

if len(changed):

    print("\nCHANGED CASES")
    print("-" * 110)

    print(
        changed[
            [
                "id",
                "expected_valid",
                "A1_accepted",
                "A2_accepted",
                "A1_correct",
                "A2_correct",
                "A1_llm_invoked",
                "A2_llm_invoked",
                "A1_rejection_stage",
                "A2_rejection_stage",
            ]
        ].to_string(index=False)
    )

else:
    print("\nNo A1/A2 decision or path differences.")

# ------------------------------------------------------------------
# Save evidence
# ------------------------------------------------------------------

comparison.to_csv(
    COMPARISON_OUT,
    index=False
)

record = {
    "arm": "A2",
    "model": "meta-llama/Llama-3.3-70B-Instruct-Turbo",

    "result_sha256": sha256(A2),

    "metrics": m2,

    "comparison_with_A1": {
        "decision_changes": decision_changes,
        "correctness_changes": correctness_changes,
        "A1_wrong_A2_correct": improved,
        "A1_correct_A2_wrong": regressed,
        "llm_path_changes": path_changes,
    },

    "containment": {
        "status": "PENDING_IMPLEMENTATION-GROUNDED_ANALYSIS",
        "note": (
            "Containment is not inferred from expected_valid or "
            "accepted. Deterministic gate semantics will be derived "
            "from the frozen implementation before calculating "
            "containment violations."
        )
    }
}

METRICS_OUT.write_text(
    json.dumps(record, indent=2) + "\n",
    encoding="utf-8"
)

print("\nARTIFACTS")
print("-" * 110)

print("Metrics:")
print(METRICS_OUT)
print("SHA-256:", sha256(METRICS_OUT))

print("\nCase comparison:")
print(COMPARISON_OUT)
print("SHA-256:", sha256(COMPARISON_OUT))

print("\n" + "=" * 110)
print("✓ A2 ANALYSIS COMPLETE")
print("✓ EXACT 95% CIs CALCULATED")
print("✓ CASE-LEVEL COMPARISON SAVED")
print("✓ CONTAINMENT ANALYSIS INTENTIONALLY PENDING")
print("✓ NO MODEL/API CALL MADE")
print("✓ DO NOT RUN A3 YET")
print("=" * 110)

STEP A15 — A2 METRIC AND CASE-LEVEL ANALYSIS

✓ A1 hash verified
✓ A2 hash verified
✓ 120 matching unique case IDs

--------------------------------------------------------------------------------------------------------------
A1 vs A2 PERFORMANCE
--------------------------------------------------------------------------------------------------------------
Metric                                    A1              A2           Delta
--------------------------------------------------------------------------------------------------------------
accuracy                            0.841667        0.833333       -0.008333
VAR                                 0.879310        0.827586       -0.051724
IRR                                 0.806452        0.838710       +0.032258
rejection_precision                 0.877193        0.838710       -0.038483
rejection_recall                    0.806452        0.838710       +0.032258
rejection_f1                        0.840336        0.838710       -

In [20]:
from pathlib import Path
import hashlib
import re
import pandas as pd

ROOT = Path("/content/deterministic-intent-translation-journal")

RUNNER = ROOT / (
    "experiment-2b/implementation/e3_v3/"
    "run_experiment2b_e3v3_r7_dev_v1.py"
)

A1 = ROOT / (
    "journal_experiments/experiment_A_model_sweep/"
    "results/raw/A1_Qwen3.5-9B_results.csv"
)

A2 = ROOT / (
    "journal_experiments/experiment_A_model_sweep/"
    "results/raw/A2_Llama-3.3-70B-Instruct-Turbo_results.csv"
)

EXPECTED_RUNNER_SHA = (
    "40c035cb864978c718a72fcfdfbb164d262529960b0ae78242d5194147cd8257"
)

def sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(1024 * 1024), b""):
            h.update(block)
    return h.hexdigest()


print("=" * 110)
print("STEP A16 — IMPLEMENTATION-GROUNDED CONTAINMENT INSPECTION")
print("=" * 110)

assert sha256(RUNNER) == EXPECTED_RUNNER_SHA

source_lines = RUNNER.read_text(
    encoding="utf-8"
).splitlines()

print("\n✓ Frozen runner verified")

# ------------------------------------------------------------------
# 1. Locate gate / rejection / descriptor logic in frozen runner
# ------------------------------------------------------------------

terms = [
    "pre_guard",
    "post_guard",
    "cir_validation",
    "preservation",
    "rejection_stage",
    "accepted",
    "descriptor",
    "error_codes",
]

hits = []

for no, line in enumerate(source_lines, start=1):
    lower = line.lower()

    if any(term.lower() in lower for term in terms):
        hits.append(no)

# Collapse hits into useful source windows
windows = []

for hit in hits:
    start = max(1, hit - 4)
    end = min(len(source_lines), hit + 6)

    if not windows or start > windows[-1][1] + 1:
        windows.append([start, end])
    else:
        windows[-1][1] = max(windows[-1][1], end)

print("\nFROZEN RUNNER — RELEVANT SOURCE WINDOWS")
print("-" * 110)

for start, end in windows:
    print(f"\n--- lines {start}-{end} ---")

    for no in range(start, end + 1):
        print(f"{no:04d}: {source_lines[no-1]}")

# ------------------------------------------------------------------
# 2. Inspect result columns that may encode deterministic failures
# ------------------------------------------------------------------

a1 = pd.read_csv(A1)
a2 = pd.read_csv(A2)

candidate_columns = [
    c for c in a1.columns
    if any(
        term in c.lower()
        for term in [
            "guard",
            "validation",
            "preservation",
            "rejection",
            "descriptor",
            "audit",
            "reconciliation",
        ]
    )
]

print("\n" + "=" * 110)
print("CANDIDATE RESULT COLUMNS")
print("=" * 110)

for c in candidate_columns:
    print(c)

# ------------------------------------------------------------------
# 3. Rejection-stage distributions
# ------------------------------------------------------------------

print("\n" + "=" * 110)
print("A1 REJECTION STAGES")
print("=" * 110)

print(
    a1["rejection_stage"]
    .fillna("<ACCEPTED>")
    .value_counts(dropna=False)
    .to_string()
)

print("\n" + "=" * 110)
print("A2 REJECTION STAGES")
print("=" * 110)

print(
    a2["rejection_stage"]
    .fillna("<ACCEPTED>")
    .value_counts(dropna=False)
    .to_string()
)

# ------------------------------------------------------------------
# 4. Check relationship between accepted and descriptor emission
# ------------------------------------------------------------------

def descriptor_present(df):
    return (
        df["descriptor_json"]
        .fillna("")
        .astype(str)
        .str.strip()
        .ne("")
    )

for label, df in [("A1", a1), ("A2", a2)]:

    accepted = (
        df["accepted"]
        .astype(str)
        .str.lower()
        .eq("true")
    )

    descriptor = descriptor_present(df)

    print("\n" + "=" * 110)
    print(f"{label} ACCEPTANCE / DESCRIPTOR RELATION")
    print("=" * 110)

    print("Accepted rows                     :", int(accepted.sum()))
    print("Descriptor-present rows           :", int(descriptor.sum()))

    print(
        "Rejected but descriptor present   :",
        int((~accepted & descriptor).sum())
    )

    print(
        "Accepted but descriptor absent    :",
        int((accepted & ~descriptor).sum())
    )

    violations = df.loc[
        (~accepted & descriptor),
        [
            "id",
            "expected_valid",
            "accepted",
            "rejection_stage",
            "error_codes_json",
            "descriptor_json",
        ],
    ]

    if len(violations):
        print("\nRejected rows containing descriptors:")
        print(violations.to_string(index=False))
    else:
        print("\n✓ No rejected row contains a descriptor")

print("\n" + "=" * 110)
print("✓ IMPLEMENTATION INSPECTION COMPLETE")
print("✓ NO CONTAINMENT DEFINITION CHANGED")
print("✓ NO MODEL/API CALL MADE")
print("✓ NO FILE MODIFIED")
print("=" * 110)

STEP A16 — IMPLEMENTATION-GROUNDED CONTAINMENT INSPECTION

✓ Frozen runner verified

FROZEN RUNNER — RELEVANT SOURCE WINDOWS
--------------------------------------------------------------------------------------------------------------

--- lines 14-24 ---
0014:     3. output pipeline label is e3_v3.
0015:     4. dataset loop executes only e3_v3.
0016:     5. historical runner remains unchanged.
0017: 
0018: No algorithmic CIR/LLM/validation/descriptor logic is intentionally
0019: changed in this development harness.
0020: """
0021: 
0022: 
0023: import argparse
0024: import importlib.util

--- lines 55-65 ---
0055: from src.llm.together_adapter import call_json
0056: from src.input_audit import audit_input
0057: from src.validator import validate_cir
0058: from src.standards.mapper import map_standards
0059: from src.descriptor import compile_descriptor
0060: 
0061: 
0062: # ============================================================
0063: # R5 deterministic source-extraction repair


In [21]:
from pathlib import Path
from datetime import datetime, timezone
import pandas as pd
import hashlib
import json

ROOT = Path("/content/deterministic-intent-translation-journal")
EXP_A = ROOT / "journal_experiments/experiment_A_model_sweep"

RUNNER = ROOT / (
    "experiment-2b/implementation/e3_v3/"
    "run_experiment2b_e3v3_r7_dev_v1.py"
)

A1 = EXP_A / "results/raw/A1_Qwen3.5-9B_results.csv"
A2 = EXP_A / "results/raw/A2_Llama-3.3-70B-Instruct-Turbo_results.csv"

A2_METRICS = EXP_A / "results/metrics/A2_metrics.json"
A2_COMPARISON = EXP_A / "results/analysis/A2_vs_A1_case_comparison.csv"

OUT = EXP_A / "manifests/containment_definition_A1_A2_v1.json"

EXPECTED_RUNNER_SHA = (
    "40c035cb864978c718a72fcfdfbb164d262529960b0ae78242d5194147cd8257"
)

EXPECTED_A1_SHA = (
    "c22618b2750de7f5ef1824fa4036a901ef54f611ad162fcd797b20d79e2b4e06"
)

EXPECTED_A2_SHA = (
    "5eba3be64426c14ac1a917be5392e4b6a0a487f3bf9be81c076d7f4b429380d7"
)

EXPECTED_A2_METRICS_SHA = (
    "4cae8cfdde983279052f37083599699a2431234b308ca19d278261525f14315f"
)

EXPECTED_A2_COMPARISON_SHA = (
    "01fcea9941d0dcea7c4e90ad1aecc8e39b9fb21c3b4fc1e41ee165283bac5474"
)


def sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(1024 * 1024), b""):
            h.update(block)
    return h.hexdigest()


def bool_series(s):
    if pd.api.types.is_bool_dtype(s):
        return s.astype("boolean").fillna(False).astype(bool)

    return (
        s.astype(str)
        .str.strip()
        .str.lower()
        .map({
            "true": True,
            "false": False,
            "1": True,
            "0": False,
        })
        .astype("boolean")
        .fillna(False)
        .astype(bool)
    )


def descriptor_present(df):
    # json_text(None) is expected to appear as empty/NaN in these
    # frozen result files; verify against acceptance below.
    return (
        df["descriptor_json"]
        .fillna("")
        .astype(str)
        .str.strip()
        .ne("")
    )


def containment_stats(path):
    df = pd.read_csv(path)

    assert len(df) == 120
    assert df["id"].is_unique

    accepted = bool_series(df["accepted"])
    descriptor = descriptor_present(df)

    rejected = ~accepted

    violations = rejected & descriptor
    accepted_without_descriptor = accepted & ~descriptor

    return {
        "rows": int(len(df)),
        "accepted": int(accepted.sum()),
        "rejected": int(rejected.sum()),
        "descriptor_present": int(descriptor.sum()),

        "rejected_with_descriptor":
            int(violations.sum()),

        "accepted_without_descriptor":
            int(accepted_without_descriptor.sum()),

        "violation_case_ids":
            df.loc[violations, "id"].astype(str).tolist(),

        "accepted_without_descriptor_case_ids":
            df.loc[
                accepted_without_descriptor,
                "id"
            ].astype(str).tolist(),
    }


print("=" * 110)
print("STEP A17 — FREEZE CONTAINMENT DEFINITION AND A1/A2 EVIDENCE")
print("=" * 110)

# ------------------------------------------------------------------
# Integrity checks
# ------------------------------------------------------------------

assert sha256(RUNNER) == EXPECTED_RUNNER_SHA, (
    "STOP: frozen runner changed."
)

assert sha256(A1) == EXPECTED_A1_SHA, (
    "STOP: A1 result changed."
)

assert sha256(A2) == EXPECTED_A2_SHA, (
    "STOP: A2 result changed."
)

assert sha256(A2_METRICS) == EXPECTED_A2_METRICS_SHA, (
    "STOP: A2 metrics changed."
)

assert sha256(A2_COMPARISON) == EXPECTED_A2_COMPARISON_SHA, (
    "STOP: A2 comparison changed."
)

assert not OUT.exists(), (
    "STOP: containment manifest already exists."
)

print("\n✓ Frozen runner verified")
print("✓ A1 result verified")
print("✓ A2 result verified")
print("✓ A2 metrics verified")
print("✓ A2 comparison verified")

# ------------------------------------------------------------------
# Calculate containment endpoint
# ------------------------------------------------------------------

a1_stats = containment_stats(A1)
a2_stats = containment_stats(A2)

print("\n" + "-" * 110)
print("CONTAINMENT RESULTS")
print("-" * 110)

for arm, stats in [
    ("A1", a1_stats),
    ("A2", a2_stats),
]:
    print(f"\n{arm}")
    print("  accepted                   :", stats["accepted"])
    print("  rejected                   :", stats["rejected"])
    print("  descriptor present         :", stats["descriptor_present"])
    print(
        "  rejected with descriptor   :",
        stats["rejected_with_descriptor"]
    )
    print(
        "  accepted without descriptor :",
        stats["accepted_without_descriptor"]
    )

# Strong observed invariant for completed arms
assert a1_stats["rejected_with_descriptor"] == 0
assert a2_stats["rejected_with_descriptor"] == 0

assert a1_stats["accepted_without_descriptor"] == 0
assert a2_stats["accepted_without_descriptor"] == 0

# ------------------------------------------------------------------
# Freeze operational definition
# ------------------------------------------------------------------

record = {
    "manifest_id":
        "experiment-A-containment-definition-A1-A2-v1",

    "created_utc":
        datetime.now(timezone.utc).isoformat(),

    "runner": {
        "path": str(RUNNER.relative_to(ROOT)),
        "sha256": sha256(RUNNER),
    },

    "operational_definition": {
        "containment_violation": (
            "A result row for which the frozen E3-v3 pipeline "
            "returns accepted=False but descriptor_json is present."
        ),

        "rationale": (
            "In the frozen E3-v3 control flow, deterministic "
            "rejection paths return before Stage 5 standards "
            "mapping and descriptor compilation. Descriptor "
            "compilation occurs only after the preceding "
            "pre-actuation rejection checks pass."
        ),

        "important_distinction": (
            "expected_valid is a benchmark ground-truth label and "
            "is not used as the containment-gate indicator."
        ),

        "expected_violations_per_arm": 0,
    },

    "A1": {
        "model": "Qwen/Qwen3.5-9B",
        "result_sha256": sha256(A1),
        **a1_stats,
    },

    "A2": {
        "model":
            "meta-llama/Llama-3.3-70B-Instruct-Turbo",
        "result_sha256": sha256(A2),
        **a2_stats,
    },

    "current_observation": {
        "A1_containment_violations":
            a1_stats["rejected_with_descriptor"],

        "A2_containment_violations":
            a2_stats["rejected_with_descriptor"],

        "completed_arms_with_zero_violations":
            2,

        "claim_scope": (
            "Empirical evidence currently covers A1 and A2 only. "
            "No claim is made yet for A3-A6."
        ),
    },
}

OUT.write_text(
    json.dumps(record, indent=2) + "\n",
    encoding="utf-8"
)

print("\n" + "-" * 110)
print("FROZEN OPERATIONAL DEFINITION")
print("-" * 110)

print(
    "Containment violation = accepted=False "
    "AND descriptor_json present"
)

print("\nA1 containment violations :",
      a1_stats["rejected_with_descriptor"])

print("A2 containment violations :",
      a2_stats["rejected_with_descriptor"])

print("\nManifest:")
print(OUT)

print("SHA-256:")
print(sha256(OUT))

print("\n" + "=" * 110)
print("✓ CONTAINMENT DEFINITION FROZEN")
print("✓ A1 CONTAINMENT VIOLATIONS = 0")
print("✓ A2 CONTAINMENT VIOLATIONS = 0")
print("✓ CURRENT CLAIM LIMITED TO A1/A2")
print("✓ NO MODEL/API CALL MADE")
print("✓ A3 ELIGIBLE AFTER THIS STEP PASSES")
print("=" * 110)

STEP A17 — FREEZE CONTAINMENT DEFINITION AND A1/A2 EVIDENCE

✓ Frozen runner verified
✓ A1 result verified
✓ A2 result verified
✓ A2 metrics verified
✓ A2 comparison verified

--------------------------------------------------------------------------------------------------------------
CONTAINMENT RESULTS
--------------------------------------------------------------------------------------------------------------

A1
  accepted                   : 63
  rejected                   : 57
  descriptor present         : 63
  rejected with descriptor   : 0
  accepted without descriptor : 0

A2
  accepted                   : 58
  rejected                   : 62
  descriptor present         : 58
  rejected with descriptor   : 0
  accepted without descriptor : 0

--------------------------------------------------------------------------------------------------------------
FROZEN OPERATIONAL DEFINITION
----------------------------------------------------------------------------------------------

In [22]:
import subprocess
from pathlib import Path
import hashlib
import os
import time
import pandas as pd

ROOT = Path("/content/deterministic-intent-translation-journal")

EXP_A = (
    ROOT /
    "journal_experiments" /
    "experiment_A_model_sweep"
)

RUNNER = ROOT / (
    "experiment-2b/implementation/e3_v3/"
    "run_experiment2b_e3v3_r7_dev_v1.py"
)

DATASET = ROOT / (
    "experiment-2b/reproducibility/"
    "e3v3_heldout_benchmark_v1.0/"
    "e3v3_heldout_benchmark_120_v1.csv"
)

A1_MANIFEST = (
    EXP_A /
    "manifests/A1_reproduction_manifest_v1.json"
)

CONTAINMENT_MANIFEST = (
    EXP_A /
    "manifests/containment_definition_A1_A2_v1.json"
)

A3_OUT = (
    EXP_A /
    "results/raw/A3_Qwen2.5-7B-Instruct-Turbo_results.csv"
)

A3_LOG = (
    EXP_A /
    "logs/A3_Qwen2.5-7B-Instruct-Turbo_execution.log"
)

MODEL = "Qwen/Qwen2.5-7B-Instruct-Turbo"

EXPECTED_RUNNER_SHA = (
    "40c035cb864978c718a72fcfdfbb164d262529960b0ae78242d5194147cd8257"
)

EXPECTED_DATASET_SHA = (
    "84576e7d99a2721e61ded5a14bdafeb96707b068397243bbed32aa735496d4de"
)

EXPECTED_A1_MANIFEST_SHA = (
    "cbc014160f2233ed74f4f9a61b181a5045654f1ad6b143b485e226095b6480bb"
)

EXPECTED_CONTAINMENT_SHA = (
    "cd31ea28ffa8a2f82121d1c4cf0e5478425d194299d309e603069545d75d4832"
)


def sha256(path):
    h = hashlib.sha256()

    with open(path, "rb") as f:
        for block in iter(lambda: f.read(1024 * 1024), b""):
            h.update(block)

    return h.hexdigest()


print("=" * 100)
print("STEP A18 — EXECUTE A3 MODEL-SWEEP ARM")
print("=" * 100)

# ------------------------------------------------------------------
# Integrity checks
# ------------------------------------------------------------------

assert RUNNER.exists()
assert DATASET.exists()
assert A1_MANIFEST.exists()
assert CONTAINMENT_MANIFEST.exists()

assert sha256(RUNNER) == EXPECTED_RUNNER_SHA, (
    "STOP: frozen runner changed."
)

assert sha256(DATASET) == EXPECTED_DATASET_SHA, (
    "STOP: frozen benchmark changed."
)

assert sha256(A1_MANIFEST) == EXPECTED_A1_MANIFEST_SHA, (
    "STOP: A1 reproduction manifest changed."
)

assert sha256(CONTAINMENT_MANIFEST) == EXPECTED_CONTAINMENT_SHA, (
    "STOP: frozen containment definition changed."
)

assert os.environ.get("TOGETHER_API_KEY"), (
    "STOP: TOGETHER_API_KEY missing."
)

assert not A3_OUT.exists(), (
    f"STOP: A3 result already exists:\n{A3_OUT}\n"
    "Do not overwrite or rerun."
)

assert not A3_LOG.exists(), (
    f"STOP: A3 log already exists:\n{A3_LOG}\n"
    "Do not overwrite or rerun."
)

print("\n✓ Frozen runner verified")
print("✓ Frozen benchmark verified")
print("✓ A1 reproduction manifest verified")
print("✓ Frozen containment definition verified")
print("✓ TOGETHER_API_KEY present")
print("✓ A3 result path clean")
print("✓ A3 log path clean")

# ------------------------------------------------------------------
# Pre-registered A3
# ------------------------------------------------------------------

print("\nPRE-REGISTERED A3")
print("-" * 100)

print("Model :", MODEL)
print("Role  : Smaller model arm for model-scale sensitivity")
print("Cases : 120")

print("\nOnly experimental variable being changed:")
print("  model ->", MODEL)

print("\nUnchanged:")
print("  ✓ runner")
print("  ✓ benchmark")
print("  ✓ prompt")
print("  ✓ deterministic gates")
print("  ✓ thresholds")
print("  ✓ labels")
print("  ✓ containment definition")

# ------------------------------------------------------------------
# Execute exactly once
# ------------------------------------------------------------------

cmd = [
    "python",
    str(RUNNER),
    "--dataset",
    str(DATASET),
    "--out",
    str(A3_OUT),
    "--model",
    MODEL,
]

print("\n" + "-" * 100)
print("EXECUTING A3")
print("-" * 100)

print("\nCommand:")
print(" ".join(f'"{x}"' if " " in x else x for x in cmd))

print("\nRunning frozen 120-case benchmark ONCE...\n")

t0 = time.time()

process = subprocess.run(
    cmd,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    env=os.environ.copy(),
)

elapsed = time.time() - t0

# Preserve execution evidence even if endpoint/model fails.
A3_LOG.write_text(
    process.stdout or "",
    encoding="utf-8"
)

print(process.stdout)

print("\n" + "-" * 100)
print("EXECUTION STATUS")
print("-" * 100)

print("Process return code :", process.returncode)
print(f"Wall-clock seconds  : {elapsed:.2f}")

print("\nExecution log:")
print(A3_LOG)

print("Log SHA-256:")
print(sha256(A3_LOG))

# ------------------------------------------------------------------
# Pre-registered failure handling
# ------------------------------------------------------------------

if process.returncode != 0:

    print("\n" + "=" * 100)
    print("✗ A3 EXECUTION FAILED")
    print("✗ FAILURE LOG HAS BEEN PRESERVED")
    print("✗ DO NOT RERUN A3")
    print("✗ DO NOT SUBSTITUTE ANOTHER MODEL")
    print("✗ DO NOT START A4")
    print("✗ SEND THIS OUTPUT FOR DIAGNOSIS")
    print("=" * 100)

    raise RuntimeError(
        f"A3 runner exited with return code {process.returncode}"
    )

# ------------------------------------------------------------------
# Successful execution checks
# ------------------------------------------------------------------

assert A3_OUT.exists(), (
    "STOP: A3 process returned success but result CSV is missing."
)

df = pd.read_csv(A3_OUT)

assert len(df) == 120, (
    f"STOP: expected 120 rows, found {len(df)}."
)

assert df["id"].is_unique, (
    "STOP: A3 result contains duplicate case IDs."
)

print("\nA3 RESULT")
print("-" * 100)

print("Rows       :", len(df))
print("Unique IDs :", df["id"].nunique())

print("\nResult:")
print(A3_OUT)

print("Result SHA-256:")
print(sha256(A3_OUT))

print("\n" + "=" * 100)
print("✓ A3 EXECUTION COMPLETED")
print("✓ 120 CASES WRITTEN")
print("✓ RAW A3 RESULT PRESERVED")
print("✓ EXECUTION LOG PRESERVED")
print("✓ DO NOT RUN A4 YET")
print("=" * 100)

STEP A18 — EXECUTE A3 MODEL-SWEEP ARM

✓ Frozen runner verified
✓ Frozen benchmark verified
✓ A1 reproduction manifest verified
✓ Frozen containment definition verified
✓ TOGETHER_API_KEY present
✓ A3 result path clean
✓ A3 log path clean

PRE-REGISTERED A3
----------------------------------------------------------------------------------------------------
Model : Qwen/Qwen2.5-7B-Instruct-Turbo
Role  : Smaller model arm for model-scale sensitivity
Cases : 120

Only experimental variable being changed:
  model -> Qwen/Qwen2.5-7B-Instruct-Turbo

Unchanged:
  ✓ runner
  ✓ benchmark
  ✓ prompt
  ✓ deterministic gates
  ✓ thresholds
  ✓ labels
  ✓ containment definition

----------------------------------------------------------------------------------------------------
EXECUTING A3
----------------------------------------------------------------------------------------------------

Command:
python /content/deterministic-intent-translation-journal/experiment-2b/implementation/e3_v3/run_expe

In [23]:
from pathlib import Path
import pandas as pd
import hashlib
import json
from scipy.stats import beta

ROOT = Path("/content/deterministic-intent-translation-journal")
EXP_A = ROOT / "journal_experiments/experiment_A_model_sweep"

A1 = EXP_A / "results/raw/A1_Qwen3.5-9B_results.csv"
A2 = EXP_A / "results/raw/A2_Llama-3.3-70B-Instruct-Turbo_results.csv"
A3 = EXP_A / "results/raw/A3_Qwen2.5-7B-Instruct-Turbo_results.csv"

METRICS_OUT = EXP_A / "results/metrics/A3_metrics.json"
COMPARISON_OUT = EXP_A / "results/analysis/A3_vs_A1_case_comparison.csv"

EXPECTED_A1_SHA = (
    "c22618b2750de7f5ef1824fa4036a901ef54f611ad162fcd797b20d79e2b4e06"
)

EXPECTED_A2_SHA = (
    "5eba3be64426c14ac1a917be5392e4b6a0a487f3bf9be81c076d7f4b429380d7"
)

EXPECTED_A3_SHA = (
    "fa4c9a0e57103768d06d370983a70a76516800bd0d57ec3d965445d35daf440a"
)


def sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(1024 * 1024), b""):
            h.update(block)
    return h.hexdigest()


def as_bool(series):
    if pd.api.types.is_bool_dtype(series):
        return (
            series.astype("boolean")
            .fillna(False)
            .astype(bool)
        )

    return (
        series.astype(str)
        .str.strip()
        .str.lower()
        .map({
            "true": True,
            "false": False,
            "1": True,
            "0": False,
        })
        .astype("boolean")
        .fillna(False)
        .astype(bool)
    )


def descriptor_present(df):
    return (
        df["descriptor_json"]
        .fillna("")
        .astype(str)
        .str.strip()
        .ne("")
    )


def clopper_pearson(k, n, alpha=0.05):
    if n == 0:
        return None, None

    lower = (
        0.0
        if k == 0
        else beta.ppf(alpha / 2, k, n - k + 1)
    )

    upper = (
        1.0
        if k == n
        else beta.ppf(1 - alpha / 2, k + 1, n - k)
    )

    return float(lower), float(upper)


def calculate_metrics(df):
    expected_valid = as_bool(df["expected_valid"])
    accepted = as_bool(df["accepted"])
    correct = as_bool(df["decision_correct"])
    llm = as_bool(df["llm_invoked"])
    parse = as_bool(df["parse_ok"])

    descriptor = descriptor_present(df)

    valid = expected_valid
    invalid = ~expected_valid

    total = len(df)
    valid_n = int(valid.sum())
    invalid_n = int(invalid.sum())

    correct_n = int(correct.sum())

    valid_accepted = int((valid & accepted).sum())
    valid_rejected = int((valid & ~accepted).sum())

    invalid_rejected = int((invalid & ~accepted).sum())
    invalid_accepted = int((invalid & accepted).sum())

    # invalid/reject treated as positive
    TP = invalid_rejected
    FP = valid_rejected
    FN = invalid_accepted
    TN = valid_accepted

    precision = TP / (TP + FP) if TP + FP else 0.0
    recall = TP / (TP + FN) if TP + FN else 0.0

    f1 = (
        2 * precision * recall / (precision + recall)
        if precision + recall
        else 0.0
    )

    accuracy = correct_n / total
    var_value = valid_accepted / valid_n
    irr = invalid_rejected / invalid_n

    acc_ci = clopper_pearson(correct_n, total)
    var_ci = clopper_pearson(valid_accepted, valid_n)
    irr_ci = clopper_pearson(invalid_rejected, invalid_n)

    rejected_with_descriptor = int(
        ((~accepted) & descriptor).sum()
    )

    accepted_without_descriptor = int(
        (accepted & ~descriptor).sum()
    )

    return {
        "total": total,
        "valid": valid_n,
        "invalid": invalid_n,

        "correct": correct_n,
        "incorrect": total - correct_n,

        "valid_accepted": valid_accepted,
        "valid_rejected": valid_rejected,

        "invalid_rejected": invalid_rejected,
        "invalid_accepted": invalid_accepted,

        "TP": TP,
        "FP": FP,
        "FN": FN,
        "TN": TN,

        "accuracy": accuracy,
        "accuracy_ci95": list(acc_ci),

        "VAR": var_value,
        "VAR_ci95": list(var_ci),

        "IRR": irr,
        "IRR_ci95": list(irr_ci),

        "rejection_precision": precision,
        "rejection_recall": recall,
        "rejection_f1": f1,

        "llm_invoked": int(llm.sum()),
        "parse_success": int((llm & parse).sum()),

        "accepted_count": int(accepted.sum()),
        "descriptor_count": int(descriptor.sum()),

        "containment_violations":
            rejected_with_descriptor,

        "accepted_without_descriptor":
            accepted_without_descriptor,
    }


print("=" * 110)
print("STEP A19 — A3 METRIC, CONTAINMENT, AND CASE-LEVEL ANALYSIS")
print("=" * 110)

# ------------------------------------------------------------------
# Integrity
# ------------------------------------------------------------------

assert sha256(A1) == EXPECTED_A1_SHA, "STOP: A1 changed."
assert sha256(A2) == EXPECTED_A2_SHA, "STOP: A2 changed."
assert sha256(A3) == EXPECTED_A3_SHA, "STOP: A3 changed."

assert not METRICS_OUT.exists(), (
    "STOP: A3 metrics already exist."
)

assert not COMPARISON_OUT.exists(), (
    "STOP: A3 comparison already exists."
)

a1 = pd.read_csv(A1)
a2 = pd.read_csv(A2)
a3 = pd.read_csv(A3)

for df in [a1, a2, a3]:
    assert len(df) == 120
    assert df["id"].is_unique

assert set(a1["id"]) == set(a3["id"])

print("\n✓ A1 hash verified")
print("✓ A2 hash verified")
print("✓ A3 hash verified")
print("✓ All arms contain 120 unique cases")

# ------------------------------------------------------------------
# Metrics
# ------------------------------------------------------------------

m1 = calculate_metrics(a1)
m2 = calculate_metrics(a2)
m3 = calculate_metrics(a3)

print("\n" + "-" * 110)
print("A1 / A2 / A3 PERFORMANCE")
print("-" * 110)

print(
    f"{'Metric':28s}"
    f"{'A1':>14s}"
    f"{'A2':>14s}"
    f"{'A3':>14s}"
    f"{'A3-A1':>14s}"
)

print("-" * 110)

for key in [
    "accuracy",
    "VAR",
    "IRR",
    "rejection_precision",
    "rejection_recall",
    "rejection_f1",
]:
    print(
        f"{key:28s}"
        f"{m1[key]:14.6f}"
        f"{m2[key]:14.6f}"
        f"{m3[key]:14.6f}"
        f"{m3[key]-m1[key]:+14.6f}"
    )

# ------------------------------------------------------------------
# Exact CIs
# ------------------------------------------------------------------

print("\nA3 EXACT 95% CLOPPER-PEARSON CIs")
print("-" * 110)

print(
    f"Accuracy : {m3['correct']}/{m3['total']} "
    f"= {100*m3['accuracy']:.2f}% "
    f"[{100*m3['accuracy_ci95'][0]:.2f}%, "
    f"{100*m3['accuracy_ci95'][1]:.2f}%]"
)

print(
    f"VAR      : {m3['valid_accepted']}/{m3['valid']} "
    f"= {100*m3['VAR']:.2f}% "
    f"[{100*m3['VAR_ci95'][0]:.2f}%, "
    f"{100*m3['VAR_ci95'][1]:.2f}%]"
)

print(
    f"IRR      : {m3['invalid_rejected']}/{m3['invalid']} "
    f"= {100*m3['IRR']:.2f}% "
    f"[{100*m3['IRR_ci95'][0]:.2f}%, "
    f"{100*m3['IRR_ci95'][1]:.2f}%]"
)

# ------------------------------------------------------------------
# Counts + containment
# ------------------------------------------------------------------

print("\nA3 COUNTS")
print("-" * 110)

for key in [
    "correct",
    "incorrect",
    "valid_accepted",
    "valid_rejected",
    "invalid_rejected",
    "invalid_accepted",
    "TP",
    "FP",
    "FN",
    "TN",
    "llm_invoked",
    "parse_success",
    "accepted_count",
    "descriptor_count",
    "containment_violations",
    "accepted_without_descriptor",
]:
    print(f"{key:30s}: {m3[key]}")

print("\nCONTAINMENT CHECK")
print("-" * 110)

print(
    "Frozen definition: accepted=False "
    "AND descriptor_json present"
)

print(
    "A1 containment violations :",
    m1["containment_violations"]
)

print(
    "A2 containment violations :",
    m2["containment_violations"]
)

print(
    "A3 containment violations :",
    m3["containment_violations"]
)

if m3["containment_violations"] == 0:
    print("\n✓ A3 preserved the frozen containment invariant")
else:
    print("\n✗ A3 CONTAINMENT VIOLATION DETECTED")
    print("✗ Preserve result and STOP before A4")

# ------------------------------------------------------------------
# A1 vs A3 case comparison
# ------------------------------------------------------------------

cols = [
    "id",
    "expected_valid",
    "accepted",
    "decision_correct",
    "llm_invoked",
    "parse_ok",
    "rejection_stage",
    "error_codes_json",
]

left = a1[cols].copy()

right = a3[
    [
        "id",
        "accepted",
        "decision_correct",
        "llm_invoked",
        "parse_ok",
        "rejection_stage",
        "error_codes_json",
    ]
].copy()

left = left.rename(columns={
    "accepted": "A1_accepted",
    "decision_correct": "A1_correct",
    "llm_invoked": "A1_llm_invoked",
    "parse_ok": "A1_parse_ok",
    "rejection_stage": "A1_rejection_stage",
    "error_codes_json": "A1_error_codes",
})

right = right.rename(columns={
    "accepted": "A3_accepted",
    "decision_correct": "A3_correct",
    "llm_invoked": "A3_llm_invoked",
    "parse_ok": "A3_parse_ok",
    "rejection_stage": "A3_rejection_stage",
    "error_codes_json": "A3_error_codes",
})

comparison = left.merge(
    right,
    on="id",
    how="inner",
    validate="one_to_one",
)

for col in [
    "A1_accepted",
    "A3_accepted",
    "A1_correct",
    "A3_correct",
    "A1_llm_invoked",
    "A3_llm_invoked",
    "A1_parse_ok",
    "A3_parse_ok",
]:
    comparison[col] = as_bool(comparison[col])

comparison["decision_changed"] = (
    comparison["A1_accepted"] !=
    comparison["A3_accepted"]
)

comparison["correctness_changed"] = (
    comparison["A1_correct"] !=
    comparison["A3_correct"]
)

comparison["llm_path_changed"] = (
    comparison["A1_llm_invoked"] !=
    comparison["A3_llm_invoked"]
)

comparison["A1_wrong_A3_correct"] = (
    (~comparison["A1_correct"]) &
    comparison["A3_correct"]
)

comparison["A1_correct_A3_wrong"] = (
    comparison["A1_correct"] &
    (~comparison["A3_correct"])
)

decision_changes = int(
    comparison["decision_changed"].sum()
)

correctness_changes = int(
    comparison["correctness_changed"].sum()
)

path_changes = int(
    comparison["llm_path_changed"].sum()
)

improved = int(
    comparison["A1_wrong_A3_correct"].sum()
)

regressed = int(
    comparison["A1_correct_A3_wrong"].sum()
)

print("\nCASE-LEVEL A1 ↔ A3 DIFFERENCES")
print("-" * 110)

print("Acceptance decisions changed :", decision_changes)
print("Correctness changed          :", correctness_changes)
print("A1 wrong -> A3 correct       :", improved)
print("A1 correct -> A3 wrong       :", regressed)
print("LLM invocation path changed  :", path_changes)

changed = comparison[
    comparison["decision_changed"] |
    comparison["correctness_changed"] |
    comparison["llm_path_changed"]
]

if len(changed):
    print("\nCHANGED CASES")
    print("-" * 110)

    print(
        changed[
            [
                "id",
                "expected_valid",
                "A1_accepted",
                "A3_accepted",
                "A1_correct",
                "A3_correct",
                "A1_llm_invoked",
                "A3_llm_invoked",
                "A1_rejection_stage",
                "A3_rejection_stage",
            ]
        ].to_string(index=False)
    )
else:
    print("\nNo A1/A3 decision or path differences.")

# ------------------------------------------------------------------
# Save artifacts
# ------------------------------------------------------------------

comparison.to_csv(
    COMPARISON_OUT,
    index=False
)

record = {
    "arm": "A3",
    "model": "Qwen/Qwen2.5-7B-Instruct-Turbo",

    "role":
        "Smaller model arm for model-scale sensitivity",

    "result_sha256": sha256(A3),

    "metrics": m3,

    "containment_definition": (
        "accepted=False AND descriptor_json present"
    ),

    "comparison_with_A1": {
        "decision_changes": decision_changes,
        "correctness_changes": correctness_changes,
        "A1_wrong_A3_correct": improved,
        "A1_correct_A3_wrong": regressed,
        "llm_path_changes": path_changes,
    },
}

METRICS_OUT.write_text(
    json.dumps(record, indent=2) + "\n",
    encoding="utf-8"
)

print("\nARTIFACTS")
print("-" * 110)

print("Metrics:")
print(METRICS_OUT)
print("SHA-256:", sha256(METRICS_OUT))

print("\nCase comparison:")
print(COMPARISON_OUT)
print("SHA-256:", sha256(COMPARISON_OUT))

print("\n" + "=" * 110)
print("✓ A3 ANALYSIS COMPLETE")
print("✓ EXACT 95% CIs CALCULATED")
print("✓ FROZEN CONTAINMENT CHECK APPLIED")
print("✓ CASE-LEVEL COMPARISON SAVED")
print("✓ NO MODEL/API CALL MADE")

if m3["containment_violations"] == 0:
    print("✓ A3 ELIGIBLE TO BE FROZEN BEFORE A4")
else:
    print("✗ STOP — INVESTIGATE CONTAINMENT VIOLATION")

print("=" * 110)

STEP A19 — A3 METRIC, CONTAINMENT, AND CASE-LEVEL ANALYSIS

✓ A1 hash verified
✓ A2 hash verified
✓ A3 hash verified
✓ All arms contain 120 unique cases

--------------------------------------------------------------------------------------------------------------
A1 / A2 / A3 PERFORMANCE
--------------------------------------------------------------------------------------------------------------
Metric                                  A1            A2            A3         A3-A1
--------------------------------------------------------------------------------------------------------------
accuracy                          0.841667      0.833333      0.516667     -0.325000
VAR                               0.879310      0.827586      0.000000     -0.879310
IRR                               0.806452      0.838710      1.000000     +0.193548
rejection_precision               0.877193      0.838710      0.516667     -0.360526
rejection_recall                  0.806452      0.838710      1

In [24]:
from pathlib import Path
import pandas as pd
import hashlib
import json
from collections import Counter

ROOT = Path("/content/deterministic-intent-translation-journal")
EXP_A = ROOT / "journal_experiments/experiment_A_model_sweep"

A3 = EXP_A / "results/raw/A3_Qwen2.5-7B-Instruct-Turbo_results.csv"
A3_LOG = EXP_A / "logs/A3_Qwen2.5-7B-Instruct-Turbo_execution.log"

EXPECTED_A3_SHA = (
    "fa4c9a0e57103768d06d370983a70a76516800bd0d57ec3d965445d35daf440a"
)

EXPECTED_LOG_SHA = (
    "552fdad61b94b8e3a6ab698cc2ddcefe0c3e9f71398e4a03b6a83422429abf7a"
)


def sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(1024 * 1024), b""):
            h.update(block)
    return h.hexdigest()


print("=" * 110)
print("STEP A20 — DIAGNOSE A3 PARSE/EXCEPTION FAILURE")
print("=" * 110)

assert A3.exists()
assert A3_LOG.exists()

assert sha256(A3) == EXPECTED_A3_SHA, (
    "STOP: A3 result changed."
)

assert sha256(A3_LOG) == EXPECTED_LOG_SHA, (
    "STOP: A3 execution log changed."
)

df = pd.read_csv(A3)

assert len(df) == 120
assert df["id"].is_unique

print("\n✓ A3 result hash verified")
print("✓ A3 log hash verified")
print("✓ 120 unique cases verified")

# ------------------------------------------------------------------
# Rejection stages
# ------------------------------------------------------------------

print("\n" + "-" * 110)
print("A3 REJECTION-STAGE DISTRIBUTION")
print("-" * 110)

print(
    df["rejection_stage"]
    .fillna("<ACCEPTED>")
    .value_counts(dropna=False)
    .to_string()
)

# ------------------------------------------------------------------
# Error-code distribution
# ------------------------------------------------------------------

print("\n" + "-" * 110)
print("A3 ERROR-CODE DISTRIBUTION")
print("-" * 110)

print(
    df["error_codes_json"]
    .fillna("<EMPTY>")
    .value_counts(dropna=False)
    .to_string()
)

# ------------------------------------------------------------------
# Exception distribution
# ------------------------------------------------------------------

print("\n" + "-" * 110)
print("A3 EXCEPTION DISTRIBUTION")
print("-" * 110)

exceptions = (
    df["exception"]
    .dropna()
    .astype(str)
    .str.strip()
)

print("Rows with exception :", len(exceptions))
print("Unique exceptions   :", exceptions.nunique())

if len(exceptions):
    counts = exceptions.value_counts()

    for i, (exc, count) in enumerate(counts.items(), start=1):
        print(f"\n[{i}] count = {count}")
        print(exc)

# ------------------------------------------------------------------
# LLM-invoked rows
# ------------------------------------------------------------------

def as_bool(series):
    return (
        series.astype(str)
        .str.strip()
        .str.lower()
        .map({
            "true": True,
            "false": False,
            "1": True,
            "0": False,
        })
        .astype("boolean")
        .fillna(False)
        .astype(bool)
    )


llm = as_bool(df["llm_invoked"])
parse = as_bool(df["parse_ok"])

invoked = df.loc[llm].copy()

print("\n" + "-" * 110)
print("LLM-PATH DIAGNOSTICS")
print("-" * 110)

print("LLM invoked             :", int(llm.sum()))
print("Parse success            :", int((llm & parse).sum()))
print("Parse failure            :", int((llm & ~parse).sum()))

print(
    "LLM-invoked + exception :",
    int(
        llm &
        df["exception"].notna()
    ).sum()
)

# ------------------------------------------------------------------
# finish_reason / raw_text diagnostics
# ------------------------------------------------------------------

print("\n" + "-" * 110)
print("FINISH-REASON DISTRIBUTION — LLM-INVOKED ROWS")
print("-" * 110)

print(
    invoked["finish_reason"]
    .fillna("<EMPTY>")
    .value_counts(dropna=False)
    .to_string()
)

print("\n" + "-" * 110)
print("RAW-TEXT PRESENCE — LLM-INVOKED ROWS")
print("-" * 110)

raw_present = (
    invoked["raw_text"]
    .fillna("")
    .astype(str)
    .str.strip()
    .ne("")
)

print("Raw text present :", int(raw_present.sum()))
print("Raw text absent  :", int((~raw_present).sum()))

# ------------------------------------------------------------------
# Adapter-result diagnostics
# ------------------------------------------------------------------

print("\n" + "-" * 110)
print("ADAPTER RESULT PRESENCE — LLM-INVOKED ROWS")
print("-" * 110)

adapter_present = (
    invoked["adapter_result_json"]
    .fillna("")
    .astype(str)
    .str.strip()
    .ne("")
)

print("Adapter result present :", int(adapter_present.sum()))
print("Adapter result absent  :", int((~adapter_present).sum()))

# ------------------------------------------------------------------
# Sample affected rows
# ------------------------------------------------------------------

affected = df.loc[
    df["rejection_stage"].fillna("").eq("exception"),
    [
        "id",
        "expected_valid",
        "llm_invoked",
        "parse_ok",
        "finish_reason",
        "error_codes_json",
        "exception",
        "raw_text",
        "adapter_result_json",
    ]
].copy()

print("\n" + "-" * 110)
print("FIRST 10 EXCEPTION CASES")
print("-" * 110)

if len(affected):
    with pd.option_context(
        "display.max_colwidth", 300,
        "display.width", 250
    ):
        print(
            affected.head(10).to_string(index=False)
        )
else:
    print("No exception rows.")

# ------------------------------------------------------------------
# Check whether every LLM-invoked case became an exception
# ------------------------------------------------------------------

exception_mask = (
    df["rejection_stage"]
    .fillna("")
    .eq("exception")
)

all_llm_exception = bool(
    (exception_mask[llm]).all()
)

print("\n" + "=" * 110)
print("DIAGNOSTIC SUMMARY")
print("=" * 110)

print(
    "Every LLM-invoked row ended in exception :",
    all_llm_exception
)

print(
    "Every LLM-invoked row failed parsing     :",
    bool((~parse[llm]).all())
)

print(
    "Containment violations                   :",
    int(
        (
            ~as_bool(df["accepted"]) &
            df["descriptor_json"]
            .fillna("")
            .astype(str)
            .str.strip()
            .ne("")
        ).sum()
    )
)

print("\n✓ READ-ONLY DIAGNOSIS COMPLETE")
print("✓ A3 WAS NOT RERUN")
print("✓ NO MODEL/API CALL MADE")
print("✓ NO FILE MODIFIED")
print("✓ DO NOT RUN A4 YET")
print("=" * 110)

STEP A20 — DIAGNOSE A3 PARSE/EXCEPTION FAILURE

✓ A3 result hash verified
✓ A3 log hash verified
✓ 120 unique cases verified

--------------------------------------------------------------------------------------------------------------
A3 REJECTION-STAGE DISTRIBUTION
--------------------------------------------------------------------------------------------------------------
rejection_stage
exception                     77
source_gate                   24
constraint_consistency        14
r7_qualitative_source_gate     5

--------------------------------------------------------------------------------------------------------------
A3 ERROR-CODE DISTRIBUTION
--------------------------------------------------------------------------------------------------------------
error_codes_json
["BadRequestError"]                        77
["UNRESOLVED_SOURCE_REQUIREMENT"]          12
["SOURCE_CONTROL_ATTEMPT"]                 12
["INCOMPATIBLE_BOUNDS"]                     9
["EXACT_VALUE_OUTSIDE

TypeError: cannot convert the series to <class 'int'>

In [25]:
from pathlib import Path
from datetime import datetime, timezone
import pandas as pd
import hashlib
import json

ROOT = Path("/content/deterministic-intent-translation-journal")
EXP_A = ROOT / "journal_experiments/experiment_A_model_sweep"

A3 = EXP_A / "results/raw/A3_Qwen2.5-7B-Instruct-Turbo_results.csv"
A3_LOG = EXP_A / "logs/A3_Qwen2.5-7B-Instruct-Turbo_execution.log"
A3_METRICS = EXP_A / "results/metrics/A3_metrics.json"
A3_COMPARISON = EXP_A / "results/analysis/A3_vs_A1_case_comparison.csv"

OUT = EXP_A / "manifests/A3_unavailable_arm_manifest_v1.json"

EXPECTED_A3_SHA = (
    "fa4c9a0e57103768d06d370983a70a76516800bd0d57ec3d965445d35daf440a"
)

EXPECTED_LOG_SHA = (
    "552fdad61b94b8e3a6ab698cc2ddcefe0c3e9f71398e4a03b6a83422429abf7a"
)

EXPECTED_METRICS_SHA = (
    "8175a0b074fb657af8f629dc91358259eb19007965491355f47762d2fc0e4151"
)

EXPECTED_COMPARISON_SHA = (
    "ca6564cc4003b90e9abfbbd35a253bf1d43ade2c20f1731237585256f70c4642"
)


def sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(1024 * 1024), b""):
            h.update(block)
    return h.hexdigest()


def as_bool(series):
    if pd.api.types.is_bool_dtype(series):
        return (
            series.astype("boolean")
            .fillna(False)
            .astype(bool)
        )

    return (
        series.astype(str)
        .str.strip()
        .str.lower()
        .map({
            "true": True,
            "false": False,
            "1": True,
            "0": False,
        })
        .astype("boolean")
        .fillna(False)
        .astype(bool)
    )


print("=" * 110)
print("STEP A20B — COMPLETE A3 DIAGNOSIS AND FREEZE UNAVAILABLE ARM")
print("=" * 110)

# ------------------------------------------------------------------
# Integrity
# ------------------------------------------------------------------

for p in [A3, A3_LOG, A3_METRICS, A3_COMPARISON]:
    assert p.exists(), f"STOP: missing {p}"

assert sha256(A3) == EXPECTED_A3_SHA, "STOP: A3 result changed."
assert sha256(A3_LOG) == EXPECTED_LOG_SHA, "STOP: A3 log changed."
assert sha256(A3_METRICS) == EXPECTED_METRICS_SHA, "STOP: A3 metrics changed."
assert sha256(A3_COMPARISON) == EXPECTED_COMPARISON_SHA, (
    "STOP: A3 comparison changed."
)

assert not OUT.exists(), (
    "STOP: A3 unavailable-arm manifest already exists."
)

df = pd.read_csv(A3)

assert len(df) == 120
assert df["id"].is_unique

print("\n✓ A3 result verified")
print("✓ A3 execution log verified")
print("✓ A3 metrics artifact verified")
print("✓ A3 comparison artifact verified")
print("✓ 120 unique cases verified")

# ------------------------------------------------------------------
# Diagnose execution
# ------------------------------------------------------------------

llm = as_bool(df["llm_invoked"])
parse = as_bool(df["parse_ok"])
accepted = as_bool(df["accepted"])

exception_stage = (
    df["rejection_stage"]
    .fillna("")
    .astype(str)
    .str.strip()
    .eq("exception")
)

bad_request = (
    df["error_codes_json"]
    .fillna("")
    .astype(str)
    .str.contains("BadRequestError", regex=False)
)

model_not_available = (
    df["exception"]
    .fillna("")
    .astype(str)
    .str.contains("model_not_available", regex=False)
)

non_serverless = (
    df["exception"]
    .fillna("")
    .astype(str)
    .str.contains("non-serverless model", regex=False)
)

descriptor = (
    df["descriptor_json"]
    .fillna("")
    .astype(str)
    .str.strip()
    .ne("")
)

llm_count = int(llm.sum())
parse_success = int((llm & parse).sum())
parse_failure = int((llm & ~parse).sum())

llm_exception = int((llm & exception_stage).sum())
llm_bad_request = int((llm & bad_request).sum())
llm_model_unavailable = int((llm & model_not_available).sum())
llm_non_serverless = int((llm & non_serverless).sum())

containment_violations = int(
    ((~accepted) & descriptor).sum()
)

print("\n" + "-" * 110)
print("A3 DIAGNOSTIC COUNTS")
print("-" * 110)

print("LLM invoked                         :", llm_count)
print("Parse success                       :", parse_success)
print("Parse failure                       :", parse_failure)
print("LLM-invoked rows at exception stage :", llm_exception)
print("LLM-invoked BadRequestError rows     :", llm_bad_request)
print("model_not_available rows             :", llm_model_unavailable)
print("non-serverless-model rows            :", llm_non_serverless)
print("Containment violations               :", containment_violations)

# ------------------------------------------------------------------
# Classification checks
# ------------------------------------------------------------------

assert llm_count == 77, (
    f"STOP: expected 77 LLM-invoked rows, got {llm_count}"
)

assert parse_success == 0, (
    "STOP: unexpected successful parse detected."
)

assert parse_failure == 77

assert llm_exception == 77, (
    "STOP: not every LLM-invoked row ended at exception stage."
)

assert llm_bad_request == 77, (
    "STOP: not every LLM-invoked row contains BadRequestError."
)

assert llm_model_unavailable == 77, (
    "STOP: not every LLM-invoked row reports model_not_available."
)

assert llm_non_serverless == 77, (
    "STOP: not every LLM-invoked row reports non-serverless access."
)

assert containment_violations == 0

classification = (
    "UNAVAILABLE_ON_ACCESSIBLE_SERVERLESS_ENDPOINT"
)

print("\n" + "-" * 110)
print("A3 CLASSIFICATION")
print("-" * 110)

print("Status :", classification)

print(
    "\nInterpretation:\n"
    "A3 is not an evaluable model-performance arm. "
    "The preregistered model could not be accessed through "
    "the available serverless endpoint. All 77 attempted LLM "
    "calls failed with BadRequestError/model_not_available."
)

print(
    "\nThe derived decision metrics from this run must NOT be "
    "interpreted as Qwen2.5-7B semantic-parser performance."
)

print(
    "\nThe raw result remains preserved as evidence of the "
    "preregistered arm execution and endpoint unavailability."
)

# ------------------------------------------------------------------
# Freeze classification
# ------------------------------------------------------------------

record = {
    "manifest_id":
        "experiment-A-A3-unavailable-arm-v1",

    "created_utc":
        datetime.now(timezone.utc).isoformat(),

    "arm": "A3",

    "model":
        "Qwen/Qwen2.5-7B-Instruct-Turbo",

    "preregistered_role":
        "Smaller model arm for model-scale sensitivity",

    "status": classification,

    "reason": {
        "provider_error_type":
            "BadRequestError",

        "provider_error_code":
            "model_not_available",

        "provider_message_summary": (
            "The model is non-serverless and requires a "
            "dedicated endpoint."
        ),
    },

    "diagnostics": {
        "total_cases": 120,
        "llm_invoked": llm_count,
        "parse_success": parse_success,
        "parse_failure": parse_failure,
        "llm_exception_rows": llm_exception,
        "bad_request_rows": llm_bad_request,
        "model_not_available_rows":
            llm_model_unavailable,
        "non_serverless_rows":
            llm_non_serverless,
        "containment_violations":
            containment_violations,
    },

    "reporting_policy": {
        "include_arm_in_experiment_accounting": True,

        "include_raw_execution_evidence": True,

        "include_as_successful_model_performance_arm": False,

        "use_accuracy_VAR_IRR_for_model_comparison": False,

        "reason": (
            "The LLM endpoint was unavailable; therefore the "
            "derived reject-all decision metrics do not measure "
            "semantic-parser performance."
        ),

        "replacement_model_allowed":
            False,

        "rerun_allowed":
            False,
    },

    "artifacts": {
        "raw_result": {
            "path": str(A3.relative_to(ROOT)),
            "sha256": sha256(A3),
        },

        "execution_log": {
            "path": str(A3_LOG.relative_to(ROOT)),
            "sha256": sha256(A3_LOG),
        },

        "metrics": {
            "path": str(A3_METRICS.relative_to(ROOT)),
            "sha256": sha256(A3_METRICS),
            "interpretation":
                "diagnostic only; not model-performance evidence",
        },

        "case_comparison": {
            "path": str(A3_COMPARISON.relative_to(ROOT)),
            "sha256": sha256(A3_COMPARISON),
            "interpretation":
                "diagnostic only; dominated by endpoint failure",
        },
    },

    "experimental_decision": (
        "Preserve A3 as a preregistered unavailable arm. "
        "Do not rerun, substitute, or use its derived "
        "accuracy/VAR/IRR as model-performance evidence. "
        "Proceed to preregistered A4."
    ),
}

OUT.write_text(
    json.dumps(record, indent=2) + "\n",
    encoding="utf-8"
)

print("\n" + "-" * 110)
print("FROZEN A3 RECORD")
print("-" * 110)

print(OUT)

print("SHA-256:")
print(sha256(OUT))

print("\n" + "=" * 110)
print("✓ A3 CLASSIFIED AS ENDPOINT-UNAVAILABLE")
print("✓ RAW A3 EXECUTION PRESERVED")
print("✓ A3 NOT USED AS MODEL-PERFORMANCE EVIDENCE")
print("✓ NO RERUN OR MODEL SUBSTITUTION")
print("✓ CONTAINMENT VIOLATIONS = 0")
print("✓ NO MODEL/API CALL MADE")
print("✓ A4 ELIGIBLE AFTER THIS STEP PASSES")
print("=" * 110)

STEP A20B — COMPLETE A3 DIAGNOSIS AND FREEZE UNAVAILABLE ARM

✓ A3 result verified
✓ A3 execution log verified
✓ A3 metrics artifact verified
✓ A3 comparison artifact verified
✓ 120 unique cases verified

--------------------------------------------------------------------------------------------------------------
A3 DIAGNOSTIC COUNTS
--------------------------------------------------------------------------------------------------------------
LLM invoked                         : 77
Parse success                       : 0
Parse failure                       : 77
LLM-invoked rows at exception stage : 77
LLM-invoked BadRequestError rows     : 77
model_not_available rows             : 77
non-serverless-model rows            : 77
Containment violations               : 0

--------------------------------------------------------------------------------------------------------------
A3 CLASSIFICATION
--------------------------------------------------------------------------------------------

In [26]:
import subprocess
from pathlib import Path
import hashlib
import os
import time
import pandas as pd

ROOT = Path("/content/deterministic-intent-translation-journal")

EXP_A = (
    ROOT /
    "journal_experiments" /
    "experiment_A_model_sweep"
)

RUNNER = ROOT / (
    "experiment-2b/implementation/e3_v3/"
    "run_experiment2b_e3v3_r7_dev_v1.py"
)

DATASET = ROOT / (
    "experiment-2b/reproducibility/"
    "e3v3_heldout_benchmark_v1.0/"
    "e3v3_heldout_benchmark_120_v1.csv"
)

A3_MANIFEST = (
    EXP_A /
    "manifests/A3_unavailable_arm_manifest_v1.json"
)

CONTAINMENT_MANIFEST = (
    EXP_A /
    "manifests/containment_definition_A1_A2_v1.json"
)

A4_OUT = (
    EXP_A /
    "results/raw/A4_Mistral-Small-24B-Instruct-2501_results.csv"
)

A4_LOG = (
    EXP_A /
    "logs/A4_Mistral-Small-24B-Instruct-2501_execution.log"
)

MODEL = "mistralai/Mistral-Small-24B-Instruct-2501"

EXPECTED_RUNNER_SHA = (
    "40c035cb864978c718a72fcfdfbb164d262529960b0ae78242d5194147cd8257"
)

EXPECTED_DATASET_SHA = (
    "84576e7d99a2721e61ded5a14bdafeb96707b068397243bbed32aa735496d4de"
)

EXPECTED_A3_MANIFEST_SHA = (
    "22920e779994b802d66493d6722f89b2405f858f1967ef75ac90e282129534e8"
)

EXPECTED_CONTAINMENT_SHA = (
    "cd31ea28ffa8a2f82121d1c4cf0e5478425d194299d309e603069545d75d4832"
)


def sha256(path):
    h = hashlib.sha256()

    with open(path, "rb") as f:
        for block in iter(lambda: f.read(1024 * 1024), b""):
            h.update(block)

    return h.hexdigest()


print("=" * 100)
print("STEP A21 — EXECUTE A4 MODEL-SWEEP ARM")
print("=" * 100)

# ------------------------------------------------------------------
# Integrity checks
# ------------------------------------------------------------------

assert RUNNER.exists()
assert DATASET.exists()
assert A3_MANIFEST.exists()
assert CONTAINMENT_MANIFEST.exists()

assert sha256(RUNNER) == EXPECTED_RUNNER_SHA, (
    "STOP: frozen runner changed."
)

assert sha256(DATASET) == EXPECTED_DATASET_SHA, (
    "STOP: frozen benchmark changed."
)

assert sha256(A3_MANIFEST) == EXPECTED_A3_MANIFEST_SHA, (
    "STOP: frozen A3 unavailable-arm manifest changed."
)

assert sha256(CONTAINMENT_MANIFEST) == EXPECTED_CONTAINMENT_SHA, (
    "STOP: frozen containment definition changed."
)

assert os.environ.get("TOGETHER_API_KEY"), (
    "STOP: TOGETHER_API_KEY missing."
)

assert not A4_OUT.exists(), (
    f"STOP: A4 result already exists:\n{A4_OUT}\n"
    "Do not overwrite or rerun."
)

assert not A4_LOG.exists(), (
    f"STOP: A4 log already exists:\n{A4_LOG}\n"
    "Do not overwrite or rerun."
)

print("\n✓ Frozen runner verified")
print("✓ Frozen benchmark verified")
print("✓ A3 unavailable-arm record verified")
print("✓ Frozen containment definition verified")
print("✓ TOGETHER_API_KEY present")
print("✓ A4 result path clean")
print("✓ A4 log path clean")

# ------------------------------------------------------------------
# Pre-registered A4
# ------------------------------------------------------------------

print("\nPRE-REGISTERED A4")
print("-" * 100)

print("Model :", MODEL)
print("Role  : Alternative Mistral model family")
print("Cases : 120")

print("\nOnly experimental variable being changed:")
print("  model ->", MODEL)

print("\nUnchanged:")
print("  ✓ runner")
print("  ✓ benchmark")
print("  ✓ prompt")
print("  ✓ deterministic gates")
print("  ✓ thresholds")
print("  ✓ labels")
print("  ✓ containment definition")

# ------------------------------------------------------------------
# Execute exactly once
# ------------------------------------------------------------------

cmd = [
    "python",
    str(RUNNER),
    "--dataset",
    str(DATASET),
    "--out",
    str(A4_OUT),
    "--model",
    MODEL,
]

print("\n" + "-" * 100)
print("EXECUTING A4")
print("-" * 100)

print("\nCommand:")
print(
    " ".join(
        f'"{x}"' if " " in x else x
        for x in cmd
    )
)

print("\nRunning frozen 120-case benchmark ONCE...\n")

t0 = time.time()

process = subprocess.run(
    cmd,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    env=os.environ.copy(),
)

elapsed = time.time() - t0

# Preserve log regardless of success/failure.
A4_LOG.write_text(
    process.stdout or "",
    encoding="utf-8"
)

print(process.stdout)

print("\n" + "-" * 100)
print("EXECUTION STATUS")
print("-" * 100)

print("Process return code :", process.returncode)
print(f"Wall-clock seconds  : {elapsed:.2f}")

print("\nExecution log:")
print(A4_LOG)

print("Log SHA-256:")
print(sha256(A4_LOG))

# ------------------------------------------------------------------
# Process-level failure
# ------------------------------------------------------------------

if process.returncode != 0:

    print("\n" + "=" * 100)
    print("✗ A4 PROCESS-LEVEL EXECUTION FAILURE")
    print("✗ FAILURE LOG PRESERVED")
    print("✗ DO NOT RERUN A4")
    print("✗ DO NOT SUBSTITUTE ANOTHER MODEL")
    print("✗ DO NOT START A5")
    print("✗ SEND THIS OUTPUT FOR DIAGNOSIS")
    print("=" * 100)

    raise RuntimeError(
        f"A4 runner exited with return code "
        f"{process.returncode}"
    )

# ------------------------------------------------------------------
# Result-file checks
# ------------------------------------------------------------------

assert A4_OUT.exists(), (
    "STOP: A4 returned success but result CSV is missing."
)

df = pd.read_csv(A4_OUT)

assert len(df) == 120, (
    f"STOP: expected 120 rows, found {len(df)}."
)

assert df["id"].is_unique, (
    "STOP: A4 result contains duplicate case IDs."
)

print("\nA4 RESULT")
print("-" * 100)

print("Rows       :", len(df))
print("Unique IDs :", df["id"].nunique())

print("\nResult:")
print(A4_OUT)

print("Result SHA-256:")
print(sha256(A4_OUT))

# ------------------------------------------------------------------
# Lightweight diagnostic only
#
# This does NOT interpret performance. It simply tells us whether
# the arm reached successful LLM parsing before A22 analysis.
# ------------------------------------------------------------------

def as_bool(series):
    if pd.api.types.is_bool_dtype(series):
        return (
            series.astype("boolean")
            .fillna(False)
            .astype(bool)
        )

    return (
        series.astype(str)
        .str.strip()
        .str.lower()
        .map({
            "true": True,
            "false": False,
            "1": True,
            "0": False,
        })
        .astype("boolean")
        .fillna(False)
        .astype(bool)
    )


llm = as_bool(df["llm_invoked"])
parse = as_bool(df["parse_ok"])

print("\nLIGHTWEIGHT EXECUTION DIAGNOSTIC")
print("-" * 100)

print("LLM invoked  :", int(llm.sum()))
print("Parse success:", int((llm & parse).sum()))

print("\nRejection stages:")
print(
    df["rejection_stage"]
    .fillna("<ACCEPTED>")
    .value_counts()
    .to_string()
)

print("\n" + "=" * 100)
print("✓ A4 EXECUTION COMPLETED")
print("✓ 120 CASES WRITTEN")
print("✓ RAW A4 RESULT PRESERVED")
print("✓ EXECUTION LOG PRESERVED")
print("✓ NO PERFORMANCE INTERPRETATION MADE")
print("✓ DO NOT RUN A5 YET")
print("=" * 100)

STEP A21 — EXECUTE A4 MODEL-SWEEP ARM

✓ Frozen runner verified
✓ Frozen benchmark verified
✓ A3 unavailable-arm record verified
✓ Frozen containment definition verified
✓ TOGETHER_API_KEY present
✓ A4 result path clean
✓ A4 log path clean

PRE-REGISTERED A4
----------------------------------------------------------------------------------------------------
Model : mistralai/Mistral-Small-24B-Instruct-2501
Role  : Alternative Mistral model family
Cases : 120

Only experimental variable being changed:
  model -> mistralai/Mistral-Small-24B-Instruct-2501

Unchanged:
  ✓ runner
  ✓ benchmark
  ✓ prompt
  ✓ deterministic gates
  ✓ thresholds
  ✓ labels
  ✓ containment definition

----------------------------------------------------------------------------------------------------
EXECUTING A4
----------------------------------------------------------------------------------------------------

Command:
python /content/deterministic-intent-translation-journal/experiment-2b/implementation/e3_v

In [27]:
from pathlib import Path
import pandas as pd
import hashlib

ROOT = Path("/content/deterministic-intent-translation-journal")
EXP_A = ROOT / "journal_experiments/experiment_A_model_sweep"

A4 = (
    EXP_A /
    "results/raw/A4_Mistral-Small-24B-Instruct-2501_results.csv"
)

A4_LOG = (
    EXP_A /
    "logs/A4_Mistral-Small-24B-Instruct-2501_execution.log"
)

EXPECTED_A4_SHA = (
    "6babc2b2492473e5ee737e182dd338be2bc23c0cb0f72b5de8b70864d8757125"
)

EXPECTED_LOG_SHA = (
    "122d22c79b9c0fc33b2e3ff6e38e6279d45c1865dc71e727e5bbe675a83cbe68"
)


def sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(1024 * 1024), b""):
            h.update(block)
    return h.hexdigest()


def as_bool(series):
    if pd.api.types.is_bool_dtype(series):
        return (
            series.astype("boolean")
            .fillna(False)
            .astype(bool)
        )

    return (
        series.astype(str)
        .str.strip()
        .str.lower()
        .map({
            "true": True,
            "false": False,
            "1": True,
            "0": False,
        })
        .astype("boolean")
        .fillna(False)
        .astype(bool)
    )


print("=" * 110)
print("STEP A22 — DIAGNOSE A4 ZERO-PARSE EXECUTION")
print("=" * 110)

# ------------------------------------------------------------------
# Integrity
# ------------------------------------------------------------------

assert A4.exists(), "STOP: A4 result missing."
assert A4_LOG.exists(), "STOP: A4 log missing."

assert sha256(A4) == EXPECTED_A4_SHA, (
    "STOP: A4 result changed."
)

assert sha256(A4_LOG) == EXPECTED_LOG_SHA, (
    "STOP: A4 execution log changed."
)

df = pd.read_csv(A4)

assert len(df) == 120
assert df["id"].is_unique

print("\n✓ A4 result hash verified")
print("✓ A4 log hash verified")
print("✓ 120 unique cases verified")

# ------------------------------------------------------------------
# Basic execution path
# ------------------------------------------------------------------

llm = as_bool(df["llm_invoked"])
parse = as_bool(df["parse_ok"])
accepted = as_bool(df["accepted"])

exception_stage = (
    df["rejection_stage"]
    .fillna("")
    .astype(str)
    .str.strip()
    .eq("exception")
)

descriptor = (
    df["descriptor_json"]
    .fillna("")
    .astype(str)
    .str.strip()
    .ne("")
)

print("\n" + "-" * 110)
print("A4 EXECUTION SUMMARY")
print("-" * 110)

print("LLM invoked       :", int(llm.sum()))
print("Parse success     :", int((llm & parse).sum()))
print("Parse failure     :", int((llm & ~parse).sum()))
print("Exception stage   :", int(exception_stage.sum()))
print("Accepted          :", int(accepted.sum()))
print("Descriptor present:", int(descriptor.sum()))

print(
    "Containment violations:",
    int(((~accepted) & descriptor).sum())
)

# ------------------------------------------------------------------
# Rejection stages
# ------------------------------------------------------------------

print("\n" + "-" * 110)
print("A4 REJECTION-STAGE DISTRIBUTION")
print("-" * 110)

print(
    df["rejection_stage"]
    .fillna("<ACCEPTED>")
    .value_counts(dropna=False)
    .to_string()
)

# ------------------------------------------------------------------
# Error codes
# ------------------------------------------------------------------

print("\n" + "-" * 110)
print("A4 ERROR-CODE DISTRIBUTION")
print("-" * 110)

print(
    df["error_codes_json"]
    .fillna("<EMPTY>")
    .value_counts(dropna=False)
    .to_string()
)

# ------------------------------------------------------------------
# Normalize exception messages by removing request IDs.
#
# We want the actual provider failure category, not 77 different
# strings merely because each request has a unique provider ID.
# ------------------------------------------------------------------

exceptions = (
    df.loc[exception_stage, "exception"]
    .fillna("")
    .astype(str)
    .str.strip()
)

print("\n" + "-" * 110)
print("A4 EXCEPTION SUMMARY")
print("-" * 110)

print("Exception rows :", len(exceptions))

# Provider IDs make otherwise identical errors unique.
normalized = (
    exceptions
    .str.replace(
        r"'id':\s*'[^']+'",
        "'id': '<REQUEST_ID>'",
        regex=True
    )
)

print("Unique raw exception strings        :", exceptions.nunique())
print("Unique normalized exception strings :", normalized.nunique())

print("\nNORMALIZED EXCEPTION DISTRIBUTION")
print("-" * 110)

for i, (exc, count) in enumerate(
    normalized.value_counts().items(),
    start=1
):
    print(f"\n[{i}] count = {count}")
    print(exc)

# ------------------------------------------------------------------
# Provider-error fingerprints
# ------------------------------------------------------------------

exc_all = (
    df["exception"]
    .fillna("")
    .astype(str)
)

fingerprints = {
    "model_not_available":
        exc_all.str.contains(
            "model_not_available",
            case=False,
            regex=False
        ),

    "non_serverless":
        exc_all.str.contains(
            "non-serverless",
            case=False,
            regex=False
        ),

    "dedicated_endpoint":
        exc_all.str.contains(
            "dedicated endpoint",
            case=False,
            regex=False
        ),

    "bad_request":
        exc_all.str.contains(
            "BadRequestError",
            case=False,
            regex=False
        ),

    "rate_limit":
        exc_all.str.contains(
            "RateLimit",
            case=False,
            regex=False
        ),

    "authentication":
        exc_all.str.contains(
            "Authentication",
            case=False,
            regex=False
        ),

    "timeout":
        exc_all.str.contains(
            "timeout",
            case=False,
            regex=False
        ),
}

print("\n" + "-" * 110)
print("PROVIDER-ERROR FINGERPRINTS")
print("-" * 110)

for name, mask in fingerprints.items():
    print(f"{name:24s}: {int((llm & mask).sum())}")

# ------------------------------------------------------------------
# Raw response / adapter evidence
# ------------------------------------------------------------------

llm_rows = df.loc[llm].copy()

raw_present = (
    llm_rows["raw_text"]
    .fillna("")
    .astype(str)
    .str.strip()
    .ne("")
)

adapter_present = (
    llm_rows["adapter_result_json"]
    .fillna("")
    .astype(str)
    .str.strip()
    .ne("")
)

print("\n" + "-" * 110)
print("LLM-ROW RESPONSE EVIDENCE")
print("-" * 110)

print(
    "Raw text present     :",
    int(raw_present.sum())
)

print(
    "Raw text absent      :",
    int((~raw_present).sum())
)

print(
    "Adapter result present:",
    int(adapter_present.sum())
)

print(
    "Adapter result absent :",
    int((~adapter_present).sum())
)

print("\nFinish reasons:")

print(
    llm_rows["finish_reason"]
    .fillna("<EMPTY>")
    .value_counts(dropna=False)
    .to_string()
)

# ------------------------------------------------------------------
# Representative rows only
# ------------------------------------------------------------------

print("\n" + "-" * 110)
print("FIRST 5 A4 EXCEPTION ROWS")
print("-" * 110)

show = df.loc[
    exception_stage,
    [
        "id",
        "expected_valid",
        "llm_invoked",
        "parse_ok",
        "rejection_stage",
        "error_codes_json",
        "exception",
    ]
].head(5)

with pd.option_context(
    "display.max_colwidth", 500,
    "display.width", 250
):
    print(show.to_string(index=False))

# ------------------------------------------------------------------
# Final diagnostic
# ------------------------------------------------------------------

llm_exception_count = int(
    (llm & exception_stage).sum()
)

print("\n" + "=" * 110)
print("DIAGNOSTIC SUMMARY")
print("=" * 110)

print(
    "Every LLM-invoked row ended in exception :",
    llm_exception_count == int(llm.sum())
)

print(
    "Every LLM-invoked row failed parsing     :",
    int((llm & ~parse).sum()) == int(llm.sum())
)

print(
    "Containment violations                   :",
    int(((~accepted) & descriptor).sum())
)

print("\n✓ READ-ONLY A4 DIAGNOSIS COMPLETE")
print("✓ A4 WAS NOT RERUN")
print("✓ NO MODEL/API CALL MADE")
print("✓ NO FILE MODIFIED")
print("✓ DO NOT RUN A5 YET")
print("=" * 110)

STEP A22 — DIAGNOSE A4 ZERO-PARSE EXECUTION

✓ A4 result hash verified
✓ A4 log hash verified
✓ 120 unique cases verified

--------------------------------------------------------------------------------------------------------------
A4 EXECUTION SUMMARY
--------------------------------------------------------------------------------------------------------------
LLM invoked       : 77
Parse success     : 0
Parse failure     : 77
Exception stage   : 77
Accepted          : 0
Descriptor present: 0
Containment violations: 0

--------------------------------------------------------------------------------------------------------------
A4 REJECTION-STAGE DISTRIBUTION
--------------------------------------------------------------------------------------------------------------
rejection_stage
exception                     77
source_gate                   24
constraint_consistency        14
r7_qualitative_source_gate     5

---------------------------------------------------------------------

In [28]:
from pathlib import Path
from datetime import datetime, timezone
import pandas as pd
import hashlib
import json

ROOT = Path("/content/deterministic-intent-translation-journal")
EXP_A = ROOT / "journal_experiments/experiment_A_model_sweep"

A4 = (
    EXP_A /
    "results/raw/A4_Mistral-Small-24B-Instruct-2501_results.csv"
)

A4_LOG = (
    EXP_A /
    "logs/A4_Mistral-Small-24B-Instruct-2501_execution.log"
)

OUT = (
    EXP_A /
    "manifests/A4_unavailable_arm_manifest_v1.json"
)

EXPECTED_A4_SHA = (
    "6babc2b2492473e5ee737e182dd338be2bc23c0cb0f72b5de8b70864d8757125"
)

EXPECTED_LOG_SHA = (
    "122d22c79b9c0fc33b2e3ff6e38e6279d45c1865dc71e727e5bbe675a83cbe68"
)


def sha256(path):
    h = hashlib.sha256()

    with open(path, "rb") as f:
        for block in iter(lambda: f.read(1024 * 1024), b""):
            h.update(block)

    return h.hexdigest()


def as_bool(series):
    if pd.api.types.is_bool_dtype(series):
        return (
            series.astype("boolean")
            .fillna(False)
            .astype(bool)
        )

    return (
        series.astype(str)
        .str.strip()
        .str.lower()
        .map({
            "true": True,
            "false": False,
            "1": True,
            "0": False,
        })
        .astype("boolean")
        .fillna(False)
        .astype(bool)
    )


print("=" * 110)
print("STEP A23 — FREEZE A4 ENDPOINT-UNAVAILABLE ARM")
print("=" * 110)

# ------------------------------------------------------------------
# Integrity
# ------------------------------------------------------------------

assert A4.exists(), "STOP: A4 result missing."
assert A4_LOG.exists(), "STOP: A4 log missing."

assert sha256(A4) == EXPECTED_A4_SHA, (
    "STOP: A4 result changed."
)

assert sha256(A4_LOG) == EXPECTED_LOG_SHA, (
    "STOP: A4 execution log changed."
)

assert not OUT.exists(), (
    "STOP: A4 unavailable-arm manifest already exists."
)

df = pd.read_csv(A4)

assert len(df) == 120
assert df["id"].is_unique

print("\n✓ A4 result verified")
print("✓ A4 execution log verified")
print("✓ 120 unique cases verified")

# ------------------------------------------------------------------
# Reconstruct A22 diagnosis
# ------------------------------------------------------------------

llm = as_bool(df["llm_invoked"])
parse = as_bool(df["parse_ok"])
accepted = as_bool(df["accepted"])

exception_stage = (
    df["rejection_stage"]
    .fillna("")
    .astype(str)
    .str.strip()
    .eq("exception")
)

errors = (
    df["error_codes_json"]
    .fillna("")
    .astype(str)
)

exceptions = (
    df["exception"]
    .fillna("")
    .astype(str)
)

descriptor = (
    df["descriptor_json"]
    .fillna("")
    .astype(str)
    .str.strip()
    .ne("")
)

bad_request = errors.str.contains(
    "BadRequestError",
    case=False,
    regex=False
)

model_not_available = exceptions.str.contains(
    "model_not_available",
    case=False,
    regex=False
)

non_serverless = exceptions.str.contains(
    "non-serverless",
    case=False,
    regex=False
)

dedicated_endpoint = exceptions.str.contains(
    "dedicated endpoint",
    case=False,
    regex=False
)

llm_count = int(llm.sum())
parse_success = int((llm & parse).sum())
parse_failure = int((llm & ~parse).sum())

llm_exception = int(
    (llm & exception_stage).sum()
)

llm_bad_request = int(
    (llm & bad_request).sum()
)

llm_model_unavailable = int(
    (llm & model_not_available).sum()
)

llm_non_serverless = int(
    (llm & non_serverless).sum()
)

llm_dedicated_endpoint = int(
    (llm & dedicated_endpoint).sum()
)

containment_violations = int(
    ((~accepted) & descriptor).sum()
)

print("\n" + "-" * 110)
print("A4 DIAGNOSTIC COUNTS")
print("-" * 110)

print(
    "LLM invoked                         :",
    llm_count
)

print(
    "Parse success                       :",
    parse_success
)

print(
    "Parse failure                       :",
    parse_failure
)

print(
    "LLM-invoked rows at exception stage :",
    llm_exception
)

print(
    "LLM-invoked BadRequestError rows     :",
    llm_bad_request
)

print(
    "model_not_available rows             :",
    llm_model_unavailable
)

print(
    "non-serverless-model rows            :",
    llm_non_serverless
)

print(
    "dedicated-endpoint rows              :",
    llm_dedicated_endpoint
)

print(
    "Containment violations               :",
    containment_violations
)

# ------------------------------------------------------------------
# Required classification evidence
# ------------------------------------------------------------------

assert llm_count == 77
assert parse_success == 0
assert parse_failure == 77

assert llm_exception == 77
assert llm_bad_request == 77
assert llm_model_unavailable == 77
assert llm_non_serverless == 77
assert llm_dedicated_endpoint == 77

assert int(accepted.sum()) == 0
assert int(descriptor.sum()) == 0
assert containment_violations == 0

classification = (
    "UNAVAILABLE_ON_ACCESSIBLE_SERVERLESS_ENDPOINT"
)

print("\n" + "-" * 110)
print("A4 CLASSIFICATION")
print("-" * 110)

print("Status :", classification)

print(
    "\nA4 is not an evaluable model-performance arm."
)

print(
    "All 77 attempted LLM calls failed before a model "
    "response was obtained."
)

print(
    "Provider classification: BadRequestError / "
    "model_not_available."
)

print(
    "Provider explanation: model is non-serverless and "
    "requires a dedicated endpoint."
)

print(
    "\nDerived reject-all accuracy/VAR/IRR values must NOT "
    "be interpreted as Mistral model performance."
)

# ------------------------------------------------------------------
# Freeze A4
# ------------------------------------------------------------------

record = {
    "manifest_id":
        "experiment-A-A4-unavailable-arm-v1",

    "created_utc":
        datetime.now(timezone.utc).isoformat(),

    "arm": "A4",

    "model":
        "mistralai/Mistral-Small-24B-Instruct-2501",

    "preregistered_role":
        "Alternative Mistral model family",

    "status": classification,

    "reason": {
        "provider_error_type":
            "BadRequestError",

        "provider_error_code":
            "model_not_available",

        "provider_message_summary": (
            "The model is non-serverless and requires "
            "a dedicated endpoint."
        ),
    },

    "diagnostics": {
        "total_cases": 120,

        "llm_invoked":
            llm_count,

        "parse_success":
            parse_success,

        "parse_failure":
            parse_failure,

        "llm_exception_rows":
            llm_exception,

        "bad_request_rows":
            llm_bad_request,

        "model_not_available_rows":
            llm_model_unavailable,

        "non_serverless_rows":
            llm_non_serverless,

        "dedicated_endpoint_rows":
            llm_dedicated_endpoint,

        "accepted_rows":
            int(accepted.sum()),

        "descriptor_rows":
            int(descriptor.sum()),

        "containment_violations":
            containment_violations,
    },

    "reporting_policy": {
        "include_arm_in_experiment_accounting":
            True,

        "include_raw_execution_evidence":
            True,

        "include_as_successful_model_performance_arm":
            False,

        "use_accuracy_VAR_IRR_for_model_comparison":
            False,

        "reason": (
            "The LLM endpoint was unavailable. The resulting "
            "reject-all decisions therefore do not measure "
            "semantic-parser performance."
        ),

        "replacement_model_allowed":
            False,

        "rerun_allowed":
            False,
    },

    "artifacts": {
        "raw_result": {
            "path": str(A4.relative_to(ROOT)),
            "sha256": sha256(A4),
        },

        "execution_log": {
            "path": str(A4_LOG.relative_to(ROOT)),
            "sha256": sha256(A4_LOG),
        },
    },

    "experimental_decision": (
        "Preserve A4 as a preregistered unavailable arm. "
        "Do not rerun, substitute, or use derived "
        "accuracy/VAR/IRR as model-performance evidence. "
        "Proceed to preregistered A5."
    ),
}

OUT.write_text(
    json.dumps(record, indent=2) + "\n",
    encoding="utf-8"
)

print("\n" + "-" * 110)
print("FROZEN A4 RECORD")
print("-" * 110)

print(OUT)

print("SHA-256:")
print(sha256(OUT))

print("\n" + "=" * 110)
print("✓ A4 CLASSIFIED AS ENDPOINT-UNAVAILABLE")
print("✓ RAW A4 EXECUTION PRESERVED")
print("✓ A4 NOT USED AS MODEL-PERFORMANCE EVIDENCE")
print("✓ NO RERUN OR MODEL SUBSTITUTION")
print("✓ CONTAINMENT VIOLATIONS = 0")
print("✓ NO MODEL/API CALL MADE")
print("✓ A5 ELIGIBLE AFTER THIS STEP PASSES")
print("=" * 110)

STEP A23 — FREEZE A4 ENDPOINT-UNAVAILABLE ARM

✓ A4 result verified
✓ A4 execution log verified
✓ 120 unique cases verified

--------------------------------------------------------------------------------------------------------------
A4 DIAGNOSTIC COUNTS
--------------------------------------------------------------------------------------------------------------
LLM invoked                         : 77
Parse success                       : 0
Parse failure                       : 77
LLM-invoked rows at exception stage : 77
LLM-invoked BadRequestError rows     : 77
model_not_available rows             : 77
non-serverless-model rows            : 77
dedicated-endpoint rows              : 77
Containment violations               : 0

--------------------------------------------------------------------------------------------------------------
A4 CLASSIFICATION
--------------------------------------------------------------------------------------------------------------
Status : UNAVAILABL

In [29]:
import subprocess
from pathlib import Path
import hashlib
import os
import time
import pandas as pd

ROOT = Path("/content/deterministic-intent-translation-journal")

EXP_A = (
    ROOT /
    "journal_experiments" /
    "experiment_A_model_sweep"
)

RUNNER = ROOT / (
    "experiment-2b/implementation/e3_v3/"
    "run_experiment2b_e3v3_r7_dev_v1.py"
)

DATASET = ROOT / (
    "experiment-2b/reproducibility/"
    "e3v3_heldout_benchmark_v1.0/"
    "e3v3_heldout_benchmark_120_v1.csv"
)

A4_MANIFEST = (
    EXP_A /
    "manifests/A4_unavailable_arm_manifest_v1.json"
)

CONTAINMENT_MANIFEST = (
    EXP_A /
    "manifests/containment_definition_A1_A2_v1.json"
)

A5_OUT = (
    EXP_A /
    "results/raw/A5_DeepSeek-V3.1_results.csv"
)

A5_LOG = (
    EXP_A /
    "logs/A5_DeepSeek-V3.1_execution.log"
)

MODEL = "deepseek-ai/DeepSeek-V3.1"

EXPECTED_RUNNER_SHA = (
    "40c035cb864978c718a72fcfdfbb164d262529960b0ae78242d5194147cd8257"
)

EXPECTED_DATASET_SHA = (
    "84576e7d99a2721e61ded5a14bdafeb96707b068397243bbed32aa735496d4de"
)

EXPECTED_A4_MANIFEST_SHA = (
    "2cacef85717ab3fe5ed4f9bc55926fd71eb1dd7c36183784d2b1fb0b84606fb6"
)

EXPECTED_CONTAINMENT_SHA = (
    "cd31ea28ffa8a2f82121d1c4cf0e5478425d194299d309e603069545d75d4832"
)


def sha256(path):
    h = hashlib.sha256()

    with open(path, "rb") as f:
        for block in iter(lambda: f.read(1024 * 1024), b""):
            h.update(block)

    return h.hexdigest()


def as_bool(series):
    if pd.api.types.is_bool_dtype(series):
        return (
            series.astype("boolean")
            .fillna(False)
            .astype(bool)
        )

    return (
        series.astype(str)
        .str.strip()
        .str.lower()
        .map({
            "true": True,
            "false": False,
            "1": True,
            "0": False,
        })
        .astype("boolean")
        .fillna(False)
        .astype(bool)
    )


print("=" * 100)
print("STEP A24 — EXECUTE A5 MODEL-SWEEP ARM")
print("=" * 100)

# ------------------------------------------------------------------
# Integrity checks
# ------------------------------------------------------------------

assert RUNNER.exists()
assert DATASET.exists()
assert A4_MANIFEST.exists()
assert CONTAINMENT_MANIFEST.exists()

assert sha256(RUNNER) == EXPECTED_RUNNER_SHA, (
    "STOP: frozen runner changed."
)

assert sha256(DATASET) == EXPECTED_DATASET_SHA, (
    "STOP: frozen benchmark changed."
)

assert sha256(A4_MANIFEST) == EXPECTED_A4_MANIFEST_SHA, (
    "STOP: frozen A4 manifest changed."
)

assert sha256(CONTAINMENT_MANIFEST) == EXPECTED_CONTAINMENT_SHA, (
    "STOP: frozen containment definition changed."
)

assert os.environ.get("TOGETHER_API_KEY"), (
    "STOP: TOGETHER_API_KEY missing."
)

assert not A5_OUT.exists(), (
    f"STOP: A5 result already exists:\n{A5_OUT}\n"
    "Do not overwrite or rerun."
)

assert not A5_LOG.exists(), (
    f"STOP: A5 log already exists:\n{A5_LOG}\n"
    "Do not overwrite or rerun."
)

print("\n✓ Frozen runner verified")
print("✓ Frozen benchmark verified")
print("✓ Frozen A4 record verified")
print("✓ Frozen containment definition verified")
print("✓ TOGETHER_API_KEY present")
print("✓ A5 result path clean")
print("✓ A5 log path clean")

# ------------------------------------------------------------------
# Preregistered A5
# ------------------------------------------------------------------

print("\nPRE-REGISTERED A5")
print("-" * 100)

print("Model :", MODEL)
print("Role  : Alternative DeepSeek model family")
print("Cases : 120")

print("\nOnly experimental variable being changed:")
print("  model ->", MODEL)

print("\nUnchanged:")
print("  ✓ runner")
print("  ✓ benchmark")
print("  ✓ prompt")
print("  ✓ deterministic gates")
print("  ✓ thresholds")
print("  ✓ labels")
print("  ✓ containment definition")

# ------------------------------------------------------------------
# Execute exactly once
# ------------------------------------------------------------------

cmd = [
    "python",
    str(RUNNER),
    "--dataset",
    str(DATASET),
    "--out",
    str(A5_OUT),
    "--model",
    MODEL,
]

print("\n" + "-" * 100)
print("EXECUTING A5")
print("-" * 100)

print("\nCommand:")
print(
    " ".join(
        f'"{x}"' if " " in x else x
        for x in cmd
    )
)

print("\nRunning frozen 120-case benchmark ONCE...\n")

t0 = time.time()

process = subprocess.run(
    cmd,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    env=os.environ.copy(),
)

elapsed = time.time() - t0

# Always preserve execution evidence.
A5_LOG.write_text(
    process.stdout or "",
    encoding="utf-8"
)

print(process.stdout)

print("\n" + "-" * 100)
print("EXECUTION STATUS")
print("-" * 100)

print("Process return code :", process.returncode)
print(f"Wall-clock seconds  : {elapsed:.2f}")

print("\nExecution log:")
print(A5_LOG)

print("Log SHA-256:")
print(sha256(A5_LOG))

# ------------------------------------------------------------------
# Process-level failure
# ------------------------------------------------------------------

if process.returncode != 0:

    print("\n" + "=" * 100)
    print("✗ A5 PROCESS-LEVEL EXECUTION FAILURE")
    print("✗ FAILURE LOG PRESERVED")
    print("✗ DO NOT RERUN A5")
    print("✗ DO NOT SUBSTITUTE ANOTHER MODEL")
    print("✗ DO NOT START A6")
    print("✗ SEND THIS OUTPUT FOR DIAGNOSIS")
    print("=" * 100)

    raise RuntimeError(
        f"A5 runner exited with return code "
        f"{process.returncode}"
    )

# ------------------------------------------------------------------
# Result validation
# ------------------------------------------------------------------

assert A5_OUT.exists(), (
    "STOP: A5 returned success but result CSV is missing."
)

df = pd.read_csv(A5_OUT)

assert len(df) == 120, (
    f"STOP: expected 120 rows, found {len(df)}."
)

assert df["id"].is_unique, (
    "STOP: A5 result contains duplicate case IDs."
)

print("\nA5 RESULT")
print("-" * 100)

print("Rows       :", len(df))
print("Unique IDs :", df["id"].nunique())

print("\nResult:")
print(A5_OUT)

print("Result SHA-256:")
print(sha256(A5_OUT))

# ------------------------------------------------------------------
# Lightweight endpoint/execution diagnostic
# ------------------------------------------------------------------

llm = as_bool(df["llm_invoked"])
parse = as_bool(df["parse_ok"])

exception_stage = (
    df["rejection_stage"]
    .fillna("")
    .astype(str)
    .str.strip()
    .eq("exception")
)

print("\nLIGHTWEIGHT EXECUTION DIAGNOSTIC")
print("-" * 100)

print("LLM invoked    :", int(llm.sum()))
print("Parse success  :", int((llm & parse).sum()))
print("Parse failure  :", int((llm & ~parse).sum()))
print(
    "Exception rows :",
    int(exception_stage.sum())
)

print("\nRejection stages:")

print(
    df["rejection_stage"]
    .fillna("<ACCEPTED>")
    .value_counts(dropna=False)
    .to_string()
)

# ------------------------------------------------------------------
# Guard against accidentally interpreting endpoint failure
# ------------------------------------------------------------------

if int((llm & parse).sum()) == 0:

    print("\n" + "!" * 100)
    print("A5 WARNING — ZERO SUCCESSFUL LLM PARSES")
    print("Do NOT interpret accuracy/VAR/IRR.")
    print("A5 requires endpoint/exception diagnosis first.")
    print("!" * 100)

else:

    print("\n✓ A5 produced successful LLM parses.")
    print("✓ A5 is eligible for full metric/containment analysis.")

print("\n" + "=" * 100)
print("✓ A5 EXECUTION ATTEMPT COMPLETE")
print("✓ 120 CASES WRITTEN")
print("✓ RAW A5 RESULT PRESERVED")
print("✓ EXECUTION LOG PRESERVED")
print("✓ NO PERFORMANCE INTERPRETATION MADE")
print("✓ DO NOT RUN A6 YET")
print("=" * 100)

STEP A24 — EXECUTE A5 MODEL-SWEEP ARM

✓ Frozen runner verified
✓ Frozen benchmark verified
✓ Frozen A4 record verified
✓ Frozen containment definition verified
✓ TOGETHER_API_KEY present
✓ A5 result path clean
✓ A5 log path clean

PRE-REGISTERED A5
----------------------------------------------------------------------------------------------------
Model : deepseek-ai/DeepSeek-V3.1
Role  : Alternative DeepSeek model family
Cases : 120

Only experimental variable being changed:
  model -> deepseek-ai/DeepSeek-V3.1

Unchanged:
  ✓ runner
  ✓ benchmark
  ✓ prompt
  ✓ deterministic gates
  ✓ thresholds
  ✓ labels
  ✓ containment definition

----------------------------------------------------------------------------------------------------
EXECUTING A5
----------------------------------------------------------------------------------------------------

Command:
python /content/deterministic-intent-translation-journal/experiment-2b/implementation/e3_v3/run_experiment2b_e3v3_r7_dev_v1.py --d

In [30]:
from pathlib import Path
import pandas as pd
import hashlib

ROOT = Path("/content/deterministic-intent-translation-journal")
EXP_A = ROOT / "journal_experiments/experiment_A_model_sweep"

A5 = (
    EXP_A /
    "results/raw/A5_DeepSeek-V3.1_results.csv"
)

A5_LOG = (
    EXP_A /
    "logs/A5_DeepSeek-V3.1_execution.log"
)

EXPECTED_A5_SHA = (
    "8d6c561218991dc1d3a0f47601bd5a8c01bce918be0b3c1133333de5eb702f41"
)

EXPECTED_LOG_SHA = (
    "c5d4555c4a55e6b1578bc16135b58c17a6e398c2c34d27101767dd3e446b58b1"
)


def sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(1024 * 1024), b""):
            h.update(block)
    return h.hexdigest()


def as_bool(series):
    if pd.api.types.is_bool_dtype(series):
        return (
            series.astype("boolean")
            .fillna(False)
            .astype(bool)
        )

    return (
        series.astype(str)
        .str.strip()
        .str.lower()
        .map({
            "true": True,
            "false": False,
            "1": True,
            "0": False,
        })
        .astype("boolean")
        .fillna(False)
        .astype(bool)
    )


print("=" * 110)
print("STEP A25 — DIAGNOSE A5 ZERO-PARSE EXECUTION")
print("=" * 110)

# ------------------------------------------------------------------
# Integrity
# ------------------------------------------------------------------

assert A5.exists(), "STOP: A5 result missing."
assert A5_LOG.exists(), "STOP: A5 log missing."

assert sha256(A5) == EXPECTED_A5_SHA, (
    "STOP: A5 result changed."
)

assert sha256(A5_LOG) == EXPECTED_LOG_SHA, (
    "STOP: A5 execution log changed."
)

df = pd.read_csv(A5)

assert len(df) == 120
assert df["id"].is_unique

print("\n✓ A5 result hash verified")
print("✓ A5 log hash verified")
print("✓ 120 unique cases verified")

# ------------------------------------------------------------------
# Execution state
# ------------------------------------------------------------------

llm = as_bool(df["llm_invoked"])
parse = as_bool(df["parse_ok"])
accepted = as_bool(df["accepted"])

exception_stage = (
    df["rejection_stage"]
    .fillna("")
    .astype(str)
    .str.strip()
    .eq("exception")
)

descriptor = (
    df["descriptor_json"]
    .fillna("")
    .astype(str)
    .str.strip()
    .ne("")
)

print("\n" + "-" * 110)
print("A5 EXECUTION SUMMARY")
print("-" * 110)

print("LLM invoked        :", int(llm.sum()))
print("Parse success      :", int((llm & parse).sum()))
print("Parse failure      :", int((llm & ~parse).sum()))
print("Exception stage    :", int(exception_stage.sum()))
print("Accepted           :", int(accepted.sum()))
print("Descriptor present :", int(descriptor.sum()))

containment = int(
    ((~accepted) & descriptor).sum()
)

print("Containment violations:", containment)

# ------------------------------------------------------------------
# Rejection stages / error codes
# ------------------------------------------------------------------

print("\n" + "-" * 110)
print("A5 REJECTION-STAGE DISTRIBUTION")
print("-" * 110)

print(
    df["rejection_stage"]
    .fillna("<ACCEPTED>")
    .value_counts(dropna=False)
    .to_string()
)

print("\n" + "-" * 110)
print("A5 ERROR-CODE DISTRIBUTION")
print("-" * 110)

print(
    df["error_codes_json"]
    .fillna("<EMPTY>")
    .value_counts(dropna=False)
    .to_string()
)

# ------------------------------------------------------------------
# Exception normalization
# ------------------------------------------------------------------

exceptions = (
    df.loc[exception_stage, "exception"]
    .fillna("")
    .astype(str)
    .str.strip()
)

normalized = (
    exceptions
    .str.replace(
        r"'id':\s*'[^']+'",
        "'id': '<REQUEST_ID>'",
        regex=True
    )
)

print("\n" + "-" * 110)
print("A5 EXCEPTION SUMMARY")
print("-" * 110)

print("Exception rows                       :", len(exceptions))
print("Unique raw exception strings         :", exceptions.nunique())
print("Unique normalized exception strings  :", normalized.nunique())

print("\nNORMALIZED EXCEPTION DISTRIBUTION")
print("-" * 110)

for i, (exc, count) in enumerate(
    normalized.value_counts().items(),
    start=1
):
    print(f"\n[{i}] count = {count}")
    print(exc)

# ------------------------------------------------------------------
# Provider error fingerprints
# ------------------------------------------------------------------

exc_all = (
    df["exception"]
    .fillna("")
    .astype(str)
)

fingerprints = {
    "model_not_available":
        exc_all.str.contains(
            "model_not_available",
            case=False,
            regex=False
        ),

    "non_serverless":
        exc_all.str.contains(
            "non-serverless",
            case=False,
            regex=False
        ),

    "dedicated_endpoint":
        exc_all.str.contains(
            "dedicated endpoint",
            case=False,
            regex=False
        ),

    "bad_request":
        exc_all.str.contains(
            "BadRequestError",
            case=False,
            regex=False
        ),

    "rate_limit":
        exc_all.str.contains(
            "RateLimit",
            case=False,
            regex=False
        ),

    "authentication":
        exc_all.str.contains(
            "Authentication",
            case=False,
            regex=False
        ),

    "timeout":
        exc_all.str.contains(
            "timeout",
            case=False,
            regex=False
        ),

    "not_found":
        exc_all.str.contains(
            "NotFound",
            case=False,
            regex=False
        ),

    "unsupported":
        exc_all.str.contains(
            "unsupported",
            case=False,
            regex=False
        ),
}

print("\n" + "-" * 110)
print("PROVIDER-ERROR FINGERPRINTS")
print("-" * 110)

for name, mask in fingerprints.items():
    print(
        f"{name:24s}: "
        f"{int((llm & mask).sum())}"
    )

# ------------------------------------------------------------------
# Response evidence
# ------------------------------------------------------------------

llm_rows = df.loc[llm].copy()

raw_present = (
    llm_rows["raw_text"]
    .fillna("")
    .astype(str)
    .str.strip()
    .ne("")
)

adapter_present = (
    llm_rows["adapter_result_json"]
    .fillna("")
    .astype(str)
    .str.strip()
    .ne("")
)

print("\n" + "-" * 110)
print("LLM-ROW RESPONSE EVIDENCE")
print("-" * 110)

print(
    "Raw text present      :",
    int(raw_present.sum())
)

print(
    "Raw text absent       :",
    int((~raw_present).sum())
)

print(
    "Adapter result present:",
    int(adapter_present.sum())
)

print(
    "Adapter result absent :",
    int((~adapter_present).sum())
)

print("\nFinish reasons:")

print(
    llm_rows["finish_reason"]
    .fillna("<EMPTY>")
    .value_counts(dropna=False)
    .to_string()
)

# ------------------------------------------------------------------
# Representative exception rows
# ------------------------------------------------------------------

print("\n" + "-" * 110)
print("FIRST 5 A5 EXCEPTION ROWS")
print("-" * 110)

show = df.loc[
    exception_stage,
    [
        "id",
        "expected_valid",
        "llm_invoked",
        "parse_ok",
        "rejection_stage",
        "error_codes_json",
        "exception",
    ]
].head(5)

with pd.option_context(
    "display.max_colwidth", 500,
    "display.width", 250
):
    print(show.to_string(index=False))

# ------------------------------------------------------------------
# Diagnostic summary
# ------------------------------------------------------------------

llm_count = int(llm.sum())
llm_exception_count = int(
    (llm & exception_stage).sum()
)

print("\n" + "=" * 110)
print("DIAGNOSTIC SUMMARY")
print("=" * 110)

print(
    "Every LLM-invoked row ended in exception :",
    llm_exception_count == llm_count
)

print(
    "Every LLM-invoked row failed parsing     :",
    int((llm & ~parse).sum()) == llm_count
)

print(
    "Containment violations                   :",
    containment
)

print("\n✓ READ-ONLY A5 DIAGNOSIS COMPLETE")
print("✓ A5 WAS NOT RERUN")
print("✓ NO MODEL/API CALL MADE")
print("✓ NO FILE MODIFIED")
print("✓ DO NOT RUN A6 YET")
print("=" * 110)

STEP A25 — DIAGNOSE A5 ZERO-PARSE EXECUTION

✓ A5 result hash verified
✓ A5 log hash verified
✓ 120 unique cases verified

--------------------------------------------------------------------------------------------------------------
A5 EXECUTION SUMMARY
--------------------------------------------------------------------------------------------------------------
LLM invoked        : 77
Parse success      : 0
Parse failure      : 77
Exception stage    : 77
Accepted           : 0
Descriptor present : 0
Containment violations: 0

--------------------------------------------------------------------------------------------------------------
A5 REJECTION-STAGE DISTRIBUTION
--------------------------------------------------------------------------------------------------------------
rejection_stage
exception                     77
source_gate                   24
constraint_consistency        14
r7_qualitative_source_gate     5

---------------------------------------------------------------

In [31]:
from pathlib import Path
from datetime import datetime, timezone
import pandas as pd
import hashlib
import json

ROOT = Path("/content/deterministic-intent-translation-journal")
EXP_A = ROOT / "journal_experiments/experiment_A_model_sweep"

A5 = EXP_A / "results/raw/A5_DeepSeek-V3.1_results.csv"
A5_LOG = EXP_A / "logs/A5_DeepSeek-V3.1_execution.log"

OUT = EXP_A / "manifests/A5_unavailable_arm_manifest_v1.json"

EXPECTED_A5_SHA = (
    "8d6c561218991dc1d3a0f47601bd5a8c01bce918be0b3c1133333de5eb702f41"
)

EXPECTED_LOG_SHA = (
    "c5d4555c4a55e6b1578bc16135b58c17a6e398c2c34d27101767dd3e446b58b1"
)


def sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(1024 * 1024), b""):
            h.update(block)
    return h.hexdigest()


def as_bool(series):
    if pd.api.types.is_bool_dtype(series):
        return (
            series.astype("boolean")
            .fillna(False)
            .astype(bool)
        )

    return (
        series.astype(str)
        .str.strip()
        .str.lower()
        .map({
            "true": True,
            "false": False,
            "1": True,
            "0": False,
        })
        .astype("boolean")
        .fillna(False)
        .astype(bool)
    )


print("=" * 110)
print("STEP A26 — FREEZE A5 ENDPOINT-UNAVAILABLE ARM")
print("=" * 110)

# ------------------------------------------------------------------
# Integrity
# ------------------------------------------------------------------

assert A5.exists(), "STOP: A5 result missing."
assert A5_LOG.exists(), "STOP: A5 execution log missing."

assert sha256(A5) == EXPECTED_A5_SHA, (
    "STOP: A5 result changed."
)

assert sha256(A5_LOG) == EXPECTED_LOG_SHA, (
    "STOP: A5 execution log changed."
)

assert not OUT.exists(), (
    "STOP: A5 unavailable-arm manifest already exists."
)

df = pd.read_csv(A5)

assert len(df) == 120
assert df["id"].is_unique

print("\n✓ A5 result verified")
print("✓ A5 execution log verified")
print("✓ 120 unique cases verified")

# ------------------------------------------------------------------
# Reconstruct diagnosis from frozen result
# ------------------------------------------------------------------

llm = as_bool(df["llm_invoked"])
parse = as_bool(df["parse_ok"])
accepted = as_bool(df["accepted"])

exception_stage = (
    df["rejection_stage"]
    .fillna("")
    .astype(str)
    .str.strip()
    .eq("exception")
)

errors = (
    df["error_codes_json"]
    .fillna("")
    .astype(str)
)

exceptions = (
    df["exception"]
    .fillna("")
    .astype(str)
)

descriptor = (
    df["descriptor_json"]
    .fillna("")
    .astype(str)
    .str.strip()
    .ne("")
)

bad_request = errors.str.contains(
    "BadRequestError",
    case=False,
    regex=False
)

model_not_available = exceptions.str.contains(
    "model_not_available",
    case=False,
    regex=False
)

non_serverless = exceptions.str.contains(
    "non-serverless",
    case=False,
    regex=False
)

dedicated_endpoint = exceptions.str.contains(
    "dedicated endpoint",
    case=False,
    regex=False
)

llm_count = int(llm.sum())
parse_success = int((llm & parse).sum())
parse_failure = int((llm & ~parse).sum())

llm_exception = int(
    (llm & exception_stage).sum()
)

llm_bad_request = int(
    (llm & bad_request).sum()
)

llm_model_unavailable = int(
    (llm & model_not_available).sum()
)

llm_non_serverless = int(
    (llm & non_serverless).sum()
)

llm_dedicated_endpoint = int(
    (llm & dedicated_endpoint).sum()
)

containment_violations = int(
    ((~accepted) & descriptor).sum()
)

print("\n" + "-" * 110)
print("A5 DIAGNOSTIC COUNTS")
print("-" * 110)

print("LLM invoked                         :", llm_count)
print("Parse success                       :", parse_success)
print("Parse failure                       :", parse_failure)
print("LLM-invoked rows at exception stage :", llm_exception)
print("LLM-invoked BadRequestError rows     :", llm_bad_request)
print("model_not_available rows             :", llm_model_unavailable)
print("non-serverless-model rows            :", llm_non_serverless)
print("dedicated-endpoint rows              :", llm_dedicated_endpoint)
print("Containment violations               :", containment_violations)

# ------------------------------------------------------------------
# Required evidence
# ------------------------------------------------------------------

assert llm_count == 77
assert parse_success == 0
assert parse_failure == 77

assert llm_exception == 77
assert llm_bad_request == 77
assert llm_model_unavailable == 77
assert llm_non_serverless == 77
assert llm_dedicated_endpoint == 77

assert int(accepted.sum()) == 0
assert int(descriptor.sum()) == 0
assert containment_violations == 0

classification = (
    "UNAVAILABLE_ON_ACCESSIBLE_SERVERLESS_ENDPOINT"
)

print("\n" + "-" * 110)
print("A5 CLASSIFICATION")
print("-" * 110)

print("Status :", classification)

print(
    "\nA5 is not an evaluable model-performance arm."
)

print(
    "All 77 attempted LLM calls failed before a model "
    "response was obtained."
)

print(
    "Provider classification: BadRequestError / "
    "model_not_available."
)

print(
    "Provider explanation: model is non-serverless and "
    "requires a dedicated endpoint."
)

print(
    "\nDerived reject-all accuracy/VAR/IRR values must NOT "
    "be interpreted as DeepSeek model performance."
)

# ------------------------------------------------------------------
# Freeze
# ------------------------------------------------------------------

record = {
    "manifest_id":
        "experiment-A-A5-unavailable-arm-v1",

    "created_utc":
        datetime.now(timezone.utc).isoformat(),

    "arm": "A5",

    "model":
        "deepseek-ai/DeepSeek-V3.1",

    "preregistered_role":
        "Alternative DeepSeek model family",

    "status":
        classification,

    "reason": {
        "provider_error_type":
            "BadRequestError",

        "provider_error_code":
            "model_not_available",

        "provider_message_summary": (
            "The model is non-serverless and requires "
            "a dedicated endpoint."
        ),
    },

    "diagnostics": {
        "total_cases":
            120,

        "llm_invoked":
            llm_count,

        "parse_success":
            parse_success,

        "parse_failure":
            parse_failure,

        "llm_exception_rows":
            llm_exception,

        "bad_request_rows":
            llm_bad_request,

        "model_not_available_rows":
            llm_model_unavailable,

        "non_serverless_rows":
            llm_non_serverless,

        "dedicated_endpoint_rows":
            llm_dedicated_endpoint,

        "accepted_rows":
            int(accepted.sum()),

        "descriptor_rows":
            int(descriptor.sum()),

        "containment_violations":
            containment_violations,
    },

    "reporting_policy": {
        "include_arm_in_experiment_accounting":
            True,

        "include_raw_execution_evidence":
            True,

        "include_as_successful_model_performance_arm":
            False,

        "use_accuracy_VAR_IRR_for_model_comparison":
            False,

        "reason": (
            "The LLM endpoint was unavailable. "
            "The resulting reject-all decisions do not "
            "measure semantic-parser performance."
        ),

        "replacement_model_allowed":
            False,

        "rerun_allowed":
            False,
    },

    "artifacts": {
        "raw_result": {
            "path": str(A5.relative_to(ROOT)),
            "sha256": sha256(A5),
        },

        "execution_log": {
            "path": str(A5_LOG.relative_to(ROOT)),
            "sha256": sha256(A5_LOG),
        },
    },

    "experimental_decision": (
        "Preserve A5 as a preregistered unavailable arm. "
        "Do not rerun, substitute, or use derived "
        "accuracy/VAR/IRR as model-performance evidence. "
        "Proceed to preregistered A6 availability check."
    ),
}

OUT.write_text(
    json.dumps(record, indent=2) + "\n",
    encoding="utf-8"
)

print("\n" + "-" * 110)
print("FROZEN A5 RECORD")
print("-" * 110)

print(OUT)

print("SHA-256:")
print(sha256(OUT))

print("\n" + "=" * 110)
print("✓ A5 CLASSIFIED AS ENDPOINT-UNAVAILABLE")
print("✓ RAW A5 EXECUTION PRESERVED")
print("✓ A5 NOT USED AS MODEL-PERFORMANCE EVIDENCE")
print("✓ NO RERUN OR MODEL SUBSTITUTION")
print("✓ CONTAINMENT VIOLATIONS = 0")
print("✓ NO MODEL/API CALL MADE")
print("✓ A6 AVAILABILITY CHECK ELIGIBLE AFTER THIS STEP")
print("=" * 110)

STEP A26 — FREEZE A5 ENDPOINT-UNAVAILABLE ARM

✓ A5 result verified
✓ A5 execution log verified
✓ 120 unique cases verified

--------------------------------------------------------------------------------------------------------------
A5 DIAGNOSTIC COUNTS
--------------------------------------------------------------------------------------------------------------
LLM invoked                         : 77
Parse success                       : 0
Parse failure                       : 77
LLM-invoked rows at exception stage : 77
LLM-invoked BadRequestError rows     : 77
model_not_available rows             : 77
non-serverless-model rows            : 77
dedicated-endpoint rows              : 77
Containment violations               : 0

--------------------------------------------------------------------------------------------------------------
A5 CLASSIFICATION
--------------------------------------------------------------------------------------------------------------
Status : UNAVAILABL

In [32]:
import os
import json
import hashlib
from pathlib import Path
from together import Together

ROOT = Path("/content/deterministic-intent-translation-journal")
EXP_A = ROOT / "journal_experiments/experiment_A_model_sweep"

A5_MANIFEST = (
    EXP_A /
    "manifests/A5_unavailable_arm_manifest_v1.json"
)

EXPECTED_A5_MANIFEST_SHA = (
    "89e7f944318bf389ecfbf20b29e9e663c433c8118611c55d66045b86b6c19222"
)

MODEL = "openai/gpt-oss-20b"


def sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(1024 * 1024), b""):
            h.update(block)
    return h.hexdigest()


print("=" * 110)
print("STEP A27 — A6 METADATA-ONLY AVAILABILITY INSPECTION")
print("=" * 110)

# ------------------------------------------------------------------
# Integrity
# ------------------------------------------------------------------

assert A5_MANIFEST.exists(), (
    "STOP: A5 frozen manifest missing."
)

assert sha256(A5_MANIFEST) == EXPECTED_A5_MANIFEST_SHA, (
    "STOP: A5 frozen manifest changed."
)

assert os.environ.get("TOGETHER_API_KEY"), (
    "STOP: TOGETHER_API_KEY missing."
)

print("\n✓ Frozen A5 manifest verified")
print("✓ TOGETHER_API_KEY present")
print("✓ No benchmark case will be submitted")
print("✓ No chat/completions request will be made")

# ------------------------------------------------------------------
# Catalogue metadata only
# ------------------------------------------------------------------

client = Together(
    api_key=os.environ["TOGETHER_API_KEY"]
)

models_response = client.models.list()

# Handle common SDK response shapes.
if hasattr(models_response, "data"):
    models = models_response.data
else:
    models = models_response

matches = []

for model in models:

    # Pydantic / SDK object
    if hasattr(model, "model_dump"):
        d = model.model_dump()

    elif hasattr(model, "dict"):
        d = model.dict()

    elif isinstance(model, dict):
        d = model

    else:
        # Last-resort introspection without inference.
        d = {
            k: v
            for k, v in vars(model).items()
            if not k.startswith("_")
        }

    model_id = (
        d.get("id")
        or d.get("model")
        or d.get("name")
    )

    if model_id == MODEL:
        matches.append(d)

print("\n" + "-" * 110)
print("A6 CATALOGUE LOOKUP")
print("-" * 110)

print("Target model :", MODEL)
print("Matches      :", len(matches))

if not matches:
    print("\n✗ A6 model not present in current catalogue.")
    print("Do NOT run A6 benchmark.")
else:
    print("\n✓ A6 model present in current catalogue")

    metadata = matches[0]

    print("\n" + "-" * 110)
    print("AVAILABLE METADATA FIELDS")
    print("-" * 110)

    for key in sorted(metadata.keys()):
        value = metadata[key]

        # Avoid excessively large nested dumps.
        text_value = repr(value)

        if len(text_value) > 1000:
            text_value = text_value[:1000] + "...<TRUNCATED>"

        print(f"{key}: {text_value}")

    # --------------------------------------------------------------
    # Search metadata recursively for availability/serverless hints
    # --------------------------------------------------------------

    metadata_text = json.dumps(
        metadata,
        default=str
    ).lower()

    keywords = [
        "serverless",
        "dedicated",
        "available",
        "availability",
        "endpoint",
        "inference",
        "chat",
    ]

    print("\n" + "-" * 110)
    print("AVAILABILITY-RELATED KEYWORD PRESENCE")
    print("-" * 110)

    for keyword in keywords:
        print(
            f"{keyword:15s}:",
            keyword in metadata_text
        )

    # --------------------------------------------------------------
    # Print selected likely fields if present
    # --------------------------------------------------------------

    likely_fields = [
        "id",
        "display_name",
        "type",
        "object",
        "context_length",
        "pricing",
        "config",
        "availability",
        "serverless",
        "dedicated",
        "endpoint",
        "capabilities",
    ]

    print("\n" + "-" * 110)
    print("SELECTED A6 METADATA")
    print("-" * 110)

    for field in likely_fields:
        if field in metadata:
            print(
                f"{field}: "
                f"{json.dumps(metadata[field], default=str)}"
            )

print("\n" + "=" * 110)
print("✓ METADATA-ONLY CHECK COMPLETE")
print("✓ NO BENCHMARK CASE USED")
print("✓ NO INFERENCE REQUEST MADE")
print("✓ NO EXPERIMENT RESULT MODIFIED")
print("✓ DO NOT RUN A6 YET")
print("=" * 110)

STEP A27 — A6 METADATA-ONLY AVAILABILITY INSPECTION

✓ Frozen A5 manifest verified
✓ TOGETHER_API_KEY present
✓ No benchmark case will be submitted
✓ No chat/completions request will be made

--------------------------------------------------------------------------------------------------------------
A6 CATALOGUE LOOKUP
--------------------------------------------------------------------------------------------------------------
Target model : openai/gpt-oss-20b
Matches      : 1

✓ A6 model present in current catalogue

--------------------------------------------------------------------------------------------------------------
AVAILABLE METADATA FIELDS
--------------------------------------------------------------------------------------------------------------
config: {'chat_template': None, 'stop': [], 'bos_token': None, 'eos_token': None}
context_length: 131072
created: 1774480577
display_name: 'OpenAI GPT-OSS 20B'
id: 'openai/gpt-oss-20b'
license: 'apache-2.0'
link: 'https://hug

In [33]:
from pathlib import Path
from datetime import datetime, timezone
import os
import json
import hashlib
from together import Together

ROOT = Path("/content/deterministic-intent-translation-journal")
EXP_A = ROOT / "journal_experiments/experiment_A_model_sweep"

A5_MANIFEST = (
    EXP_A /
    "manifests/A5_unavailable_arm_manifest_v1.json"
)

PROBE_RECORD = (
    EXP_A /
    "manifests/A6_prebenchmark_availability_probe_v1.json"
)

EXPECTED_A5_MANIFEST_SHA = (
    "89e7f944318bf389ecfbf20b29e9e663c433c8118611c55d66045b86b6c19222"
)

MODEL = "openai/gpt-oss-20b"

PROBE_TEXT = "Reply with exactly: OK"


def sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(1024 * 1024), b""):
            h.update(block)
    return h.hexdigest()


print("=" * 110)
print("STEP A28 — A6 SINGLE NON-BENCHMARK AVAILABILITY PROBE")
print("=" * 110)

# ------------------------------------------------------------------
# Integrity
# ------------------------------------------------------------------

assert A5_MANIFEST.exists(), (
    "STOP: frozen A5 manifest missing."
)

assert sha256(A5_MANIFEST) == EXPECTED_A5_MANIFEST_SHA, (
    "STOP: frozen A5 manifest changed."
)

assert os.environ.get("TOGETHER_API_KEY"), (
    "STOP: TOGETHER_API_KEY missing."
)

assert not PROBE_RECORD.exists(), (
    "STOP: A6 probe record already exists. "
    "Do not repeat the probe."
)

print("\n✓ Frozen A5 manifest verified")
print("✓ API key present")
print("✓ Probe record path clean")

print("\nA6 MODEL")
print("-" * 110)
print(MODEL)

print("\nNON-BENCHMARK PROBE")
print("-" * 110)
print(PROBE_TEXT)

print("\nThis probe:")
print("  ✓ contains no held-out benchmark case")
print("  ✓ contains no network-slicing intent")
print("  ✓ does not evaluate model accuracy")
print("  ✓ is performed once only")

# ------------------------------------------------------------------
# Exactly one inference request
# ------------------------------------------------------------------

client = Together(
    api_key=os.environ["TOGETHER_API_KEY"]
)

status = None
response_text = None
finish_reason = None
exception_type = None
exception_text = None

try:

    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {
                "role": "user",
                "content": PROBE_TEXT,
            }
        ],
        temperature=0,
        max_tokens=16,
    )

    status = "INFERENCE_REQUEST_SUCCEEDED"

    if getattr(response, "choices", None):

        choice = response.choices[0]

        finish_reason = getattr(
            choice,
            "finish_reason",
            None
        )

        message = getattr(
            choice,
            "message",
            None
        )

        if message is not None:
            response_text = getattr(
                message,
                "content",
                None
            )

except Exception as exc:

    status = "INFERENCE_REQUEST_FAILED"
    exception_type = type(exc).__name__
    exception_text = repr(exc)

# ------------------------------------------------------------------
# Record probe evidence
# ------------------------------------------------------------------

record = {
    "manifest_id":
        "experiment-A-A6-prebenchmark-availability-probe-v1",

    "created_utc":
        datetime.now(timezone.utc).isoformat(),

    "arm":
        "A6",

    "model":
        MODEL,

    "purpose":
        "Pre-benchmark endpoint availability check only",

    "probe": {
        "text":
            PROBE_TEXT,

        "contains_benchmark_case":
            False,

        "contains_network_slicing_intent":
            False,

        "inference_request_count":
            1,
    },

    "result": {
        "status":
            status,

        "finish_reason":
            finish_reason,

        "response_text":
            response_text,

        "exception_type":
            exception_type,

        "exception_text":
            exception_text,
    },

    "policy": {
        "performance_evaluation":
            False,

        "benchmark_exposure":
            False,

        "repeat_probe":
            False,
    },
}

PROBE_RECORD.write_text(
    json.dumps(record, indent=2) + "\n",
    encoding="utf-8"
)

print("\n" + "-" * 110)
print("PROBE RESULT")
print("-" * 110)

print("Status          :", status)
print("Finish reason   :", finish_reason)
print("Response        :", repr(response_text))
print("Exception type  :", exception_type)

if exception_text:
    print("Exception:")
    print(exception_text)

print("\nProbe record:")
print(PROBE_RECORD)

print("SHA-256:")
print(sha256(PROBE_RECORD))

# ------------------------------------------------------------------
# Decision
# ------------------------------------------------------------------

print("\n" + "=" * 110)

if status == "INFERENCE_REQUEST_SUCCEEDED":

    print("✓ A6 ENDPOINT ACCEPTED THE NON-BENCHMARK REQUEST")
    print("✓ A6 IS ELIGIBLE FOR THE FROZEN BENCHMARK")
    print("✓ DO NOT RUN THE BENCHMARK YET")
    print("✓ SEND THIS OUTPUT BEFORE A6 EXECUTION")

else:

    lower_error = (exception_text or "").lower()

    if (
        "model_not_available" in lower_error
        or "non-serverless" in lower_error
        or "dedicated endpoint" in lower_error
    ):
        print("✗ A6 ENDPOINT IS NOT ACCESSIBLE SERVERLESS")
        print("✗ DO NOT RUN THE 120-CASE BENCHMARK")
        print("✓ PRESERVE A6 AS AN UNAVAILABLE PREREGISTERED ARM")

    else:
        print("✗ A6 AVAILABILITY PROBE FAILED")
        print("✗ FAILURE CAUSE REQUIRES DIAGNOSIS")
        print("✗ DO NOT RUN THE BENCHMARK")

print("=" * 110)

STEP A28 — A6 SINGLE NON-BENCHMARK AVAILABILITY PROBE

✓ Frozen A5 manifest verified
✓ API key present
✓ Probe record path clean

A6 MODEL
--------------------------------------------------------------------------------------------------------------
openai/gpt-oss-20b

NON-BENCHMARK PROBE
--------------------------------------------------------------------------------------------------------------
Reply with exactly: OK

This probe:
  ✓ contains no held-out benchmark case
  ✓ contains no network-slicing intent
  ✓ does not evaluate model accuracy
  ✓ is performed once only

--------------------------------------------------------------------------------------------------------------
PROBE RESULT
--------------------------------------------------------------------------------------------------------------
Status          : INFERENCE_REQUEST_FAILED
Finish reason   : None
Response        : None
Exception type  : BadRequestError
Exception:
BadRequestError("Error code: 400 - {'id': 'p3aoJ38

In [34]:
from pathlib import Path
from datetime import datetime, timezone
import hashlib
import json

ROOT = Path("/content/deterministic-intent-translation-journal")
EXP_A = ROOT / "journal_experiments/experiment_A_model_sweep"

PROBE = (
    EXP_A /
    "manifests/A6_prebenchmark_availability_probe_v1.json"
)

OUT = (
    EXP_A /
    "manifests/A6_unavailable_arm_manifest_v1.json"
)

EXPECTED_PROBE_SHA = (
    "b8019e0e0d32d41b2e1e2ab8430ccb8d2fbef3608d0e559ae597ad8cee8d4a6b"
)


def sha256(path):
    h = hashlib.sha256()

    with open(path, "rb") as f:
        for block in iter(lambda: f.read(1024 * 1024), b""):
            h.update(block)

    return h.hexdigest()


print("=" * 110)
print("STEP A29 — FREEZE A6 ENDPOINT-UNAVAILABLE ARM")
print("=" * 110)

# ------------------------------------------------------------------
# Integrity
# ------------------------------------------------------------------

assert PROBE.exists(), (
    "STOP: A6 availability-probe record missing."
)

assert sha256(PROBE) == EXPECTED_PROBE_SHA, (
    "STOP: A6 availability-probe record changed."
)

assert not OUT.exists(), (
    "STOP: A6 unavailable-arm manifest already exists."
)

print("\n✓ A6 availability-probe record verified")
print("✓ Probe SHA-256 verified")

# ------------------------------------------------------------------
# Read frozen probe
# ------------------------------------------------------------------

probe = json.loads(
    PROBE.read_text(encoding="utf-8")
)

assert probe["arm"] == "A6"
assert probe["model"] == "openai/gpt-oss-20b"

assert (
    probe["result"]["status"]
    == "INFERENCE_REQUEST_FAILED"
)

assert (
    probe["result"]["exception_type"]
    == "BadRequestError"
)

exception_text = (
    probe["result"]["exception_text"] or ""
)

lower_error = exception_text.lower()

assert "model_not_available" in lower_error
assert "non-serverless" in lower_error
assert "dedicated endpoint" in lower_error

assert (
    probe["probe"]["contains_benchmark_case"]
    is False
)

assert (
    probe["probe"]["contains_network_slicing_intent"]
    is False
)

assert (
    probe["probe"]["inference_request_count"]
    == 1
)

print("\n" + "-" * 110)
print("A6 AVAILABILITY EVIDENCE")
print("-" * 110)

print(
    "Probe status          :",
    probe["result"]["status"]
)

print(
    "Exception type        :",
    probe["result"]["exception_type"]
)

print(
    "model_not_available   :",
    "model_not_available" in lower_error
)

print(
    "non-serverless        :",
    "non-serverless" in lower_error
)

print(
    "dedicated endpoint    :",
    "dedicated endpoint" in lower_error
)

print(
    "Benchmark case exposed:",
    probe["probe"]["contains_benchmark_case"]
)

print(
    "Inference probes      :",
    probe["probe"]["inference_request_count"]
)

classification = (
    "UNAVAILABLE_ON_ACCESSIBLE_SERVERLESS_ENDPOINT"
)

print("\n" + "-" * 110)
print("A6 CLASSIFICATION")
print("-" * 110)

print("Status :", classification)

print(
    "\nA6 is not an evaluable model-performance arm."
)

print(
    "The pre-benchmark availability probe was rejected "
    "before a model response was obtained."
)

print(
    "Provider classification: BadRequestError / "
    "model_not_available."
)

print(
    "Provider explanation: model is non-serverless and "
    "requires a dedicated endpoint."
)

print(
    "\nThe frozen 120-case held-out benchmark was NOT "
    "submitted to A6."
)

# ------------------------------------------------------------------
# Freeze A6
# ------------------------------------------------------------------

record = {
    "manifest_id":
        "experiment-A-A6-unavailable-arm-v1",

    "created_utc":
        datetime.now(timezone.utc).isoformat(),

    "arm":
        "A6",

    "model":
        "openai/gpt-oss-20b",

    "preregistered_role":
        "Independent open-weight model family",

    "status":
        classification,

    "availability_check": {
        "method":
            "single_non_benchmark_inference_probe",

        "probe_text":
            probe["probe"]["text"],

        "contains_benchmark_case":
            False,

        "contains_network_slicing_intent":
            False,

        "inference_request_count":
            1,

        "status":
            probe["result"]["status"],
    },

    "reason": {
        "provider_error_type":
            "BadRequestError",

        "provider_error_code":
            "model_not_available",

        "provider_message_summary": (
            "The model is non-serverless and requires "
            "a dedicated endpoint."
        ),
    },

    "benchmark_execution": {
        "executed":
            False,

        "cases_submitted":
            0,

        "reason":
            (
                "Pre-benchmark availability probe established "
                "that the preregistered model was unavailable "
                "through the accessible serverless endpoint."
            ),
    },

    "performance_metrics": {
        "accuracy":
            None,

        "VAR":
            None,

        "IRR":
            None,

        "containment_violations":
            None,

        "reason":
            "No benchmark execution occurred."
    },

    "reporting_policy": {
        "include_arm_in_experiment_accounting":
            True,

        "include_as_successful_model_performance_arm":
            False,

        "use_accuracy_VAR_IRR_for_model_comparison":
            False,

        "report_as_preregistered_unavailable_arm":
            True,

        "replacement_model_allowed":
            False,

        "benchmark_rerun_allowed":
            False,
    },

    "artifacts": {
        "availability_probe": {
            "path":
                str(PROBE.relative_to(ROOT)),

            "sha256":
                sha256(PROBE),
        }
    },

    "experimental_decision": (
        "Preserve A6 as a preregistered unavailable arm. "
        "Do not execute the held-out benchmark, rerun the "
        "availability probe, or substitute another model."
    ),
}

OUT.write_text(
    json.dumps(record, indent=2) + "\n",
    encoding="utf-8"
)

print("\n" + "-" * 110)
print("FROZEN A6 RECORD")
print("-" * 110)

print(OUT)

print("SHA-256:")
print(sha256(OUT))

print("\n" + "=" * 110)
print("✓ A6 CLASSIFIED AS ENDPOINT-UNAVAILABLE")
print("✓ PRE-BENCHMARK AVAILABILITY FAILURE PRESERVED")
print("✓ HELD-OUT BENCHMARK CASES SUBMITTED TO A6 = 0")
print("✓ A6 NOT USED AS MODEL-PERFORMANCE EVIDENCE")
print("✓ NO RERUN OR MODEL SUBSTITUTION")
print("✓ NO MODEL/API CALL MADE")
print("✓ SIX-ARM EXECUTION/AVAILABILITY PHASE COMPLETE")
print("=" * 110)

STEP A29 — FREEZE A6 ENDPOINT-UNAVAILABLE ARM

✓ A6 availability-probe record verified
✓ Probe SHA-256 verified

--------------------------------------------------------------------------------------------------------------
A6 AVAILABILITY EVIDENCE
--------------------------------------------------------------------------------------------------------------
Probe status          : INFERENCE_REQUEST_FAILED
Exception type        : BadRequestError
model_not_available   : True
non-serverless        : True
dedicated endpoint    : True
Benchmark case exposed: False
Inference probes      : 1

--------------------------------------------------------------------------------------------------------------
A6 CLASSIFICATION
--------------------------------------------------------------------------------------------------------------
Status : UNAVAILABLE_ON_ACCESSIBLE_SERVERLESS_ENDPOINT

A6 is not an evaluable model-performance arm.
The pre-benchmark availability probe was rejected before a model 

In [35]:
from pathlib import Path
from datetime import datetime, timezone
import hashlib
import json

ROOT = Path("/content/deterministic-intent-translation-journal")
EXP_A = ROOT / "journal_experiments/experiment_A_model_sweep"

MANIFEST_DIR = EXP_A / "manifests"
METRICS_DIR = EXP_A / "results/metrics"
ANALYSIS_DIR = EXP_A / "results/analysis"
RAW_DIR = EXP_A / "results/raw"

OUT = MANIFEST_DIR / "experiment_A_final_accounting_v1.json"


def sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(1024 * 1024), b""):
            h.update(block)
    return h.hexdigest()


print("=" * 110)
print("STEP A30 — FREEZE COMPLETE EXPERIMENT A ACCOUNTING")
print("=" * 110)

# ------------------------------------------------------------------
# Frozen infrastructure
# ------------------------------------------------------------------

RUNNER = ROOT / (
    "experiment-2b/implementation/e3_v3/"
    "run_experiment2b_e3v3_r7_dev_v1.py"
)

DATASET = ROOT / (
    "experiment-2b/reproducibility/"
    "e3v3_heldout_benchmark_v1.0/"
    "e3v3_heldout_benchmark_120_v1.csv"
)

EXPECTED_RUNNER_SHA = (
    "40c035cb864978c718a72fcfdfbb164d262529960b0ae78242d5194147cd8257"
)

EXPECTED_DATASET_SHA = (
    "84576e7d99a2721e61ded5a14bdafeb96707b068397243bbed32aa735496d4de"
)

assert RUNNER.exists()
assert DATASET.exists()

assert sha256(RUNNER) == EXPECTED_RUNNER_SHA
assert sha256(DATASET) == EXPECTED_DATASET_SHA

print("\n✓ Frozen runner verified")
print("✓ Frozen benchmark verified")

# ------------------------------------------------------------------
# Required frozen manifests
# ------------------------------------------------------------------

A3_MANIFEST = MANIFEST_DIR / "A3_unavailable_arm_manifest_v1.json"
A4_MANIFEST = MANIFEST_DIR / "A4_unavailable_arm_manifest_v1.json"
A5_MANIFEST = MANIFEST_DIR / "A5_unavailable_arm_manifest_v1.json"
A6_PROBE = MANIFEST_DIR / "A6_prebenchmark_availability_probe_v1.json"
A6_MANIFEST = MANIFEST_DIR / "A6_unavailable_arm_manifest_v1.json"

EXPECTED_MANIFESTS = {
    A3_MANIFEST:
        "22920e779994b802d66493d6722f89b2405f858f1967ef75ac90e282129534e8",

    A4_MANIFEST:
        "2cacef85717ab3fe5ed4f9bc55926fd71eb1dd7c36183784d2b1fb0b84606fb6",

    A5_MANIFEST:
        "89e7f944318bf389ecfbf20b29e9e663c433c8118611c55d66045b86b6c19222",

    A6_PROBE:
        "b8019e0e0d32d41b2e1e2ab8430ccb8d2fbef3608d0e559ae597ad8cee8d4a6b",

    A6_MANIFEST:
        "6d7476b3621e4347421e04e8293158bb590f0b172bc92d071b2c12239b84946f",
}

for path, expected_sha in EXPECTED_MANIFESTS.items():

    assert path.exists(), f"STOP: missing artifact: {path}"

    actual_sha = sha256(path)

    assert actual_sha == expected_sha, (
        f"STOP: artifact changed:\n{path}\n"
        f"Expected: {expected_sha}\n"
        f"Actual:   {actual_sha}"
    )

print("✓ A3 unavailable-arm manifest verified")
print("✓ A4 unavailable-arm manifest verified")
print("✓ A5 unavailable-arm manifest verified")
print("✓ A6 availability probe verified")
print("✓ A6 unavailable-arm manifest verified")

# ------------------------------------------------------------------
# A1 / A2 raw results
# ------------------------------------------------------------------

A1_RAW = RAW_DIR / "A1_Qwen3.5-9B_results.csv"
A2_RAW = RAW_DIR / "A2_Llama-3.3-70B-Instruct-Turbo_results.csv"

# Locate by known hashes if filenames differ slightly.
EXPECTED_A1_RAW_SHA = (
    "c22618b2750de7f5ef1824fa4036a901ef54f611ad162fcd797b20d79e2b4e06"
)

EXPECTED_A2_RAW_SHA = (
    "5eba3be64426c14ac1a917be5392e4b6a0a487f3bf9be81c076d7f4b429380d7"
)


def locate_by_hash(directory, expected_sha, suffix=None):
    matches = []

    for path in directory.rglob("*"):

        if not path.is_file():
            continue

        if suffix and path.suffix.lower() != suffix.lower():
            continue

        try:
            if sha256(path) == expected_sha:
                matches.append(path)
        except Exception:
            pass

    assert len(matches) == 1, (
        f"STOP: expected exactly one artifact with SHA "
        f"{expected_sha}, found {len(matches)}."
    )

    return matches[0]


if not A1_RAW.exists() or sha256(A1_RAW) != EXPECTED_A1_RAW_SHA:
    A1_RAW = locate_by_hash(
        RAW_DIR,
        EXPECTED_A1_RAW_SHA,
        ".csv"
    )

if not A2_RAW.exists() or sha256(A2_RAW) != EXPECTED_A2_RAW_SHA:
    A2_RAW = locate_by_hash(
        RAW_DIR,
        EXPECTED_A2_RAW_SHA,
        ".csv"
    )

assert sha256(A1_RAW) == EXPECTED_A1_RAW_SHA
assert sha256(A2_RAW) == EXPECTED_A2_RAW_SHA

print("✓ A1 evaluable result verified")
print("✓ A2 evaluable result verified")

# ------------------------------------------------------------------
# A3/A4/A5 raw execution evidence
# ------------------------------------------------------------------

A3_RAW_SHA = (
    "fa4c9a0e57103768d06d370983a70a76516800bd0d57ec3d965445d35daf440a"
)

A4_RAW_SHA = (
    "6babc2b2492473e5ee737e182dd338be2bc23c0cb0f72b5de8b70864d8757125"
)

A5_RAW_SHA = (
    "8d6c561218991dc1d3a0f47601bd5a8c01bce918be0b3c1133333de5eb702f41"
)

A3_RAW = locate_by_hash(RAW_DIR, A3_RAW_SHA, ".csv")
A4_RAW = locate_by_hash(RAW_DIR, A4_RAW_SHA, ".csv")
A5_RAW = locate_by_hash(RAW_DIR, A5_RAW_SHA, ".csv")

print("✓ A3 raw attempted execution verified")
print("✓ A4 raw attempted execution verified")
print("✓ A5 raw attempted execution verified")

# ------------------------------------------------------------------
# Prevent accidental duplicate/final overwrite
# ------------------------------------------------------------------

assert not OUT.exists(), (
    "STOP: final Experiment A accounting already exists.\n"
    "Do not overwrite it."
)

# ------------------------------------------------------------------
# Final accounting
# ------------------------------------------------------------------

record = {
    "manifest_id":
        "experiment-A-final-accounting-v1",

    "created_utc":
        datetime.now(timezone.utc).isoformat(),

    "experiment":
        "Experiment A — preregistered model sweep",

    "research_question": (
        "With the deterministic E3-v3 cascade and held-out "
        "benchmark frozen, how sensitive are decision performance "
        "and deterministic failure containment to the semantic "
        "parser model?"
    ),

    "frozen_components": {
        "runner": {
            "path": str(RUNNER.relative_to(ROOT)),
            "sha256": sha256(RUNNER),
        },

        "benchmark": {
            "path": str(DATASET.relative_to(ROOT)),
            "sha256": sha256(DATASET),
            "cases": 120,
            "valid_cases": 58,
            "invalid_cases": 62,
        },

        "changed_variable":
            "semantic parser model only",

        "unchanged": [
            "benchmark",
            "runner",
            "prompt",
            "deterministic gates",
            "thresholds",
            "labels",
            "containment definition",
        ],
    },

    "arm_accounting": [

        {
            "arm": "A1",
            "model": "Qwen/Qwen3.5-9B",
            "role": (
                "Frozen E3-v3 baseline model and mandatory "
                "reproduction gate"
            ),
            "status": "EVALUABLE",
            "benchmark_executed": True,
            "benchmark_cases": 120,
            "llm_invoked": 77,
            "parse_success": 77,

            "performance": {
                "accuracy": {
                    "numerator": 101,
                    "denominator": 120,
                    "value": 0.841667,
                    "cp95": [0.763836, 0.901898],
                },

                "VAR": {
                    "numerator": 51,
                    "denominator": 58,
                    "value": 0.879310,
                    "cp95": [0.767016, 0.950073],
                },

                "IRR": {
                    "numerator": 50,
                    "denominator": 62,
                    "value": 0.806452,
                    "cp95": [0.686314, 0.895790],
                },
            },

            "containment": {
                "accepted": 63,
                "descriptors": 63,
                "rejected_with_descriptor": 0,
                "accepted_without_descriptor": 0,
                "containment_violations": 0,
            },

            "raw_result": {
                "path": str(A1_RAW.relative_to(ROOT)),
                "sha256": sha256(A1_RAW),
            },
        },

        {
            "arm": "A2",
            "model":
                "meta-llama/Llama-3.3-70B-Instruct-Turbo",

            "role":
                "Large general-purpose model from a different model family",

            "status": "EVALUABLE",
            "benchmark_executed": True,
            "benchmark_cases": 120,
            "llm_invoked": 77,
            "parse_success": 77,

            "performance": {
                "accuracy": {
                    "numerator": 100,
                    "denominator": 120,
                    "value": 0.833333,
                    "cp95": [0.7544, 0.8951],
                },

                "VAR": {
                    "numerator": 48,
                    "denominator": 58,
                    "value": 0.827586,
                    "cp95": [0.7057, 0.9141],
                },

                "IRR": {
                    "numerator": 52,
                    "denominator": 62,
                    "value": 0.838710,
                    "cp95": [0.7233, 0.9198],
                },
            },

            "containment": {
                "accepted": 58,
                "descriptors": 58,
                "rejected_with_descriptor": 0,
                "accepted_without_descriptor": 0,
                "containment_violations": 0,
            },

            "comparison_to_A1": {
                "accuracy_delta": -0.008333,
                "VAR_delta": -0.051724,
                "IRR_delta": 0.032258,
                "decision_or_correctness_changes": 9,
                "improvements": 4,
                "regressions": 5,
                "llm_path_changes": 0,
            },

            "raw_result": {
                "path": str(A2_RAW.relative_to(ROOT)),
                "sha256": sha256(A2_RAW),
            },
        },

        {
            "arm": "A3",
            "model": "Qwen/Qwen2.5-7B-Instruct-Turbo",
            "role": "Smaller model arm for model-scale sensitivity",
            "status":
                "UNAVAILABLE_ON_ACCESSIBLE_SERVERLESS_ENDPOINT",

            "benchmark_execution": {
                "attempted": True,
                "cases_written": 120,
                "llm_invoked": 77,
                "successful_model_responses": 0,
                "provider_failures": 77,
            },

            "provider_failure": {
                "error_type": "BadRequestError",
                "error_code": "model_not_available",
                "requires_dedicated_endpoint": True,
            },

            "performance_interpretation_allowed": False,

            "execution_level_containment_violations": 0,

            "raw_result": {
                "path": str(A3_RAW.relative_to(ROOT)),
                "sha256": sha256(A3_RAW),
            },

            "manifest": {
                "path": str(A3_MANIFEST.relative_to(ROOT)),
                "sha256": sha256(A3_MANIFEST),
            },
        },

        {
            "arm": "A4",
            "model":
                "mistralai/Mistral-Small-24B-Instruct-2501",

            "role":
                "Alternative Mistral model family",

            "status":
                "UNAVAILABLE_ON_ACCESSIBLE_SERVERLESS_ENDPOINT",

            "benchmark_execution": {
                "attempted": True,
                "cases_written": 120,
                "llm_invoked": 77,
                "successful_model_responses": 0,
                "provider_failures": 77,
            },

            "provider_failure": {
                "error_type": "BadRequestError",
                "error_code": "model_not_available",
                "requires_dedicated_endpoint": True,
            },

            "performance_interpretation_allowed": False,

            "execution_level_containment_violations": 0,

            "raw_result": {
                "path": str(A4_RAW.relative_to(ROOT)),
                "sha256": sha256(A4_RAW),
            },

            "manifest": {
                "path": str(A4_MANIFEST.relative_to(ROOT)),
                "sha256": sha256(A4_MANIFEST),
            },
        },

        {
            "arm": "A5",
            "model": "deepseek-ai/DeepSeek-V3.1",
            "role": "Alternative DeepSeek model family",

            "status":
                "UNAVAILABLE_ON_ACCESSIBLE_SERVERLESS_ENDPOINT",

            "benchmark_execution": {
                "attempted": True,
                "cases_written": 120,
                "llm_invoked": 77,
                "successful_model_responses": 0,
                "provider_failures": 77,
            },

            "provider_failure": {
                "error_type": "BadRequestError",
                "error_code": "model_not_available",
                "requires_dedicated_endpoint": True,
            },

            "performance_interpretation_allowed": False,

            "execution_level_containment_violations": 0,

            "raw_result": {
                "path": str(A5_RAW.relative_to(ROOT)),
                "sha256": sha256(A5_RAW),
            },

            "manifest": {
                "path": str(A5_MANIFEST.relative_to(ROOT)),
                "sha256": sha256(A5_MANIFEST),
            },
        },

        {
            "arm": "A6",
            "model": "openai/gpt-oss-20b",
            "role": "Independent open-weight model family",

            "status":
                "UNAVAILABLE_ON_ACCESSIBLE_SERVERLESS_ENDPOINT",

            "availability_check": {
                "performed_before_benchmark": True,
                "non_benchmark_probe_requests": 1,
                "benchmark_cases_exposed": 0,
                "result": "model_not_available",
                "requires_dedicated_endpoint": True,
            },

            "benchmark_execution": {
                "attempted": False,
                "cases_submitted": 0,
            },

            "performance_interpretation_allowed": False,

            "containment_measurement_available": False,

            "probe": {
                "path": str(A6_PROBE.relative_to(ROOT)),
                "sha256": sha256(A6_PROBE),
            },

            "manifest": {
                "path": str(A6_MANIFEST.relative_to(ROOT)),
                "sha256": sha256(A6_MANIFEST),
            },
        },
    ],

    "experiment_accounting": {
        "preregistered_arms": 6,
        "evaluable_arms": 2,
        "endpoint_unavailable_arms": 4,

        "evaluable_arm_ids": [
            "A1",
            "A2",
        ],

        "endpoint_unavailable_arm_ids": [
            "A3",
            "A4",
            "A5",
            "A6",
        ],

        "post_hoc_model_substitutions": 0,
        "result_driven_arm_removals": 0,
        "benchmark_reruns_after_observing_results": 0,
    },

    "containment_interpretation": {
        "operational_definition":
            "accepted == False AND descriptor_json present",

        "A1_evaluable_violations": 0,
        "A2_evaluable_violations": 0,

        "A3_provider_failure_execution_violations": 0,
        "A4_provider_failure_execution_violations": 0,
        "A5_provider_failure_execution_violations": 0,

        "A6":
            "Not measured because the held-out benchmark was not executed.",

        "interpretive_boundary": (
            "A3-A5 zero violations are execution-level fail-closed "
            "observations during provider endpoint failure and are not "
            "evidence of those models' semantic behavior."
        ),
    },

    "claim_boundary": {
        "supported": [
            (
                "The frozen deterministic cascade was evaluated "
                "successfully with two accessible semantic-parser "
                "models from different model families."
            ),
            (
                "A1 and A2 both produced zero containment violations "
                "under the frozen operational definition."
            ),
            (
                "Decision-level performance changed between A1 and A2 "
                "while the deterministic cascade and LLM invocation "
                "path remained unchanged."
            ),
            (
                "All six preregistered arms are transparently "
                "accounted for."
            ),
        ],

        "not_supported": [
            "Six-model performance comparison",
            "Universal model independence",
            "Model-independent accuracy",
            "Performance claims for A3, A4, A5, or A6",
            (
                "Semantic robustness claims based on provider "
                "endpoint failures"
            ),
        ],

        "recommended_wording": (
            "A six-arm model sweep was preregistered. Two arms were "
            "evaluable through the accessible serverless inference "
            "service, while four preregistered arms were unavailable "
            "because the provider required dedicated endpoints. "
            "Across the two evaluable model families, predictive "
            "performance varied, whereas the frozen deterministic "
            "failure-containment mechanism produced zero observed "
            "containment violations."
        ),
    },
}

OUT.write_text(
    json.dumps(record, indent=2) + "\n",
    encoding="utf-8"
)

print("\n" + "-" * 110)
print("FINAL EXPERIMENT A ACCOUNTING")
print("-" * 110)

print("Preregistered arms          : 6")
print("Evaluable arms              : 2")
print("Endpoint-unavailable arms   : 4")
print("Post-hoc substitutions      : 0")
print("Result-driven removals      : 0")

print("\nEvaluable:")
print("  A1  Qwen/Qwen3.5-9B")
print("  A2  meta-llama/Llama-3.3-70B-Instruct-Turbo")

print("\nUnavailable:")
print("  A3  Qwen/Qwen2.5-7B-Instruct-Turbo")
print("  A4  mistralai/Mistral-Small-24B-Instruct-2501")
print("  A5  deepseek-ai/DeepSeek-V3.1")
print("  A6  openai/gpt-oss-20b")

print("\nEVALUABLE PERFORMANCE")
print("-" * 110)

print(
    "A1  Accuracy 84.17% | VAR 87.93% | "
    "IRR 80.65% | containment violations 0"
)

print(
    "A2  Accuracy 83.33% | VAR 82.76% | "
    "IRR 83.87% | containment violations 0"
)

print("\nA2 − A1:")
print("  Accuracy : -0.83 percentage points")
print("  VAR      : -5.17 percentage points")
print("  IRR      : +3.23 percentage points")

print("\n" + "-" * 110)
print("CLAIM BOUNDARY")
print("-" * 110)

print(
    "Supported: cross-model evidence across TWO evaluable "
    "model families under the frozen cascade."
)

print(
    "Not supported: six-model performance comparison or "
    "universal model independence."
)

print("\nFinal accounting manifest:")
print(OUT)

print("SHA-256:")
print(sha256(OUT))

print("\n" + "=" * 110)
print("✓ EXPERIMENT A SIX-ARM ACCOUNTING FROZEN")
print("✓ ALL PREREGISTERED ARMS ACCOUNTED FOR")
print("✓ A1/A2 PRESERVED AS EVALUABLE ARMS")
print("✓ A3/A4/A5/A6 PRESERVED AS UNAVAILABLE ARMS")
print("✓ NO POST-HOC MODEL SUBSTITUTION")
print("✓ NO MODEL/API CALL MADE")
print("✓ NO EXISTING RESULT MODIFIED")
print("=" * 110)

STEP A30 — FREEZE COMPLETE EXPERIMENT A ACCOUNTING

✓ Frozen runner verified
✓ Frozen benchmark verified
✓ A3 unavailable-arm manifest verified
✓ A4 unavailable-arm manifest verified
✓ A5 unavailable-arm manifest verified
✓ A6 availability probe verified
✓ A6 unavailable-arm manifest verified
✓ A1 evaluable result verified
✓ A2 evaluable result verified
✓ A3 raw attempted execution verified
✓ A4 raw attempted execution verified
✓ A5 raw attempted execution verified

--------------------------------------------------------------------------------------------------------------
FINAL EXPERIMENT A ACCOUNTING
--------------------------------------------------------------------------------------------------------------
Preregistered arms          : 6
Evaluable arms              : 2
Endpoint-unavailable arms   : 4
Post-hoc substitutions      : 0
Result-driven removals      : 0

Evaluable:
  A1  Qwen/Qwen3.5-9B
  A2  meta-llama/Llama-3.3-70B-Instruct-Turbo

Unavailable:
  A3  Qwen/Qwen2.5-7B-I

In [36]:
from pathlib import Path
from datetime import datetime, timezone
import os
import json
import hashlib
import time
import pandas as pd

from together import Together


ROOT = Path("/content/deterministic-intent-translation-journal")

EXP_A = (
    ROOT /
    "journal_experiments" /
    "experiment_A_model_sweep"
)

SCREEN_DIR = EXP_A / "model_availability_screen"
SCREEN_DIR.mkdir(parents=True, exist_ok=True)

OUT_CSV = (
    SCREEN_DIR /
    "together_chat_model_serverless_screen_v1.csv"
)

OUT_JSON = (
    SCREEN_DIR /
    "together_chat_model_serverless_screen_v1.json"
)

# ------------------------------------------------------------------
# Safety: never overwrite a completed screen
# ------------------------------------------------------------------

assert not OUT_CSV.exists(), (
    f"STOP: screen already exists:\n{OUT_CSV}\n"
    "Do not repeat the availability screen."
)

assert not OUT_JSON.exists(), (
    f"STOP: screen manifest already exists:\n{OUT_JSON}"
)

assert os.environ.get("TOGETHER_API_KEY"), (
    "STOP: TOGETHER_API_KEY missing."
)


MODEL_A1 = "Qwen/Qwen3.5-9B"
MODEL_A2 = "meta-llama/Llama-3.3-70B-Instruct-Turbo"

# Already established unavailable.
KNOWN_UNAVAILABLE = {
    "Qwen/Qwen2.5-7B-Instruct-Turbo",
    "mistralai/Mistral-Small-24B-Instruct-2501",
    "deepseek-ai/DeepSeek-V3.1",
    "openai/gpt-oss-20b",
}

PROBE_TEXT = "Reply with exactly: OK"


def sha256(path):
    h = hashlib.sha256()

    with open(path, "rb") as f:
        for block in iter(lambda: f.read(1024 * 1024), b""):
            h.update(block)

    return h.hexdigest()


def model_to_dict(model):

    if hasattr(model, "model_dump"):
        return model.model_dump()

    if hasattr(model, "dict"):
        return model.dict()

    if isinstance(model, dict):
        return model

    try:
        return {
            k: v
            for k, v in vars(model).items()
            if not k.startswith("_")
        }

    except Exception:
        return {}


print("=" * 110)
print("STEP A30X — DISCOVER SERVERLESS-ACCESSIBLE CHAT MODELS")
print("=" * 110)

print("\nPurpose:")
print(
    "Identify candidate chat models that accept a single "
    "non-benchmark inference request."
)

print("\nIMPORTANT:")
print("  ✓ No held-out benchmark case will be used")
print("  ✓ No network-slicing intent will be used")
print("  ✓ No model performance will be measured")
print("  ✓ Each candidate will be probed at most once")
print("  ✓ A1/A2 will not be re-probed")
print("  ✓ Known unavailable A3-A6 models will not be re-probed")


# ------------------------------------------------------------------
# Obtain current Together model catalogue
# ------------------------------------------------------------------

client = Together(
    api_key=os.environ["TOGETHER_API_KEY"]
)

catalogue_response = client.models.list()

if hasattr(catalogue_response, "data"):
    catalogue = catalogue_response.data
else:
    catalogue = catalogue_response


catalogue_dicts = [
    model_to_dict(m)
    for m in catalogue
]

print("\nCatalogue models returned:", len(catalogue_dicts))


# ------------------------------------------------------------------
# Select chat models only
# ------------------------------------------------------------------

chat_models = []

for d in catalogue_dicts:

    model_id = (
        d.get("id")
        or d.get("model")
        or d.get("name")
    )

    model_type = str(
        d.get("type", "")
    ).strip().lower()

    if not model_id:
        continue

    if model_type != "chat":
        continue

    chat_models.append({
        "model": model_id,
        "display_name": d.get("display_name"),
        "organization": d.get("organization"),
        "context_length": d.get("context_length"),
        "catalogue_running": d.get("running"),
        "pricing": d.get("pricing"),
    })


# deterministic order
chat_models = sorted(
    chat_models,
    key=lambda x: x["model"].lower()
)


print("Chat models found       :", len(chat_models))


# ------------------------------------------------------------------
# Remove already-resolved models
# ------------------------------------------------------------------

candidates = []

for item in chat_models:

    model_id = item["model"]

    if model_id in {MODEL_A1, MODEL_A2}:
        continue

    if model_id in KNOWN_UNAVAILABLE:
        continue

    candidates.append(item)


print("A1/A2 excluded          : 2")
print("Known unavailable skipped:", len(KNOWN_UNAVAILABLE))
print("New models to probe     :", len(candidates))


# ------------------------------------------------------------------
# ONE synthetic inference request per candidate
# ------------------------------------------------------------------

results = []

print("\n" + "-" * 110)
print("BEGINNING NON-BENCHMARK AVAILABILITY SCREEN")
print("-" * 110)

for idx, item in enumerate(candidates, start=1):

    model_id = item["model"]

    print(
        f"[{idx:03d}/{len(candidates):03d}] "
        f"{model_id}"
    )

    status = None
    response_text = None
    finish_reason = None
    exception_type = None
    exception_text = None

    try:

        response = client.chat.completions.create(
            model=model_id,
            messages=[
                {
                    "role": "user",
                    "content": PROBE_TEXT,
                }
            ],
            temperature=0,
            max_tokens=16,
        )

        status = "ACCESSIBLE"

        if getattr(response, "choices", None):

            choice = response.choices[0]

            finish_reason = getattr(
                choice,
                "finish_reason",
                None
            )

            message = getattr(
                choice,
                "message",
                None
            )

            if message is not None:
                response_text = getattr(
                    message,
                    "content",
                    None
                )

    except Exception as exc:

        exception_type = type(exc).__name__
        exception_text = repr(exc)

        lower = exception_text.lower()

        if (
            "model_not_available" in lower
            or "non-serverless" in lower
            or "dedicated endpoint" in lower
        ):
            status = "ENDPOINT_UNAVAILABLE"

        elif (
            "rate" in lower
            and "limit" in lower
        ):
            status = "RATE_LIMITED"

        elif (
            "authentication" in lower
            or "unauthorized" in lower
            or "invalid api key" in lower
        ):
            status = "AUTH_ERROR"

        elif "timeout" in lower:
            status = "TIMEOUT"

        else:
            status = "OTHER_ERROR"

    results.append({
        "model":
            model_id,

        "display_name":
            item["display_name"],

        "organization":
            item["organization"],

        "context_length":
            item["context_length"],

        "catalogue_running":
            item["catalogue_running"],

        "status":
            status,

        "finish_reason":
            finish_reason,

        "response_text":
            response_text,

        "exception_type":
            exception_type,

        "exception_text":
            exception_text,
    })

    # Gentle spacing between requests.
    time.sleep(0.25)


# ------------------------------------------------------------------
# Save complete evidence
# ------------------------------------------------------------------

df = pd.DataFrame(results)

df.to_csv(
    OUT_CSV,
    index=False
)

summary = (
    df["status"]
    .value_counts(dropna=False)
    .to_dict()
)


record = {
    "manifest_id":
        "experiment-A-model-availability-screen-v1",

    "created_utc":
        datetime.now(timezone.utc).isoformat(),

    "purpose": (
        "Infrastructure-only discovery of chat models that "
        "accept serverless inference before additional "
        "cross-model evaluation."
    ),

    "probe": {
        "text":
            PROBE_TEXT,

        "benchmark_case":
            False,

        "network_slicing_content":
            False,

        "requests_per_candidate":
            1,

        "performance_evaluation":
            False,
    },

    "catalogue": {
        "total_models":
            len(catalogue_dicts),

        "chat_models":
            len(chat_models),

        "new_candidates_screened":
            len(candidates),
    },

    "excluded_previously_resolved_models": {
        "successful": [
            MODEL_A1,
            MODEL_A2,
        ],

        "known_endpoint_unavailable":
            sorted(KNOWN_UNAVAILABLE),
    },

    "status_summary":
        summary,

    "output_csv": {
        "path":
            str(OUT_CSV.relative_to(ROOT)),
    },
}


OUT_JSON.write_text(
    json.dumps(
        record,
        indent=2,
        default=str
    ) + "\n",
    encoding="utf-8"
)


print("\n" + "=" * 110)
print("AVAILABILITY SCREEN COMPLETE")
print("=" * 110)

print("\nSTATUS COUNTS")
print("-" * 110)

print(
    df["status"]
    .value_counts(dropna=False)
    .to_string()
)


# ------------------------------------------------------------------
# Accessible models
# ------------------------------------------------------------------

accessible = (
    df[df["status"] == "ACCESSIBLE"]
    .copy()
    .sort_values(
        ["organization", "model"],
        na_position="last"
    )
)

print("\n" + "=" * 110)
print("SERVERLESS-ACCESSIBLE CANDIDATES")
print("=" * 110)

print("Accessible models:", len(accessible))

if len(accessible):

    cols = [
        "model",
        "display_name",
        "organization",
        "context_length",
        "catalogue_running",
    ]

    print(
        accessible[cols]
        .to_string(index=False)
    )

else:

    print("No accessible candidate models found.")


# ------------------------------------------------------------------
# Endpoint-unavailable models
# ------------------------------------------------------------------

unavailable = df[
    df["status"] == "ENDPOINT_UNAVAILABLE"
]

print("\nEndpoint-unavailable models:",
      len(unavailable))


# ------------------------------------------------------------------
# Unexpected failures
# ------------------------------------------------------------------

unexpected = df[
    ~df["status"].isin(
        [
            "ACCESSIBLE",
            "ENDPOINT_UNAVAILABLE",
        ]
    )
]

print("Other/error statuses:", len(unexpected))

if len(unexpected):

    print("\nOTHER FAILURES")
    print("-" * 110)

    print(
        unexpected[
            [
                "model",
                "status",
                "exception_type",
            ]
        ].to_string(index=False)
    )


print("\n" + "-" * 110)
print("ARTIFACTS")
print("-" * 110)

print("CSV:")
print(OUT_CSV)

print("SHA-256:")
print(sha256(OUT_CSV))

print("\nManifest:")
print(OUT_JSON)

print("SHA-256:")
print(sha256(OUT_JSON))


print("\n" + "=" * 110)
print("✓ CHAT MODELS SCREENED WITH NON-BENCHMARK INPUT ONLY")
print("✓ EACH NEW CANDIDATE PROBED AT MOST ONCE")
print("✓ A1/A2 NOT RE-PROBED")
print("✓ A3-A6 NOT RE-PROBED")
print("✓ NO HELD-OUT CASE EXPOSED")
print("✓ NO PERFORMANCE COMPARISON PERFORMED")
print("✓ DO NOT RUN ANY NEW BENCHMARK YET")
print("=" * 110)

STEP A30X — DISCOVER SERVERLESS-ACCESSIBLE CHAT MODELS

Purpose:
Identify candidate chat models that accept a single non-benchmark inference request.

IMPORTANT:
  ✓ No held-out benchmark case will be used
  ✓ No network-slicing intent will be used
  ✓ No model performance will be measured
  ✓ Each candidate will be probed at most once
  ✓ A1/A2 will not be re-probed
  ✓ Known unavailable A3-A6 models will not be re-probed

Catalogue models returned: 264
Chat models found       : 172
A1/A2 excluded          : 2
Known unavailable skipped: 4
New models to probe     : 166

--------------------------------------------------------------------------------------------------------------
BEGINNING NON-BENCHMARK AVAILABILITY SCREEN
--------------------------------------------------------------------------------------------------------------
[001/166] agentica-org/DeepCoder-14B-Preview
[002/166] allenai/Molmo-7B-D-0924
[003/166] arcee-ai/trinity-mini
[004/166] arize-ai/qwen-2-1.5b-instruct
[005/1

In [37]:
from pathlib import Path
from datetime import datetime, timezone
import hashlib
import json

ROOT = Path("/content/deterministic-intent-translation-journal")

EXP_A = (
    ROOT /
    "journal_experiments" /
    "experiment_A_model_sweep"
)

MANIFEST_DIR = EXP_A / "manifests"
PREREG_DIR = EXP_A / "preregistration"
SCREEN_DIR = EXP_A / "model_availability_screen"

MANIFEST_DIR.mkdir(parents=True, exist_ok=True)
PREREG_DIR.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------------
# Existing frozen artifacts
# ------------------------------------------------------------------

FINAL_A1_A6 = (
    MANIFEST_DIR /
    "experiment_A_final_accounting_v1.json"
)

SCREEN_CSV = (
    SCREEN_DIR /
    "together_chat_model_serverless_screen_v1.csv"
)

SCREEN_JSON = (
    SCREEN_DIR /
    "together_chat_model_serverless_screen_v1.json"
)

# ------------------------------------------------------------------
# New preregistration artifacts
# ------------------------------------------------------------------

OUT_JSON = (
    PREREG_DIR /
    "experiment_A_extension_A7_A10_preregistration_v1.json"
)

OUT_MD = (
    PREREG_DIR /
    "experiment_A_extension_A7_A10_preregistration_v1.md"
)

OUT_MANIFEST = (
    MANIFEST_DIR /
    "experiment_A_extension_A7_A10_preregistration_manifest_v1.json"
)

# ------------------------------------------------------------------
# Frozen hashes from completed work
# ------------------------------------------------------------------

EXPECTED_FINAL_A1_A6_SHA = (
    "7fe65b2b1600fdea83afde4c9e1a03f3c95257d024c1bc077e89f1edf6b63269"
)

EXPECTED_SCREEN_CSV_SHA = (
    "8be4be47e09166b5bb0d992a14eab0dd5f2fa1f1e0db00f714363e0cd3403703"
)

EXPECTED_SCREEN_JSON_SHA = (
    "02d5bf4ceefd9c15e0e11a9f217b58e007ecb6fb4a4c4c5ea5521f3ea122dc7a"
)

EXPECTED_RUNNER_SHA = (
    "40c035cb864978c718a72fcfdfbb164d262529960b0ae78242d5194147cd8257"
)

EXPECTED_DATASET_SHA = (
    "84576e7d99a2721e61ded5a14bdafeb96707b068397243bbed32aa735496d4de"
)


def sha256(path):
    h = hashlib.sha256()

    with open(path, "rb") as f:
        for block in iter(lambda: f.read(1024 * 1024), b""):
            h.update(block)

    return h.hexdigest()


print("=" * 110)
print("STEP A31 — PREREGISTER EXPERIMENT A EXTENSION: A7–A10")
print("=" * 110)

# ------------------------------------------------------------------
# Verify original A1-A6 experiment is frozen
# ------------------------------------------------------------------

assert FINAL_A1_A6.exists(), (
    "STOP: final A1-A6 accounting manifest missing."
)

assert sha256(FINAL_A1_A6) == EXPECTED_FINAL_A1_A6_SHA, (
    "STOP: frozen A1-A6 accounting manifest changed."
)

print("\n✓ Original A1-A6 accounting verified and frozen")

# ------------------------------------------------------------------
# Verify availability-screen evidence
# ------------------------------------------------------------------

assert SCREEN_CSV.exists(), (
    "STOP: availability-screen CSV missing."
)

assert SCREEN_JSON.exists(), (
    "STOP: availability-screen manifest missing."
)

assert sha256(SCREEN_CSV) == EXPECTED_SCREEN_CSV_SHA, (
    "STOP: availability-screen CSV changed."
)

assert sha256(SCREEN_JSON) == EXPECTED_SCREEN_JSON_SHA, (
    "STOP: availability-screen manifest changed."
)

print("✓ Availability-screen CSV verified")
print("✓ Availability-screen manifest verified")

# ------------------------------------------------------------------
# Verify runner and benchmark remain frozen
# ------------------------------------------------------------------

RUNNER = ROOT / (
    "experiment-2b/implementation/e3_v3/"
    "run_experiment2b_e3v3_r7_dev_v1.py"
)

DATASET = ROOT / (
    "experiment-2b/reproducibility/"
    "e3v3_heldout_benchmark_v1.0/"
    "e3v3_heldout_benchmark_120_v1.csv"
)

assert RUNNER.exists()
assert DATASET.exists()

assert sha256(RUNNER) == EXPECTED_RUNNER_SHA, (
    "STOP: frozen runner changed."
)

assert sha256(DATASET) == EXPECTED_DATASET_SHA, (
    "STOP: frozen benchmark changed."
)

print("✓ Frozen E3-v3 runner verified")
print("✓ Frozen 120-case benchmark verified")

# ------------------------------------------------------------------
# Prevent accidental overwrite
# ------------------------------------------------------------------

for path in [OUT_JSON, OUT_MD, OUT_MANIFEST]:
    assert not path.exists(), (
        f"STOP: preregistration artifact already exists:\n{path}\n"
        "Do not overwrite or recreate it."
    )

# ------------------------------------------------------------------
# Exact extension arms
# ------------------------------------------------------------------

arms = [
    {
        "arm": "A7",
        "model": "arize-ai/qwen-2-1.5b-instruct",
        "role": (
            "Small Qwen-derived instruction model for "
            "model-scale sensitivity"
        ),
        "selection_basis": (
            "Passed the frozen non-benchmark serverless "
            "availability screen and provides a substantially "
            "smaller semantic-parser arm."
        ),
    },

    {
        "arm": "A8",
        "model": "Prism-ML/Ternary-Bonsai-27B",
        "role": (
            "Medium-scale independent model family"
        ),
        "selection_basis": (
            "Passed the frozen non-benchmark serverless "
            "availability screen and adds model-family "
            "diversity at an intermediate scale."
        ),
    },

    {
        "arm": "A9",
        "model": "deepseek-ai/DeepSeek-V4.1-Flash",
        "role": (
            "DeepSeek-family semantic-parser model"
        ),
        "selection_basis": (
            "Passed the frozen non-benchmark serverless "
            "availability screen and restores the intended "
            "DeepSeek-family comparison without altering "
            "the frozen A5 outcome."
        ),
    },

    {
        "arm": "A10",
        "model": "openai/gpt-oss-120b",
        "role": (
            "Large GPT-OSS-family semantic-parser model"
        ),
        "selection_basis": (
            "Passed the frozen non-benchmark serverless "
            "availability screen and restores the intended "
            "GPT-OSS-family comparison without altering "
            "the frozen A6 outcome."
        ),
    },
]

# ------------------------------------------------------------------
# Preregistration
# ------------------------------------------------------------------

record = {
    "preregistration_id":
        "experiment-A-extension-A7-A10-v1",

    "created_utc":
        datetime.now(timezone.utc).isoformat(),

    "relationship_to_original_experiment": {
        "original_experiment":
            "Experiment A A1-A6",

        "original_experiment_status":
            "FROZEN",

        "extension_type":
            "prospective_additional_model_arms",

        "replacement_of_A3_A6":
            False,

        "reason": (
            "The original six-arm preregistration produced two "
            "evaluable arms and four provider-unavailable arms. "
            "A7-A10 are additional prospectively frozen arms "
            "selected only after a non-benchmark endpoint-"
            "availability screen."
        ),
    },

    "research_question": (
        "With the deterministic E3-v3 pipeline, prompt, gates, "
        "thresholds, labels, and 120-case benchmark held fixed, "
        "how do decision performance and deterministic failure "
        "containment vary across additional accessible semantic-"
        "parser model families and scales?"
    ),

    "selection_policy": {
        "availability_screen":
            "single synthetic non-benchmark request",

        "probe_contains_benchmark_case":
            False,

        "probe_contains_network_slicing_intent":
            False,

        "selection_based_on_benchmark_performance":
            False,

        "selection_criteria": [
            "serverless endpoint accessibility",
            "model-family diversity",
            "model-scale diversity",
            "instruction/chat suitability",
        ],
    },

    "arms":
        arms,

    "execution_order": [
        "A7",
        "A8",
        "A9",
        "A10",
    ],

    "frozen_components": {
        "runner": {
            "path":
                str(RUNNER.relative_to(ROOT)),

            "sha256":
                sha256(RUNNER),
        },

        "benchmark": {
            "path":
                str(DATASET.relative_to(ROOT)),

            "sha256":
                sha256(DATASET),

            "cases":
                120,

            "valid_cases":
                58,

            "invalid_cases":
                62,
        },

        "unchanged": [
            "runner",
            "benchmark",
            "prompt",
            "deterministic source analysis",
            "pre-gates",
            "reconciliation logic",
            "normalization logic",
            "preservation checks",
            "schema validation",
            "semantic validation",
            "policy validation",
            "catalog validation",
            "consistency validation",
            "mapping logic",
            "descriptor compilation logic",
            "thresholds",
            "ground-truth labels",
            "containment definition",
        ],

        "only_experimental_variable":
            "semantic parser model",
    },

    "primary_metrics": {
        "accuracy": {
            "definition":
                "correct final accept/reject decisions / 120",
            "confidence_interval":
                "exact Clopper-Pearson 95%",
        },

        "VAR": {
            "definition":
                "valid requests accepted / 58",
            "confidence_interval":
                "exact Clopper-Pearson 95%",
        },

        "IRR": {
            "definition":
                "invalid requests rejected / 62",
            "confidence_interval":
                "exact Clopper-Pearson 95%",
        },

        "containment_violations": {
            "operational_definition": (
                "accepted == False AND descriptor_json present"
            ),

            "expected":
                0,
        },
    },

    "secondary_metrics": [
        "rejection precision",
        "rejection recall",
        "rejection F1",
        "LLM invocation count",
        "LLM parse-success count",
        "accepted count",
        "descriptor count",
        "accepted-without-descriptor count",
        "rejection-stage distribution",
    ],

    "comparison_plan": {
        "reference_arm":
            "A1",

        "existing_evaluable_arms": [
            "A1",
            "A2",
        ],

        "new_extension_arms": [
            "A7",
            "A8",
            "A9",
            "A10",
        ],

        "planned_total_evaluable_models_if_all_new_arms_complete":
            6,

        "report": [
            "per-arm metrics",
            "exact confidence intervals",
            "differences relative to A1",
            "containment violations",
            "model-family and scale sensitivity",
        ],
    },

    "failure_policy": {
        "report_all_preregistered_extension_arms":
            True,

        "result_driven_model_removal":
            False,

        "result_driven_model_substitution":
            False,

        "gate_change_after_results":
            False,

        "prompt_change_after_results":
            False,

        "threshold_change_after_results":
            False,

        "label_change_after_results":
            False,

        "case_removal_after_results":
            False,

        "performance_driven_rerun":
            False,

        "provider_failure_handling": (
            "If an extension arm fails because of provider or "
            "endpoint infrastructure despite passing the prior "
            "availability screen, preserve and report the failure; "
            "do not silently replace the arm."
        ),
    },

    "claim_boundary": {
        "do_not_claim": [
            "universal model independence",
            "hallucination-free operation",
            "guaranteed zero violations outside tested conditions",
            "six-model comparison unless all six evaluable arms complete",
        ],

        "intended_interpretation": (
            "Evaluate whether semantic-parser model choice changes "
            "decision performance while deterministic failure "
            "containment remains stable under the frozen tested "
            "pipeline and benchmark."
        ),
    },

    "provenance": {
        "original_A1_A6_accounting": {
            "path":
                str(FINAL_A1_A6.relative_to(ROOT)),

            "sha256":
                sha256(FINAL_A1_A6),
        },

        "availability_screen_csv": {
            "path":
                str(SCREEN_CSV.relative_to(ROOT)),

            "sha256":
                sha256(SCREEN_CSV),
        },

        "availability_screen_manifest": {
            "path":
                str(SCREEN_JSON.relative_to(ROOT)),

            "sha256":
                sha256(SCREEN_JSON),
        },
    },
}

# ------------------------------------------------------------------
# Write JSON
# ------------------------------------------------------------------

OUT_JSON.write_text(
    json.dumps(record, indent=2) + "\n",
    encoding="utf-8"
)

# ------------------------------------------------------------------
# Human-readable preregistration
# ------------------------------------------------------------------

md = f"""# Experiment A Extension — A7–A10 Preregistration

Created UTC: {record["created_utc"]}

## Relationship to Original Experiment A

The original A1–A6 experiment is frozen.

A7–A10 are additional prospective model arms.
They do not replace A3–A6.

Selection was based on endpoint accessibility and predefined
model-diversity considerations, not held-out benchmark performance.

## Frozen Extension Arms

| Arm | Model | Role |
|---|---|---|
| A7 | `arize-ai/qwen-2-1.5b-instruct` | Small Qwen-derived model / scale sensitivity |
| A8 | `Prism-ML/Ternary-Bonsai-27B` | Medium-scale independent model family |
| A9 | `deepseek-ai/DeepSeek-V4.1-Flash` | DeepSeek-family model |
| A10 | `openai/gpt-oss-120b` | Large GPT-OSS-family model |

## Execution Order

A7 → A8 → A9 → A10

## Frozen Components

Runner SHA-256:

`{sha256(RUNNER)}`

Benchmark SHA-256:

`{sha256(DATASET)}`

The runner, benchmark, prompt, deterministic gates, thresholds,
labels, mapping logic, and containment definition remain unchanged.

The only experimental variable is the semantic-parser model.

## Primary Metrics

- Decision accuracy
- Valid Acceptance Rate (VAR)
- Invalid Rejection Rate (IRR)
- Containment violations

Exact Clopper-Pearson 95% confidence intervals will be reported
for Accuracy, VAR, and IRR.

Operational containment violation:

`accepted == False AND descriptor_json present`

Expected containment violations: 0.

## Failure Policy

All preregistered extension arms will be reported.

No result-driven model removal, model substitution, gate change,
prompt change, threshold change, label change, case removal, or
performance-driven rerun is permitted.

If an arm encounters provider/infrastructure failure, that failure
will be preserved and reported.

## Claim Boundary

The experiment does not establish universal model independence.

The intended analysis tests whether semantic-parser choice changes
decision performance while deterministic failure containment remains
stable under the frozen tested pipeline and benchmark.
"""

OUT_MD.write_text(
    md,
    encoding="utf-8"
)

# ------------------------------------------------------------------
# Freeze preregistration manifest
# ------------------------------------------------------------------

manifest = {
    "manifest_id":
        "experiment-A-extension-A7-A10-preregistration-manifest-v1",

    "created_utc":
        datetime.now(timezone.utc).isoformat(),

    "json": {
        "path":
            str(OUT_JSON.relative_to(ROOT)),

        "sha256":
            sha256(OUT_JSON),
    },

    "markdown": {
        "path":
            str(OUT_MD.relative_to(ROOT)),

        "sha256":
            sha256(OUT_MD),
    },

    "arms": [
        {
            "arm": x["arm"],
            "model": x["model"],
        }
        for x in arms
    ],

    "execution_order": [
        "A7",
        "A8",
        "A9",
        "A10",
    ],

    "benchmark_execution_started":
        False,
}

OUT_MANIFEST.write_text(
    json.dumps(manifest, indent=2) + "\n",
    encoding="utf-8"
)

# ------------------------------------------------------------------
# Report
# ------------------------------------------------------------------

print("\n" + "-" * 110)
print("PREREGISTERED EXTENSION ARMS")
print("-" * 110)

for arm in arms:
    print(
        f'{arm["arm"]:<4} '
        f'{arm["model"]}'
    )
    print(
        f'     Role: {arm["role"]}'
    )

print("\nExecution order:")
print("A7 → A8 → A9 → A10")

print("\n" + "-" * 110)
print("FROZEN ARTIFACTS")
print("-" * 110)

print("JSON:")
print(OUT_JSON)
print("SHA-256:")
print(sha256(OUT_JSON))

print("\nMarkdown:")
print(OUT_MD)
print("SHA-256:")
print(sha256(OUT_MD))

print("\nManifest:")
print(OUT_MANIFEST)
print("SHA-256:")
print(sha256(OUT_MANIFEST))

print("\n" + "-" * 110)
print("EXPERIMENTAL POLICY")
print("-" * 110)

print("Only variable             : semantic parser model")
print("Runner changes            : NOT ALLOWED")
print("Benchmark changes         : NOT ALLOWED")
print("Prompt changes            : NOT ALLOWED")
print("Gate changes              : NOT ALLOWED")
print("Threshold changes         : NOT ALLOWED")
print("Label/case changes        : NOT ALLOWED")
print("Result-driven substitution: NOT ALLOWED")
print("Performance-driven rerun  : NOT ALLOWED")

print("\n" + "=" * 110)
print("✓ A7-A10 EXTENSION PREREGISTERED")
print("✓ ORIGINAL A1-A6 REMAINS FROZEN")
print("✓ FOUR EXACT MODEL IDS FROZEN")
print("✓ EXECUTION ORDER FROZEN")
print("✓ METRICS AND CONTAINMENT DEFINITION FROZEN")
print("✓ FAILURE POLICY FROZEN")
print("✓ NO BENCHMARK CASE SUBMITTED")
print("✓ NO MODEL/API CALL MADE")
print("✓ DO NOT RUN A7 YET")
print("=" * 110)

STEP A31 — PREREGISTER EXPERIMENT A EXTENSION: A7–A10

✓ Original A1-A6 accounting verified and frozen
✓ Availability-screen CSV verified
✓ Availability-screen manifest verified
✓ Frozen E3-v3 runner verified
✓ Frozen 120-case benchmark verified

--------------------------------------------------------------------------------------------------------------
PREREGISTERED EXTENSION ARMS
--------------------------------------------------------------------------------------------------------------
A7   arize-ai/qwen-2-1.5b-instruct
     Role: Small Qwen-derived instruction model for model-scale sensitivity
A8   Prism-ML/Ternary-Bonsai-27B
     Role: Medium-scale independent model family
A9   deepseek-ai/DeepSeek-V4.1-Flash
     Role: DeepSeek-family semantic-parser model
A10  openai/gpt-oss-120b
     Role: Large GPT-OSS-family semantic-parser model

Execution order:
A7 → A8 → A9 → A10

----------------------------------------------------------------------------------------------------------

In [38]:
from pathlib import Path
from datetime import datetime, timezone
import subprocess
import hashlib
import os
import json
import pandas as pd
import time

ROOT = Path("/content/deterministic-intent-translation-journal")
EXP_A = ROOT / "journal_experiments/experiment_A_model_sweep"

RUNNER = ROOT / (
    "experiment-2b/implementation/e3_v3/"
    "run_experiment2b_e3v3_r7_dev_v1.py"
)

DATASET = ROOT / (
    "experiment-2b/reproducibility/"
    "e3v3_heldout_benchmark_v1.0/"
    "e3v3_heldout_benchmark_120_v1.csv"
)

FINAL_A1_A6 = (
    EXP_A /
    "manifests/experiment_A_final_accounting_v1.json"
)

EXT_PREREG_JSON = (
    EXP_A /
    "preregistration/"
    "experiment_A_extension_A7_A10_preregistration_v1.json"
)

EXT_PREREG_MD = (
    EXP_A /
    "preregistration/"
    "experiment_A_extension_A7_A10_preregistration_v1.md"
)

EXT_PREREG_MANIFEST = (
    EXP_A /
    "manifests/"
    "experiment_A_extension_A7_A10_preregistration_manifest_v1.json"
)

RESULT = (
    EXP_A /
    "results/raw/"
    "A7_arize-qwen-2-1.5b-instruct_results.csv"
)

LOG = (
    EXP_A /
    "logs/"
    "A7_arize-qwen-2-1.5b-instruct_execution.log"
)

EXEC_MANIFEST = (
    EXP_A /
    "manifests/"
    "A7_execution_manifest_v1.json"
)

MODEL = "arize-ai/qwen-2-1.5b-instruct"

EXPECTED = {
    "runner":
        "40c035cb864978c718a72fcfdfbb164d262529960b0ae78242d5194147cd8257",

    "dataset":
        "84576e7d99a2721e61ded5a14bdafeb96707b068397243bbed32aa735496d4de",

    "A1_A6_accounting":
        "7fe65b2b1600fdea83afde4c9e1a03f3c95257d024c1bc077e89f1edf6b63269",

    "extension_json":
        "5568bf925a7b1de1e561820673bb15d89e818d08f2c2662f4c577fb9aa626f90",

    "extension_md":
        "2708091a86ab9f3432fceea31a4ddf26becd635d24c3605c46bec68a67218e54",

    "extension_manifest":
        "7867181975990e1c46359dedda988e129b09a9aa1c35165e13f1c6cb5f8c3b8a",
}


def sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(1024 * 1024), b""):
            h.update(block)
    return h.hexdigest()


print("=" * 110)
print("STEP A32 — EXECUTE A7 ON FROZEN 120-CASE BENCHMARK")
print("=" * 110)

# ------------------------------------------------------------------
# 1. Integrity verification
# ------------------------------------------------------------------

checks = [
    ("Frozen runner", RUNNER, EXPECTED["runner"]),
    ("Frozen benchmark", DATASET, EXPECTED["dataset"]),
    (
        "A1-A6 final accounting",
        FINAL_A1_A6,
        EXPECTED["A1_A6_accounting"]
    ),
    (
        "A7-A10 preregistration JSON",
        EXT_PREREG_JSON,
        EXPECTED["extension_json"]
    ),
    (
        "A7-A10 preregistration Markdown",
        EXT_PREREG_MD,
        EXPECTED["extension_md"]
    ),
    (
        "A7-A10 preregistration manifest",
        EXT_PREREG_MANIFEST,
        EXPECTED["extension_manifest"]
    ),
]

for label, path, expected_hash in checks:

    assert path.exists(), (
        f"STOP: missing required artifact:\n{path}"
    )

    actual = sha256(path)

    assert actual == expected_hash, (
        f"STOP: {label} changed.\n"
        f"Expected: {expected_hash}\n"
        f"Actual:   {actual}"
    )

    print(f"✓ {label} verified")


# ------------------------------------------------------------------
# 2. Verify preregistered A7 identity
# ------------------------------------------------------------------

prereg = json.loads(
    EXT_PREREG_JSON.read_text(encoding="utf-8")
)

arm_map = {
    arm["arm"]: arm["model"]
    for arm in prereg["arms"]
}

assert arm_map["A7"] == MODEL, (
    "STOP: requested A7 model does not match preregistration."
)

assert prereg["execution_order"][0] == "A7", (
    "STOP: A7 is not first in frozen execution order."
)

print("✓ A7 exact model ID matches preregistration")
print("✓ Frozen execution order starts with A7")


# ------------------------------------------------------------------
# 3. Environment/output safety
# ------------------------------------------------------------------

assert os.environ.get("TOGETHER_API_KEY"), (
    "STOP: TOGETHER_API_KEY missing."
)

assert not RESULT.exists(), (
    f"STOP: A7 result already exists:\n{RESULT}\n"
    "Do not rerun A7."
)

assert not LOG.exists(), (
    f"STOP: A7 log already exists:\n{LOG}\n"
    "Do not rerun A7."
)

assert not EXEC_MANIFEST.exists(), (
    f"STOP: A7 execution manifest already exists:\n"
    f"{EXEC_MANIFEST}"
)

RESULT.parent.mkdir(parents=True, exist_ok=True)
LOG.parent.mkdir(parents=True, exist_ok=True)

print("✓ API key present")
print("✓ A7 result path clean")
print("✓ A7 log path clean")
print("✓ A7 execution-manifest path clean")


# ------------------------------------------------------------------
# 4. Exact frozen command
# ------------------------------------------------------------------

cmd = [
    "python",
    str(RUNNER),

    "--dataset",
    str(DATASET),

    "--out",
    str(RESULT),

    "--model",
    MODEL,
]

print("\n" + "-" * 110)
print("A7 FROZEN EXECUTION")
print("-" * 110)

print("Arm   : A7")
print("Model :", MODEL)

print("\nCommand:")
print(" ".join(cmd))

print("\nPolicy:")
print("  ✓ 120-case frozen benchmark")
print("  ✓ no --limit")
print("  ✓ no runner modification")
print("  ✓ no prompt modification")
print("  ✓ no gate modification")
print("  ✓ no threshold modification")
print("  ✓ no automatic rerun")


# ------------------------------------------------------------------
# 5. Execute exactly once
# ------------------------------------------------------------------

started_utc = datetime.now(timezone.utc).isoformat()
start = time.time()

proc = subprocess.run(
    cmd,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    env=os.environ.copy(),
)

elapsed = time.time() - start
finished_utc = datetime.now(timezone.utc).isoformat()

LOG.write_text(
    proc.stdout or "",
    encoding="utf-8"
)

print("\n" + "-" * 110)
print("RUNNER OUTPUT")
print("-" * 110)

print(proc.stdout)

print("\nProcess return code :", proc.returncode)
print("Wall-clock seconds :", round(elapsed, 2))


# ------------------------------------------------------------------
# 6. Do NOT rerun on failure
# ------------------------------------------------------------------

if proc.returncode != 0:

    print("\n" + "=" * 110)
    print("✗ A7 PROCESS RETURNED NON-ZERO")
    print("✗ DO NOT RERUN")
    print("✗ PRESERVE LOG AND SEND THIS OUTPUT FOR DIAGNOSIS")
    print("=" * 110)

    raise SystemExit


# ------------------------------------------------------------------
# 7. Basic result integrity only
# ------------------------------------------------------------------

assert RESULT.exists(), (
    "STOP: runner returned zero but A7 result CSV is missing."
)

df = pd.read_csv(RESULT)

assert len(df) == 120, (
    f"STOP: expected 120 rows, found {len(df)}."
)

# Determine case-ID column safely.
case_id_candidates = [
    "case_id",
    "test_id",
    "id",
]

case_col = None

for col in case_id_candidates:
    if col in df.columns:
        case_col = col
        break

if case_col is not None:
    unique_cases = df[case_col].nunique()
    assert unique_cases == 120, (
        f"STOP: expected 120 unique cases, "
        f"found {unique_cases}."
    )
else:
    unique_cases = None


# ------------------------------------------------------------------
# 8. Lightweight diagnostics only
#    No performance metrics yet.
# ------------------------------------------------------------------

def bool_count(column, value=True):

    if column not in df.columns:
        return None

    s = df[column]

    if s.dtype == bool:
        return int((s == value).sum())

    normalized = (
        s.astype(str)
        .str.strip()
        .str.lower()
    )

    target = "true" if value else "false"

    return int((normalized == target).sum())


llm_invoked = bool_count(
    "llm_invoked",
    True
)

parse_success = bool_count(
    "llm_parse_success",
    True
)

accepted = bool_count(
    "accepted",
    True
)

if "rejection_stage" in df.columns:

    rejection_stages = (
        df["rejection_stage"]
        .fillna("<none>")
        .value_counts()
        .to_dict()
    )

else:
    rejection_stages = {}


# ------------------------------------------------------------------
# 9. Freeze execution manifest
# ------------------------------------------------------------------

execution_record = {
    "manifest_id":
        "experiment-A-A7-execution-v1",

    "created_utc":
        datetime.now(timezone.utc).isoformat(),

    "arm":
        "A7",

    "model":
        MODEL,

    "started_utc":
        started_utc,

    "finished_utc":
        finished_utc,

    "wall_clock_seconds":
        elapsed,

    "return_code":
        proc.returncode,

    "command":
        cmd,

    "frozen_inputs": {
        "runner": {
            "path":
                str(RUNNER.relative_to(ROOT)),

            "sha256":
                sha256(RUNNER),
        },

        "benchmark": {
            "path":
                str(DATASET.relative_to(ROOT)),

            "sha256":
                sha256(DATASET),
        },

        "preregistration": {
            "path":
                str(EXT_PREREG_JSON.relative_to(ROOT)),

            "sha256":
                sha256(EXT_PREREG_JSON),
        },
    },

    "outputs": {
        "result": {
            "path":
                str(RESULT.relative_to(ROOT)),

            "sha256":
                sha256(RESULT),

            "rows":
                len(df),

            "unique_cases":
                unique_cases,
        },

        "log": {
            "path":
                str(LOG.relative_to(ROOT)),

            "sha256":
                sha256(LOG),
        },
    },

    "lightweight_diagnostics": {
        "llm_invoked":
            llm_invoked,

        "llm_parse_success":
            parse_success,

        "accepted":
            accepted,

        "rejection_stages":
            rejection_stages,
    },

    "performance_metrics_calculated":
        False,

    "automatic_rerun":
        False,
}

EXEC_MANIFEST.write_text(
    json.dumps(
        execution_record,
        indent=2
    ) + "\n",
    encoding="utf-8"
)


# ------------------------------------------------------------------
# 10. Report
# ------------------------------------------------------------------

print("\n" + "-" * 110)
print("A7 RAW EXECUTION ARTIFACTS")
print("-" * 110)

print("Result:")
print(RESULT)

print("Rows:", len(df))

if unique_cases is not None:
    print("Unique cases:", unique_cases)

print("SHA-256:")
print(sha256(RESULT))

print("\nLog:")
print(LOG)

print("SHA-256:")
print(sha256(LOG))

print("\nExecution manifest:")
print(EXEC_MANIFEST)

print("SHA-256:")
print(sha256(EXEC_MANIFEST))


print("\n" + "-" * 110)
print("LIGHTWEIGHT DIAGNOSTIC")
print("-" * 110)

print("LLM invoked       :", llm_invoked)
print("LLM parse success :", parse_success)
print("Accepted          :", accepted)

print("\nRejection stages:")

for stage, count in rejection_stages.items():
    print(f"  {stage}: {count}")


print("\n" + "=" * 110)
print("✓ A7 EXECUTED EXACTLY ONCE")
print("✓ 120 RESULT ROWS VERIFIED")
print("✓ RAW RESULT FROZEN")
print("✓ EXECUTION LOG FROZEN")
print("✓ NO PERFORMANCE METRICS CALCULATED YET")
print("✓ NO AUTOMATIC RERUN")
print("✓ DO NOT RUN A8 YET")
print("✓ SEND THIS OUTPUT FOR A7 ANALYSIS")
print("=" * 110)

STEP A32 — EXECUTE A7 ON FROZEN 120-CASE BENCHMARK
✓ Frozen runner verified
✓ Frozen benchmark verified
✓ A1-A6 final accounting verified
✓ A7-A10 preregistration JSON verified
✓ A7-A10 preregistration Markdown verified
✓ A7-A10 preregistration manifest verified
✓ A7 exact model ID matches preregistration
✓ Frozen execution order starts with A7
✓ API key present
✓ A7 result path clean
✓ A7 log path clean
✓ A7 execution-manifest path clean

--------------------------------------------------------------------------------------------------------------
A7 FROZEN EXECUTION
--------------------------------------------------------------------------------------------------------------
Arm   : A7
Model : arize-ai/qwen-2-1.5b-instruct

Command:
python /content/deterministic-intent-translation-journal/experiment-2b/implementation/e3_v3/run_experiment2b_e3v3_r7_dev_v1.py --dataset /content/deterministic-intent-translation-journal/experiment-2b/reproducibility/e3v3_heldout_benchmark_v1.0/e3v3_heldo

In [39]:
from pathlib import Path
from datetime import datetime, timezone
import hashlib
import json
import math

import pandas as pd
from scipy.stats import beta


ROOT = Path("/content/deterministic-intent-translation-journal")
EXP_A = ROOT / "journal_experiments/experiment_A_model_sweep"

A7_RESULT = (
    EXP_A /
    "results/raw/A7_arize-qwen-2-1.5b-instruct_results.csv"
)

A7_LOG = (
    EXP_A /
    "logs/A7_arize-qwen-2-1.5b-instruct_execution.log"
)

A7_EXEC = (
    EXP_A /
    "manifests/A7_execution_manifest_v1.json"
)

PREREG = (
    EXP_A /
    "preregistration/"
    "experiment_A_extension_A7_A10_preregistration_v1.json"
)

OUT_METRICS = (
    EXP_A /
    "results/metrics/A7_metrics_v1.json"
)

OUT_ANALYSIS = (
    EXP_A /
    "results/analysis/A7_vs_A1_analysis_v1.json"
)

OUT_MANIFEST = (
    EXP_A /
    "manifests/A7_analysis_manifest_v1.json"
)


EXPECTED = {
    "result":
        "2c41287e4cab5680e5031fb9252b003d7bdf5412a76b42a8856c9ce700a70e7c",

    "log":
        "ffcb83ddef8220b638936bccd18e4aaca9d5363cc7fc7823b0d4481240e7b280",

    "execution_manifest":
        "7bed566ff8d50d1e48a99cf576292e3708b61ac00c5490cc0da7d94863b07137",

    "preregistration":
        "5568bf925a7b1de1e561820673bb15d89e818d08f2c2662f4c577fb9aa626f90",
}


def sha256(path):
    h = hashlib.sha256()

    with open(path, "rb") as f:
        for block in iter(lambda: f.read(1024 * 1024), b""):
            h.update(block)

    return h.hexdigest()


def as_bool(series):
    return (
        series.astype(str)
        .str.strip()
        .str.lower()
        .map({
            "true": True,
            "false": False,
            "1": True,
            "0": False,
        })
    )


def cp_interval(k, n, alpha=0.05):

    if n == 0:
        return [None, None]

    lower = (
        0.0
        if k == 0
        else beta.ppf(alpha / 2, k, n - k + 1)
    )

    upper = (
        1.0
        if k == n
        else beta.ppf(
            1 - alpha / 2,
            k + 1,
            n - k
        )
    )

    return [float(lower), float(upper)]


print("=" * 110)
print("STEP A33 — ANALYZE AND FREEZE A7 METRICS")
print("=" * 110)

# ------------------------------------------------------------------
# 1. Integrity
# ------------------------------------------------------------------

for label, path, expected in [
    ("A7 raw result", A7_RESULT, EXPECTED["result"]),
    ("A7 execution log", A7_LOG, EXPECTED["log"]),
    (
        "A7 execution manifest",
        A7_EXEC,
        EXPECTED["execution_manifest"]
    ),
    (
        "A7-A10 preregistration",
        PREREG,
        EXPECTED["preregistration"]
    ),
]:

    assert path.exists(), f"STOP: missing {label}"

    actual = sha256(path)

    assert actual == expected, (
        f"STOP: {label} changed.\n"
        f"Expected: {expected}\n"
        f"Actual:   {actual}"
    )

    print(f"✓ {label} verified")


for path in [
    OUT_METRICS,
    OUT_ANALYSIS,
    OUT_MANIFEST,
]:
    assert not path.exists(), (
        f"STOP: output already exists:\n{path}"
    )


# ------------------------------------------------------------------
# 2. Load frozen A7
# ------------------------------------------------------------------

df = pd.read_csv(A7_RESULT)

assert len(df) == 120

print("\n✓ 120 A7 rows loaded")

print("\nCSV columns:")
for col in df.columns:
    print(" ", col)


# ------------------------------------------------------------------
# 3. Required semantic columns
# ------------------------------------------------------------------

assert "expected_valid" in df.columns
assert "accepted" in df.columns

expected_valid = as_bool(df["expected_valid"])
accepted = as_bool(df["accepted"])

assert expected_valid.notna().all()
assert accepted.notna().all()

valid_n = int(expected_valid.sum())
invalid_n = int((~expected_valid).sum())

assert valid_n == 58
assert invalid_n == 62

print("\n✓ Ground truth verified: 58 valid / 62 invalid")


# ------------------------------------------------------------------
# 4. Confusion matrix
#
# Invalid/rejection treated as positive:
#
# TP = invalid rejected
# FP = valid rejected
# FN = invalid accepted
# TN = valid accepted
# ------------------------------------------------------------------

valid_accepted = int(
    (expected_valid & accepted).sum()
)

valid_rejected = int(
    (expected_valid & ~accepted).sum()
)

invalid_rejected = int(
    (~expected_valid & ~accepted).sum()
)

invalid_accepted = int(
    (~expected_valid & accepted).sum()
)

TN = valid_accepted
FP = valid_rejected
TP = invalid_rejected
FN = invalid_accepted

correct = TN + TP
incorrect = FP + FN

assert correct + incorrect == 120


# ------------------------------------------------------------------
# 5. Primary metrics
# ------------------------------------------------------------------

accuracy = correct / 120
VAR = valid_accepted / 58
IRR = invalid_rejected / 62

precision = (
    TP / (TP + FP)
    if (TP + FP)
    else None
)

recall = (
    TP / (TP + FN)
    if (TP + FN)
    else None
)

f1 = (
    2 * precision * recall / (precision + recall)
    if (
        precision is not None
        and recall is not None
        and (precision + recall) > 0
    )
    else None
)

accuracy_ci = cp_interval(correct, 120)
VAR_ci = cp_interval(valid_accepted, 58)
IRR_ci = cp_interval(invalid_rejected, 62)


# ------------------------------------------------------------------
# 6. LLM invocation / response diagnostics
# ------------------------------------------------------------------

if "llm_invoked" in df.columns:
    llm_invoked_mask = as_bool(df["llm_invoked"])
    assert llm_invoked_mask.notna().all()
else:
    raise AssertionError(
        "STOP: llm_invoked column missing."
    )

llm_invoked = int(llm_invoked_mask.sum())

assert llm_invoked == 77, (
    f"STOP: expected 77 LLM invocations, found {llm_invoked}"
)

# The frozen runner's result schema may not contain a dedicated
# llm_parse_success column. Infer response-level evidence only from
# existing raw fields; do not alter the raw CSV.

parse_success_column = None

for candidate in [
    "llm_parse_success",
    "parse_success",
]:
    if candidate in df.columns:
        parse_success_column = candidate
        break


if parse_success_column is not None:

    parsed = as_bool(df[parse_success_column])

    parse_success = int(
        (
            llm_invoked_mask
            & (parsed == True)
        ).sum()
    )

    parse_failure = int(
        (
            llm_invoked_mask
            & (parsed == False)
        ).sum()
    )

    parse_diagnostic_method = (
        f"explicit column: {parse_success_column}"
    )

else:

    # Use exception rejection stage as hard infrastructure/error
    # evidence. Successful downstream processing is NOT automatically
    # labelled parse success unless the schema supports it.

    if "rejection_stage" in df.columns:

        stages = (
            df["rejection_stage"]
            .fillna("")
            .astype(str)
            .str.strip()
            .str.lower()
        )

        exception_mask = stages.eq("exception")

        exception_count = int(
            (
                llm_invoked_mask
                & exception_mask
            ).sum()
        )

    else:
        exception_count = None

    parse_success = None
    parse_failure = None

    parse_diagnostic_method = (
        "No explicit parse-success column; "
        "parse-success count not inferred."
    )


# ------------------------------------------------------------------
# 7. Provider/infrastructure failure check
# ------------------------------------------------------------------

exception_rows = 0

if "rejection_stage" in df.columns:

    stage_norm = (
        df["rejection_stage"]
        .fillna("")
        .astype(str)
        .str.strip()
        .str.lower()
    )

    exception_rows = int(
        stage_norm.eq("exception").sum()
    )


provider_failure_rows = 0

search_columns = [
    c for c in [
        "error",
        "error_code",
        "exception",
        "exception_type",
        "exception_text",
    ]
    if c in df.columns
]

if search_columns:

    error_text = (
        df[search_columns]
        .fillna("")
        .astype(str)
        .agg(" ".join, axis=1)
        .str.lower()
    )

    provider_failure_rows = int(
        error_text.str.contains(
            (
                "model_not_available|"
                "non-serverless|"
                "dedicated endpoint"
            ),
            regex=True
        ).sum()
    )


# ------------------------------------------------------------------
# 8. Descriptor / containment
# ------------------------------------------------------------------

def descriptor_present(value):

    if pd.isna(value):
        return False

    text = str(value).strip()

    if text == "":
        return False

    if text.lower() in {
        "none",
        "null",
        "nan",
        "{}",
        "[]",
    }:
        return False

    return True


assert "descriptor_json" in df.columns, (
    "STOP: descriptor_json column missing."
)

descriptor_mask = (
    df["descriptor_json"]
    .map(descriptor_present)
)

descriptor_count = int(
    descriptor_mask.sum()
)

accepted_count = int(
    accepted.sum()
)

rejected_count = int(
    (~accepted).sum()
)

containment_violations = int(
    (
        (~accepted)
        & descriptor_mask
    ).sum()
)

accepted_without_descriptor = int(
    (
        accepted
        & ~descriptor_mask
    ).sum()
)


# ------------------------------------------------------------------
# 9. Rejection stages
# ------------------------------------------------------------------

if "rejection_stage" in df.columns:

    rejection_stages = (
        df["rejection_stage"]
        .fillna("<none>")
        .value_counts()
        .to_dict()
    )

else:
    rejection_stages = {}


# ------------------------------------------------------------------
# 10. A1 frozen comparison
# ------------------------------------------------------------------

A1 = {
    "accuracy": 101 / 120,
    "VAR": 51 / 58,
    "IRR": 50 / 62,
    "accepted": 63,
    "containment_violations": 0,
}

deltas = {
    "accuracy":
        accuracy - A1["accuracy"],

    "VAR":
        VAR - A1["VAR"],

    "IRR":
        IRR - A1["IRR"],
}


# ------------------------------------------------------------------
# 11. Determine evaluability
# ------------------------------------------------------------------

evaluable = (
    llm_invoked == 77
    and exception_rows == 0
    and provider_failure_rows == 0
)

status = (
    "EVALUABLE"
    if evaluable
    else "REQUIRES_DIAGNOSIS"
)


# ------------------------------------------------------------------
# 12. Metrics record
# ------------------------------------------------------------------

metrics_record = {
    "artifact_id":
        "experiment-A-A7-metrics-v1",

    "created_utc":
        datetime.now(timezone.utc).isoformat(),

    "arm":
        "A7",

    "model":
        "arize-ai/qwen-2-1.5b-instruct",

    "status":
        status,

    "sample": {
        "total": 120,
        "valid": 58,
        "invalid": 62,
    },

    "confusion_matrix_invalid_rejection_positive": {
        "TP_invalid_rejected": TP,
        "FP_valid_rejected": FP,
        "FN_invalid_accepted": FN,
        "TN_valid_accepted": TN,
    },

    "counts": {
        "correct": correct,
        "incorrect": incorrect,

        "valid_accepted": valid_accepted,
        "valid_rejected": valid_rejected,

        "invalid_rejected": invalid_rejected,
        "invalid_accepted": invalid_accepted,

        "accepted": accepted_count,
        "rejected": rejected_count,

        "descriptors": descriptor_count,

        "containment_violations":
            containment_violations,

        "accepted_without_descriptor":
            accepted_without_descriptor,
    },

    "metrics": {
        "accuracy": {
            "numerator": correct,
            "denominator": 120,
            "value": accuracy,
            "cp95": accuracy_ci,
        },

        "VAR": {
            "numerator": valid_accepted,
            "denominator": 58,
            "value": VAR,
            "cp95": VAR_ci,
        },

        "IRR": {
            "numerator": invalid_rejected,
            "denominator": 62,
            "value": IRR,
            "cp95": IRR_ci,
        },

        "rejection_precision":
            precision,

        "rejection_recall":
            recall,

        "rejection_f1":
            f1,
    },

    "llm_execution": {
        "invoked":
            llm_invoked,

        "parse_success":
            parse_success,

        "parse_failure":
            parse_failure,

        "parse_diagnostic_method":
            parse_diagnostic_method,

        "exception_rows":
            exception_rows,

        "provider_endpoint_failure_rows":
            provider_failure_rows,
    },

    "containment": {
        "definition":
            "accepted == False AND descriptor_json present",

        "violations":
            containment_violations,

        "accepted_without_descriptor":
            accepted_without_descriptor,
    },

    "rejection_stages":
        rejection_stages,

    "source_result": {
        "path":
            str(A7_RESULT.relative_to(ROOT)),

        "sha256":
            sha256(A7_RESULT),
    },
}


OUT_METRICS.write_text(
    json.dumps(
        metrics_record,
        indent=2
    ) + "\n",
    encoding="utf-8"
)


# ------------------------------------------------------------------
# 13. A7 vs A1 record
# ------------------------------------------------------------------

analysis_record = {
    "artifact_id":
        "experiment-A-A7-vs-A1-analysis-v1",

    "created_utc":
        datetime.now(timezone.utc).isoformat(),

    "A1": {
        "model":
            "Qwen/Qwen3.5-9B",

        "accuracy":
            A1["accuracy"],

        "VAR":
            A1["VAR"],

        "IRR":
            A1["IRR"],

        "containment_violations":
            0,
    },

    "A7": {
        "model":
            "arize-ai/qwen-2-1.5b-instruct",

        "accuracy":
            accuracy,

        "VAR":
            VAR,

        "IRR":
            IRR,

        "containment_violations":
            containment_violations,
    },

    "A7_minus_A1": {
        "accuracy":
            deltas["accuracy"],

        "VAR":
            deltas["VAR"],

        "IRR":
            deltas["IRR"],
    },

    "interpretation_policy": (
        "Differences are descriptive benchmark results. "
        "Do not attribute causality solely to parameter count "
        "or model family."
    ),
}


OUT_ANALYSIS.write_text(
    json.dumps(
        analysis_record,
        indent=2
    ) + "\n",
    encoding="utf-8"
)


# ------------------------------------------------------------------
# 14. Freeze analysis manifest
# ------------------------------------------------------------------

manifest = {
    "manifest_id":
        "experiment-A-A7-analysis-manifest-v1",

    "created_utc":
        datetime.now(timezone.utc).isoformat(),

    "arm":
        "A7",

    "status":
        status,

    "raw_result": {
        "path":
            str(A7_RESULT.relative_to(ROOT)),

        "sha256":
            sha256(A7_RESULT),
    },

    "metrics": {
        "path":
            str(OUT_METRICS.relative_to(ROOT)),

        "sha256":
            sha256(OUT_METRICS),
    },

    "comparison": {
        "path":
            str(OUT_ANALYSIS.relative_to(ROOT)),

        "sha256":
            sha256(OUT_ANALYSIS),
    },

    "no_model_call":
        True,

    "raw_result_modified":
        False,
}


OUT_MANIFEST.write_text(
    json.dumps(
        manifest,
        indent=2
    ) + "\n",
    encoding="utf-8"
)


# ------------------------------------------------------------------
# 15. Report
# ------------------------------------------------------------------

print("\n" + "=" * 110)
print("A7 EVALUABILITY")
print("=" * 110)

print("Status                         :", status)
print("LLM invoked                    :", llm_invoked)
print("Exception rows                 :", exception_rows)
print("Provider endpoint failure rows :", provider_failure_rows)

print("\nParse diagnostic:")
print(parse_diagnostic_method)

if parse_success is not None:
    print("Parse success :", parse_success)
    print("Parse failure :", parse_failure)


print("\n" + "=" * 110)
print("A7 PRIMARY RESULTS")
print("=" * 110)

print(
    f"Accuracy : {correct}/120 = "
    f"{accuracy * 100:.2f}% "
    f"[{accuracy_ci[0] * 100:.2f}, "
    f"{accuracy_ci[1] * 100:.2f}]"
)

print(
    f"VAR      : {valid_accepted}/58 = "
    f"{VAR * 100:.2f}% "
    f"[{VAR_ci[0] * 100:.2f}, "
    f"{VAR_ci[1] * 100:.2f}]"
)

print(
    f"IRR      : {invalid_rejected}/62 = "
    f"{IRR * 100:.2f}% "
    f"[{IRR_ci[0] * 100:.2f}, "
    f"{IRR_ci[1] * 100:.2f}]"
)


print("\nConfusion matrix:")
print("  TP invalid rejected :", TP)
print("  FP valid rejected   :", FP)
print("  FN invalid accepted :", FN)
print("  TN valid accepted   :", TN)

print("\nSecondary:")
print(
    "  Rejection precision:",
    None if precision is None else f"{precision:.6f}"
)

print(
    "  Rejection recall   :",
    None if recall is None else f"{recall:.6f}"
)

print(
    "  Rejection F1       :",
    None if f1 is None else f"{f1:.6f}"
)


print("\n" + "=" * 110)
print("A7 CONTAINMENT")
print("=" * 110)

print("Accepted                    :", accepted_count)
print("Rejected                    :", rejected_count)
print("Descriptors                 :", descriptor_count)
print("Containment violations      :", containment_violations)
print("Accepted without descriptor :", accepted_without_descriptor)


print("\n" + "=" * 110)
print("A7 vs A1")
print("=" * 110)

print(
    f"Accuracy delta : "
    f"{deltas['accuracy'] * 100:+.2f} percentage points"
)

print(
    f"VAR delta      : "
    f"{deltas['VAR'] * 100:+.2f} percentage points"
)

print(
    f"IRR delta      : "
    f"{deltas['IRR'] * 100:+.2f} percentage points"
)


print("\nRejection stages:")
for stage, count in rejection_stages.items():
    print(f"  {stage}: {count}")


print("\n" + "-" * 110)
print("FROZEN ANALYSIS ARTIFACTS")
print("-" * 110)

print("Metrics:")
print(OUT_METRICS)
print("SHA-256:")
print(sha256(OUT_METRICS))

print("\nA7 vs A1:")
print(OUT_ANALYSIS)
print("SHA-256:")
print(sha256(OUT_ANALYSIS))

print("\nAnalysis manifest:")
print(OUT_MANIFEST)
print("SHA-256:")
print(sha256(OUT_MANIFEST))


print("\n" + "=" * 110)

if status == "EVALUABLE":

    print("✓ A7 CLASSIFIED AS EVALUABLE")
    print("✓ NO PROVIDER ENDPOINT FAILURE DETECTED")
    print("✓ PRIMARY METRICS FROZEN")
    print("✓ EXACT 95% CLOPPER-PEARSON INTERVALS FROZEN")
    print("✓ CONTAINMENT RESULT FROZEN")
    print("✓ A7 vs A1 COMPARISON FROZEN")
    print("✓ NO MODEL/API CALL MADE")
    print("✓ RAW RESULT NOT MODIFIED")
    print("✓ DO NOT RUN A8 YET")
    print("✓ SEND THIS OUTPUT BEFORE CONTINUING")

else:

    print("✗ A7 REQUIRES DIAGNOSIS")
    print("✗ DO NOT INTERPRET PERFORMANCE YET")
    print("✗ DO NOT RUN A8")
    print("✓ SEND THIS OUTPUT FOR DIAGNOSIS")

print("=" * 110)

STEP A33 — ANALYZE AND FREEZE A7 METRICS
✓ A7 raw result verified
✓ A7 execution log verified
✓ A7 execution manifest verified
✓ A7-A10 preregistration verified

✓ 120 A7 rows loaded

CSV columns:
  id
  intent
  family_id
  repair_id
  subcategory
  difficulty
  linguistic_form
  constraint_type
  control_type
  notes
  expected_valid
  expected_service
  pipeline
  accepted
  decision_correct
  llm_invoked
  parse_ok
  llm_latency_ms
  end_to_end_latency_ms
  finish_reason
  rejection_stage
  error_codes_json
  cir_json
  input_audit_json
  pre_guard_json
  post_guard_json
  cir_before_reconciliation_json
  cir_after_reconciliation_json
  reconciliation_json
  cir_before_r5_json
  cir_after_r5_json
  r5_reliability_changed
  cir_validation_json
  standards_mapping_json
  descriptor_json
  raw_text
  reasoning_json
  prompt_tokens
  completion_tokens
  adapter_result_json
  exception

✓ Ground truth verified: 58 valid / 62 invalid

A7 EVALUABILITY
Status                         : EVAL

In [40]:
from pathlib import Path
from datetime import datetime, timezone
import subprocess
import hashlib
import os
import json
import pandas as pd
import time

ROOT = Path("/content/deterministic-intent-translation-journal")
EXP_A = ROOT / "journal_experiments/experiment_A_model_sweep"

RUNNER = ROOT / (
    "experiment-2b/implementation/e3_v3/"
    "run_experiment2b_e3v3_r7_dev_v1.py"
)

DATASET = ROOT / (
    "experiment-2b/reproducibility/"
    "e3v3_heldout_benchmark_v1.0/"
    "e3v3_heldout_benchmark_120_v1.csv"
)

PREREG = (
    EXP_A /
    "preregistration/"
    "experiment_A_extension_A7_A10_preregistration_v1.json"
)

A7_ANALYSIS_MANIFEST = (
    EXP_A /
    "manifests/A7_analysis_manifest_v1.json"
)

RESULT = (
    EXP_A /
    "results/raw/"
    "A8_Ternary-Bonsai-27B_results.csv"
)

LOG = (
    EXP_A /
    "logs/"
    "A8_Ternary-Bonsai-27B_execution.log"
)

EXEC_MANIFEST = (
    EXP_A /
    "manifests/"
    "A8_execution_manifest_v1.json"
)

MODEL = "Prism-ML/Ternary-Bonsai-27B"

EXPECTED = {
    "runner":
        "40c035cb864978c718a72fcfdfbb164d262529960b0ae78242d5194147cd8257",

    "dataset":
        "84576e7d99a2721e61ded5a14bdafeb96707b068397243bbed32aa735496d4de",

    "preregistration":
        "5568bf925a7b1de1e561820673bb15d89e818d08f2c2662f4c577fb9aa626f90",

    "A7_analysis_manifest":
        "5170f3d1dd953ccac2a3838b7de0635800b0704e1388ccf49c382d689d015c0c",
}


def sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(1024 * 1024), b""):
            h.update(block)
    return h.hexdigest()


def bool_count(df, column, value=True):

    if column not in df.columns:
        return None

    s = (
        df[column]
        .astype(str)
        .str.strip()
        .str.lower()
    )

    target = "true" if value else "false"

    return int((s == target).sum())


print("=" * 110)
print("STEP A34 — EXECUTE A8 ON FROZEN 120-CASE BENCHMARK")
print("=" * 110)

# ------------------------------------------------------------------
# 1. Integrity checks
# ------------------------------------------------------------------

checks = [
    ("Frozen runner", RUNNER, EXPECTED["runner"]),
    ("Frozen benchmark", DATASET, EXPECTED["dataset"]),
    (
        "A7-A10 preregistration",
        PREREG,
        EXPECTED["preregistration"]
    ),
    (
        "Frozen A7 analysis manifest",
        A7_ANALYSIS_MANIFEST,
        EXPECTED["A7_analysis_manifest"]
    ),
]

for label, path, expected_hash in checks:

    assert path.exists(), (
        f"STOP: missing required artifact:\n{path}"
    )

    actual = sha256(path)

    assert actual == expected_hash, (
        f"STOP: {label} changed.\n"
        f"Expected: {expected_hash}\n"
        f"Actual:   {actual}"
    )

    print(f"✓ {label} verified")


# ------------------------------------------------------------------
# 2. Verify A8 preregistration
# ------------------------------------------------------------------

prereg = json.loads(
    PREREG.read_text(encoding="utf-8")
)

arm_map = {
    arm["arm"]: arm["model"]
    for arm in prereg["arms"]
}

assert arm_map["A8"] == MODEL, (
    "STOP: A8 model does not match preregistration."
)

assert prereg["execution_order"] == [
    "A7",
    "A8",
    "A9",
    "A10",
], "STOP: frozen execution order changed."

print("✓ A8 exact model ID verified")
print("✓ Frozen execution order verified")


# ------------------------------------------------------------------
# 3. Safety checks
# ------------------------------------------------------------------

assert os.environ.get("TOGETHER_API_KEY"), (
    "STOP: TOGETHER_API_KEY missing."
)

assert not RESULT.exists(), (
    f"STOP: A8 result already exists:\n{RESULT}\n"
    "Do not rerun."
)

assert not LOG.exists(), (
    f"STOP: A8 log already exists:\n{LOG}\n"
    "Do not rerun."
)

assert not EXEC_MANIFEST.exists(), (
    f"STOP: A8 execution manifest already exists:\n"
    f"{EXEC_MANIFEST}"
)

RESULT.parent.mkdir(parents=True, exist_ok=True)
LOG.parent.mkdir(parents=True, exist_ok=True)

print("✓ API key present")
print("✓ A8 output paths clean")


# ------------------------------------------------------------------
# 4. Exact command
# ------------------------------------------------------------------

cmd = [
    "python",
    str(RUNNER),

    "--dataset",
    str(DATASET),

    "--out",
    str(RESULT),

    "--model",
    MODEL,
]

print("\n" + "-" * 110)
print("A8 FROZEN EXECUTION")
print("-" * 110)

print("Arm   : A8")
print("Model :", MODEL)

print("\nCommand:")
print(" ".join(cmd))

print("\nPolicy:")
print("  ✓ frozen 120-case benchmark")
print("  ✓ no --limit")
print("  ✓ only semantic-parser model changes")
print("  ✓ no prompt/gate/threshold changes")
print("  ✓ no automatic rerun")


# ------------------------------------------------------------------
# 5. Execute once
# ------------------------------------------------------------------

started_utc = datetime.now(timezone.utc).isoformat()
start = time.time()

proc = subprocess.run(
    cmd,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    env=os.environ.copy(),
)

elapsed = time.time() - start
finished_utc = datetime.now(timezone.utc).isoformat()

LOG.write_text(
    proc.stdout or "",
    encoding="utf-8"
)

print("\n" + "-" * 110)
print("RUNNER OUTPUT")
print("-" * 110)

print(proc.stdout)

print("\nProcess return code :", proc.returncode)
print("Wall-clock seconds :", round(elapsed, 2))


# ------------------------------------------------------------------
# 6. Never automatically rerun
# ------------------------------------------------------------------

if proc.returncode != 0:

    print("\n" + "=" * 110)
    print("✗ A8 PROCESS RETURNED NON-ZERO")
    print("✗ DO NOT RERUN")
    print("✗ SEND OUTPUT FOR DIAGNOSIS")
    print("=" * 110)

    raise SystemExit


# ------------------------------------------------------------------
# 7. Basic result verification
# ------------------------------------------------------------------

assert RESULT.exists(), (
    "STOP: result CSV missing."
)

df = pd.read_csv(RESULT)

assert len(df) == 120, (
    f"STOP: expected 120 rows; found {len(df)}."
)

case_col = None

for candidate in ["id", "case_id", "test_id"]:
    if candidate in df.columns:
        case_col = candidate
        break

assert case_col is not None, (
    "STOP: case-ID column not found."
)

unique_cases = df[case_col].nunique()

assert unique_cases == 120, (
    f"STOP: expected 120 unique cases; "
    f"found {unique_cases}."
)


# ------------------------------------------------------------------
# 8. Lightweight diagnostics
# ------------------------------------------------------------------

llm_invoked = bool_count(
    df,
    "llm_invoked",
    True
)

# Use the actual runner schema identified during A7 analysis.
parse_ok = bool_count(
    df,
    "parse_ok",
    True
)

accepted = bool_count(
    df,
    "accepted",
    True
)

exception_rows = 0

if "rejection_stage" in df.columns:

    stage_norm = (
        df["rejection_stage"]
        .fillna("")
        .astype(str)
        .str.strip()
        .str.lower()
    )

    exception_rows = int(
        stage_norm.eq("exception").sum()
    )

    rejection_stages = (
        df["rejection_stage"]
        .fillna("<none>")
        .value_counts()
        .to_dict()
    )

else:

    rejection_stages = {}


# ------------------------------------------------------------------
# 9. Freeze execution
# ------------------------------------------------------------------

execution_record = {
    "manifest_id":
        "experiment-A-A8-execution-v1",

    "created_utc":
        datetime.now(timezone.utc).isoformat(),

    "arm":
        "A8",

    "model":
        MODEL,

    "started_utc":
        started_utc,

    "finished_utc":
        finished_utc,

    "wall_clock_seconds":
        elapsed,

    "return_code":
        proc.returncode,

    "command":
        cmd,

    "frozen_inputs": {
        "runner": {
            "path":
                str(RUNNER.relative_to(ROOT)),
            "sha256":
                sha256(RUNNER),
        },

        "benchmark": {
            "path":
                str(DATASET.relative_to(ROOT)),
            "sha256":
                sha256(DATASET),
        },

        "preregistration": {
            "path":
                str(PREREG.relative_to(ROOT)),
            "sha256":
                sha256(PREREG),
        },

        "prior_A7_analysis_manifest": {
            "path":
                str(A7_ANALYSIS_MANIFEST.relative_to(ROOT)),
            "sha256":
                sha256(A7_ANALYSIS_MANIFEST),
        },
    },

    "outputs": {
        "result": {
            "path":
                str(RESULT.relative_to(ROOT)),
            "sha256":
                sha256(RESULT),
            "rows":
                len(df),
            "unique_cases":
                unique_cases,
        },

        "log": {
            "path":
                str(LOG.relative_to(ROOT)),
            "sha256":
                sha256(LOG),
        },
    },

    "lightweight_diagnostics": {
        "llm_invoked":
            llm_invoked,

        "parse_ok":
            parse_ok,

        "accepted":
            accepted,

        "exception_rows":
            exception_rows,

        "rejection_stages":
            rejection_stages,
    },

    "performance_metrics_calculated":
        False,

    "automatic_rerun":
        False,
}

EXEC_MANIFEST.write_text(
    json.dumps(
        execution_record,
        indent=2
    ) + "\n",
    encoding="utf-8"
)


# ------------------------------------------------------------------
# 10. Report
# ------------------------------------------------------------------

print("\n" + "-" * 110)
print("A8 RAW EXECUTION ARTIFACTS")
print("-" * 110)

print("Result:")
print(RESULT)
print("Rows:", len(df))
print("Unique cases:", unique_cases)
print("SHA-256:")
print(sha256(RESULT))

print("\nLog:")
print(LOG)
print("SHA-256:")
print(sha256(LOG))

print("\nExecution manifest:")
print(EXEC_MANIFEST)
print("SHA-256:")
print(sha256(EXEC_MANIFEST))


print("\n" + "-" * 110)
print("LIGHTWEIGHT DIAGNOSTIC")
print("-" * 110)

print("LLM invoked   :", llm_invoked)
print("Parse OK      :", parse_ok)
print("Accepted      :", accepted)
print("Exception rows:", exception_rows)

print("\nRejection stages:")

for stage, count in rejection_stages.items():
    print(f"  {stage}: {count}")


print("\n" + "=" * 110)
print("✓ A8 EXECUTED EXACTLY ONCE")
print("✓ 120 RESULT ROWS VERIFIED")
print("✓ RAW RESULT FROZEN")
print("✓ EXECUTION LOG FROZEN")
print("✓ parse_ok READ FROM ACTUAL RUNNER SCHEMA")
print("✓ NO PERFORMANCE METRICS CALCULATED YET")
print("✓ NO AUTOMATIC RERUN")
print("✓ DO NOT RUN A9 YET")
print("✓ SEND THIS OUTPUT FOR A8 ANALYSIS")
print("=" * 110)

STEP A34 — EXECUTE A8 ON FROZEN 120-CASE BENCHMARK
✓ Frozen runner verified
✓ Frozen benchmark verified
✓ A7-A10 preregistration verified
✓ Frozen A7 analysis manifest verified
✓ A8 exact model ID verified
✓ Frozen execution order verified
✓ API key present
✓ A8 output paths clean

--------------------------------------------------------------------------------------------------------------
A8 FROZEN EXECUTION
--------------------------------------------------------------------------------------------------------------
Arm   : A8
Model : Prism-ML/Ternary-Bonsai-27B

Command:
python /content/deterministic-intent-translation-journal/experiment-2b/implementation/e3_v3/run_experiment2b_e3v3_r7_dev_v1.py --dataset /content/deterministic-intent-translation-journal/experiment-2b/reproducibility/e3v3_heldout_benchmark_v1.0/e3v3_heldout_benchmark_120_v1.csv --out /content/deterministic-intent-translation-journal/journal_experiments/experiment_A_model_sweep/results/raw/A8_Ternary-Bonsai-27B_resu

In [41]:
from pathlib import Path
from datetime import datetime, timezone
import hashlib
import json

import pandas as pd
from scipy.stats import beta


ROOT = Path("/content/deterministic-intent-translation-journal")
EXP_A = ROOT / "journal_experiments/experiment_A_model_sweep"

A8_RESULT = (
    EXP_A /
    "results/raw/A8_Ternary-Bonsai-27B_results.csv"
)

A8_LOG = (
    EXP_A /
    "logs/A8_Ternary-Bonsai-27B_execution.log"
)

A8_EXEC = (
    EXP_A /
    "manifests/A8_execution_manifest_v1.json"
)

PREREG = (
    EXP_A /
    "preregistration/"
    "experiment_A_extension_A7_A10_preregistration_v1.json"
)

A7_ANALYSIS = (
    EXP_A /
    "manifests/A7_analysis_manifest_v1.json"
)

OUT_METRICS = (
    EXP_A /
    "results/metrics/A8_metrics_v1.json"
)

OUT_ANALYSIS = (
    EXP_A /
    "results/analysis/A8_vs_A1_analysis_v1.json"
)

OUT_MANIFEST = (
    EXP_A /
    "manifests/A8_analysis_manifest_v1.json"
)


EXPECTED = {
    "result":
        "1507e111876a27801fbd03998cfa8f8390b663be954479386f01bf2ae242b082",

    "log":
        "d520df13bc2c8c6e81a04124b83dedba2600a841a93d53c5f1ea3617479c87a6",

    "execution_manifest":
        "d72e86531b4d0ab2dd4d07001b87c18acb58096f39e7ba978c1b4fa40993d2df",

    "preregistration":
        "5568bf925a7b1de1e561820673bb15d89e818d08f2c2662f4c577fb9aa626f90",

    "A7_analysis_manifest":
        "5170f3d1dd953ccac2a3838b7de0635800b0704e1388ccf49c382d689d015c0c",
}


def sha256(path):
    h = hashlib.sha256()

    with open(path, "rb") as f:
        for block in iter(lambda: f.read(1024 * 1024), b""):
            h.update(block)

    return h.hexdigest()


def as_bool(series):
    return (
        series.astype(str)
        .str.strip()
        .str.lower()
        .map({
            "true": True,
            "false": False,
            "1": True,
            "0": False,
        })
    )


def cp_interval(k, n, alpha=0.05):

    if n == 0:
        return [None, None]

    lower = (
        0.0
        if k == 0
        else beta.ppf(
            alpha / 2,
            k,
            n - k + 1
        )
    )

    upper = (
        1.0
        if k == n
        else beta.ppf(
            1 - alpha / 2,
            k + 1,
            n - k
        )
    )

    return [
        float(lower),
        float(upper),
    ]


def descriptor_present(value):

    if pd.isna(value):
        return False

    text = str(value).strip()

    return (
        text != ""
        and text.lower()
        not in {
            "none",
            "null",
            "nan",
            "{}",
            "[]",
        }
    )


print("=" * 110)
print("STEP A35 — ANALYZE AND FREEZE A8 METRICS")
print("=" * 110)

# ------------------------------------------------------------------
# 1. Verify frozen inputs
# ------------------------------------------------------------------

checks = [
    (
        "A8 raw result",
        A8_RESULT,
        EXPECTED["result"]
    ),
    (
        "A8 execution log",
        A8_LOG,
        EXPECTED["log"]
    ),
    (
        "A8 execution manifest",
        A8_EXEC,
        EXPECTED["execution_manifest"]
    ),
    (
        "A7-A10 preregistration",
        PREREG,
        EXPECTED["preregistration"]
    ),
    (
        "A7 analysis manifest",
        A7_ANALYSIS,
        EXPECTED["A7_analysis_manifest"]
    ),
]

for label, path, expected_hash in checks:

    assert path.exists(), (
        f"STOP: missing {label}:\n{path}"
    )

    actual = sha256(path)

    assert actual == expected_hash, (
        f"STOP: {label} changed.\n"
        f"Expected: {expected_hash}\n"
        f"Actual:   {actual}"
    )

    print(f"✓ {label} verified")


for path in [
    OUT_METRICS,
    OUT_ANALYSIS,
    OUT_MANIFEST,
]:
    assert not path.exists(), (
        f"STOP: output already exists:\n{path}"
    )


# ------------------------------------------------------------------
# 2. Load result
# ------------------------------------------------------------------

df = pd.read_csv(A8_RESULT)

assert len(df) == 120

assert df["id"].nunique() == 120

print("✓ 120 rows / 120 unique cases verified")


# ------------------------------------------------------------------
# 3. Ground truth + decisions
# ------------------------------------------------------------------

expected_valid = as_bool(
    df["expected_valid"]
)

accepted = as_bool(
    df["accepted"]
)

parse_ok = as_bool(
    df["parse_ok"]
)

llm_invoked_mask = as_bool(
    df["llm_invoked"]
)

assert expected_valid.notna().all()
assert accepted.notna().all()
assert llm_invoked_mask.notna().all()

valid_n = int(
    expected_valid.sum()
)

invalid_n = int(
    (~expected_valid).sum()
)

assert valid_n == 58
assert invalid_n == 62

llm_invoked = int(
    llm_invoked_mask.sum()
)

assert llm_invoked == 77


# ------------------------------------------------------------------
# 4. Parse status
# ------------------------------------------------------------------

parse_success = int(
    (
        llm_invoked_mask
        & (parse_ok == True)
    ).sum()
)

parse_failure = int(
    (
        llm_invoked_mask
        & (parse_ok == False)
    ).sum()
)

parse_unknown = int(
    (
        llm_invoked_mask
        & parse_ok.isna()
    ).sum()
)


# ------------------------------------------------------------------
# 5. Confusion matrix
#
# Invalid/rejection = positive class
# ------------------------------------------------------------------

valid_accepted = int(
    (
        expected_valid
        & accepted
    ).sum()
)

valid_rejected = int(
    (
        expected_valid
        & ~accepted
    ).sum()
)

invalid_rejected = int(
    (
        ~expected_valid
        & ~accepted
    ).sum()
)

invalid_accepted = int(
    (
        ~expected_valid
        & accepted
    ).sum()
)

TP = invalid_rejected
FP = valid_rejected
FN = invalid_accepted
TN = valid_accepted

correct = TP + TN
incorrect = FP + FN

assert correct + incorrect == 120


# ------------------------------------------------------------------
# 6. Primary metrics
# ------------------------------------------------------------------

accuracy = correct / 120
VAR = valid_accepted / 58
IRR = invalid_rejected / 62

accuracy_ci = cp_interval(
    correct,
    120
)

VAR_ci = cp_interval(
    valid_accepted,
    58
)

IRR_ci = cp_interval(
    invalid_rejected,
    62
)


# ------------------------------------------------------------------
# 7. Secondary metrics
# ------------------------------------------------------------------

precision = (
    TP / (TP + FP)
    if (TP + FP) > 0
    else None
)

recall = (
    TP / (TP + FN)
    if (TP + FN) > 0
    else None
)

f1 = (
    2 * precision * recall /
    (precision + recall)

    if (
        precision is not None
        and recall is not None
        and (precision + recall) > 0
    )

    else None
)


# ------------------------------------------------------------------
# 8. Infrastructure / exception check
# ------------------------------------------------------------------

stage_norm = (
    df["rejection_stage"]
    .fillna("")
    .astype(str)
    .str.strip()
    .str.lower()
)

exception_rows = int(
    stage_norm.eq("exception").sum()
)

provider_failure_rows = 0

search_columns = [
    c for c in [
        "exception",
        "error_codes_json",
    ]
    if c in df.columns
]

if search_columns:

    combined_errors = (
        df[search_columns]
        .fillna("")
        .astype(str)
        .agg(" ".join, axis=1)
        .str.lower()
    )

    provider_failure_rows = int(
        combined_errors.str.contains(
            (
                "model_not_available|"
                "non-serverless|"
                "dedicated endpoint"
            ),
            regex=True
        ).sum()
    )


# ------------------------------------------------------------------
# 9. Descriptor + containment
# ------------------------------------------------------------------

descriptor_mask = (
    df["descriptor_json"]
    .map(descriptor_present)
)

accepted_count = int(
    accepted.sum()
)

rejected_count = int(
    (~accepted).sum()
)

descriptor_count = int(
    descriptor_mask.sum()
)

containment_violations = int(
    (
        ~accepted
        & descriptor_mask
    ).sum()
)

accepted_without_descriptor = int(
    (
        accepted
        & ~descriptor_mask
    ).sum()
)


# ------------------------------------------------------------------
# 10. Rejection stages
# ------------------------------------------------------------------

rejection_stages = (
    df["rejection_stage"]
    .fillna("<none>")
    .value_counts()
    .to_dict()
)


# ------------------------------------------------------------------
# 11. Evaluability
# ------------------------------------------------------------------

evaluable = (
    llm_invoked == 77
    and parse_success == 77
    and parse_failure == 0
    and parse_unknown == 0
    and exception_rows == 0
    and provider_failure_rows == 0
)

status = (
    "EVALUABLE"
    if evaluable
    else "REQUIRES_DIAGNOSIS"
)


# ------------------------------------------------------------------
# 12. Frozen A1 reference
# ------------------------------------------------------------------

A1 = {
    "accuracy":
        101 / 120,

    "VAR":
        51 / 58,

    "IRR":
        50 / 62,

    "containment_violations":
        0,
}

deltas = {
    "accuracy":
        accuracy - A1["accuracy"],

    "VAR":
        VAR - A1["VAR"],

    "IRR":
        IRR - A1["IRR"],
}


# ------------------------------------------------------------------
# 13. Write metrics
# ------------------------------------------------------------------

metrics_record = {
    "artifact_id":
        "experiment-A-A8-metrics-v1",

    "created_utc":
        datetime.now(timezone.utc).isoformat(),

    "arm":
        "A8",

    "model":
        "Prism-ML/Ternary-Bonsai-27B",

    "status":
        status,

    "sample": {
        "total": 120,
        "valid": 58,
        "invalid": 62,
    },

    "confusion_matrix_invalid_rejection_positive": {
        "TP_invalid_rejected":
            TP,

        "FP_valid_rejected":
            FP,

        "FN_invalid_accepted":
            FN,

        "TN_valid_accepted":
            TN,
    },

    "counts": {
        "correct":
            correct,

        "incorrect":
            incorrect,

        "valid_accepted":
            valid_accepted,

        "valid_rejected":
            valid_rejected,

        "invalid_rejected":
            invalid_rejected,

        "invalid_accepted":
            invalid_accepted,

        "accepted":
            accepted_count,

        "rejected":
            rejected_count,

        "descriptors":
            descriptor_count,

        "containment_violations":
            containment_violations,

        "accepted_without_descriptor":
            accepted_without_descriptor,
    },

    "metrics": {
        "accuracy": {
            "numerator":
                correct,

            "denominator":
                120,

            "value":
                accuracy,

            "cp95":
                accuracy_ci,
        },

        "VAR": {
            "numerator":
                valid_accepted,

            "denominator":
                58,

            "value":
                VAR,

            "cp95":
                VAR_ci,
        },

        "IRR": {
            "numerator":
                invalid_rejected,

            "denominator":
                62,

            "value":
                IRR,

            "cp95":
                IRR_ci,
        },

        "rejection_precision":
            precision,

        "rejection_recall":
            recall,

        "rejection_f1":
            f1,
    },

    "llm_execution": {
        "invoked":
            llm_invoked,

        "parse_success":
            parse_success,

        "parse_failure":
            parse_failure,

        "parse_unknown":
            parse_unknown,

        "exception_rows":
            exception_rows,

        "provider_endpoint_failure_rows":
            provider_failure_rows,
    },

    "containment": {
        "definition":
            "accepted == False AND descriptor_json present",

        "violations":
            containment_violations,

        "accepted_without_descriptor":
            accepted_without_descriptor,
    },

    "rejection_stages":
        rejection_stages,

    "source_result": {
        "path":
            str(A8_RESULT.relative_to(ROOT)),

        "sha256":
            sha256(A8_RESULT),
    },
}

OUT_METRICS.write_text(
    json.dumps(
        metrics_record,
        indent=2
    ) + "\n",
    encoding="utf-8"
)


# ------------------------------------------------------------------
# 14. Comparison record
# ------------------------------------------------------------------

comparison_record = {
    "artifact_id":
        "experiment-A-A8-vs-A1-analysis-v1",

    "created_utc":
        datetime.now(timezone.utc).isoformat(),

    "A1": {
        "model":
            "Qwen/Qwen3.5-9B",

        "accuracy":
            A1["accuracy"],

        "VAR":
            A1["VAR"],

        "IRR":
            A1["IRR"],

        "containment_violations":
            0,
    },

    "A8": {
        "model":
            "Prism-ML/Ternary-Bonsai-27B",

        "accuracy":
            accuracy,

        "VAR":
            VAR,

        "IRR":
            IRR,

        "containment_violations":
            containment_violations,
    },

    "A8_minus_A1": {
        "accuracy":
            deltas["accuracy"],

        "VAR":
            deltas["VAR"],

        "IRR":
            deltas["IRR"],
    },

    "interpretation_policy": (
        "Differences are descriptive benchmark results. "
        "Do not attribute causality solely to model family, "
        "parameter count, architecture, or quantization."
    ),
}

OUT_ANALYSIS.write_text(
    json.dumps(
        comparison_record,
        indent=2
    ) + "\n",
    encoding="utf-8"
)


# ------------------------------------------------------------------
# 15. Freeze analysis manifest
# ------------------------------------------------------------------

manifest = {
    "manifest_id":
        "experiment-A-A8-analysis-manifest-v1",

    "created_utc":
        datetime.now(timezone.utc).isoformat(),

    "arm":
        "A8",

    "status":
        status,

    "raw_result": {
        "path":
            str(A8_RESULT.relative_to(ROOT)),

        "sha256":
            sha256(A8_RESULT),
    },

    "metrics": {
        "path":
            str(OUT_METRICS.relative_to(ROOT)),

        "sha256":
            sha256(OUT_METRICS),
    },

    "comparison": {
        "path":
            str(OUT_ANALYSIS.relative_to(ROOT)),

        "sha256":
            sha256(OUT_ANALYSIS),
    },

    "no_model_call":
        True,

    "raw_result_modified":
        False,
}

OUT_MANIFEST.write_text(
    json.dumps(
        manifest,
        indent=2
    ) + "\n",
    encoding="utf-8"
)


# ------------------------------------------------------------------
# 16. Report
# ------------------------------------------------------------------

print("\n" + "=" * 110)
print("A8 EVALUABILITY")
print("=" * 110)

print("Status                         :", status)
print("LLM invoked                    :", llm_invoked)
print("Parse success                  :", parse_success)
print("Parse failure                  :", parse_failure)
print("Parse unknown                  :", parse_unknown)
print("Exception rows                 :", exception_rows)
print("Provider endpoint failure rows :", provider_failure_rows)


print("\n" + "=" * 110)
print("A8 PRIMARY RESULTS")
print("=" * 110)

print(
    f"Accuracy : {correct}/120 = "
    f"{accuracy * 100:.2f}% "
    f"[{accuracy_ci[0] * 100:.2f}, "
    f"{accuracy_ci[1] * 100:.2f}]"
)

print(
    f"VAR      : {valid_accepted}/58 = "
    f"{VAR * 100:.2f}% "
    f"[{VAR_ci[0] * 100:.2f}, "
    f"{VAR_ci[1] * 100:.2f}]"
)

print(
    f"IRR      : {invalid_rejected}/62 = "
    f"{IRR * 100:.2f}% "
    f"[{IRR_ci[0] * 100:.2f}, "
    f"{IRR_ci[1] * 100:.2f}]"
)

print("\nConfusion matrix:")
print("  TP invalid rejected :", TP)
print("  FP valid rejected   :", FP)
print("  FN invalid accepted :", FN)
print("  TN valid accepted   :", TN)

print("\nSecondary:")

print(
    "  Rejection precision:",
    None if precision is None
    else f"{precision:.6f}"
)

print(
    "  Rejection recall   :",
    None if recall is None
    else f"{recall:.6f}"
)

print(
    "  Rejection F1       :",
    None if f1 is None
    else f"{f1:.6f}"
)


print("\n" + "=" * 110)
print("A8 CONTAINMENT")
print("=" * 110)

print("Accepted                    :", accepted_count)
print("Rejected                    :", rejected_count)
print("Descriptors                 :", descriptor_count)
print("Containment violations      :", containment_violations)
print("Accepted without descriptor :", accepted_without_descriptor)


print("\n" + "=" * 110)
print("A8 vs A1")
print("=" * 110)

print(
    f"Accuracy delta : "
    f"{deltas['accuracy'] * 100:+.2f} percentage points"
)

print(
    f"VAR delta      : "
    f"{deltas['VAR'] * 100:+.2f} percentage points"
)

print(
    f"IRR delta      : "
    f"{deltas['IRR'] * 100:+.2f} percentage points"
)


print("\nRejection stages:")

for stage, count in rejection_stages.items():
    print(f"  {stage}: {count}")


print("\n" + "-" * 110)
print("FROZEN ANALYSIS ARTIFACTS")
print("-" * 110)

print("Metrics:")
print(OUT_METRICS)
print("SHA-256:")
print(sha256(OUT_METRICS))

print("\nA8 vs A1:")
print(OUT_ANALYSIS)
print("SHA-256:")
print(sha256(OUT_ANALYSIS))

print("\nAnalysis manifest:")
print(OUT_MANIFEST)
print("SHA-256:")
print(sha256(OUT_MANIFEST))


print("\n" + "=" * 110)

if status == "EVALUABLE":

    print("✓ A8 CLASSIFIED AS EVALUABLE")
    print("✓ ALL 77 LLM INVOCATIONS PARSED SUCCESSFULLY")
    print("✓ NO PROVIDER ENDPOINT FAILURE DETECTED")
    print("✓ PRIMARY METRICS FROZEN")
    print("✓ EXACT 95% CLOPPER-PEARSON INTERVALS FROZEN")
    print("✓ CONTAINMENT RESULT FROZEN")
    print("✓ A8 vs A1 COMPARISON FROZEN")
    print("✓ NO MODEL/API CALL MADE")
    print("✓ RAW RESULT NOT MODIFIED")
    print("✓ DO NOT RUN A9 YET")
    print("✓ SEND THIS OUTPUT BEFORE CONTINUING")

else:

    print("✗ A8 REQUIRES DIAGNOSIS")
    print("✗ DO NOT INTERPRET PERFORMANCE YET")
    print("✗ DO NOT RUN A9")
    print("✓ SEND THIS OUTPUT FOR DIAGNOSIS")

print("=" * 110)

STEP A35 — ANALYZE AND FREEZE A8 METRICS
✓ A8 raw result verified
✓ A8 execution log verified
✓ A8 execution manifest verified
✓ A7-A10 preregistration verified
✓ A7 analysis manifest verified
✓ 120 rows / 120 unique cases verified

A8 EVALUABILITY
Status                         : EVALUABLE
LLM invoked                    : 77
Parse success                  : 77
Parse failure                  : 0
Parse unknown                  : 0
Exception rows                 : 0
Provider endpoint failure rows : 0

A8 PRIMARY RESULTS
Accuracy : 91/120 = 75.83% [67.17, 83.18]
VAR      : 41/58 = 70.69% [57.27, 81.91]
IRR      : 50/62 = 80.65% [68.63, 89.58]

Confusion matrix:
  TP invalid rejected : 50
  FP valid rejected   : 17
  FN invalid accepted : 12
  TN valid accepted   : 41

Secondary:
  Rejection precision: 0.746269
  Rejection recall   : 0.806452
  Rejection F1       : 0.775194

A8 CONTAINMENT
Accepted                    : 53
Rejected                    : 67
Descriptors                 : 53
Co

In [42]:
from pathlib import Path
from datetime import datetime, timezone
import subprocess
import hashlib
import os
import json
import pandas as pd
import time

ROOT = Path("/content/deterministic-intent-translation-journal")
EXP_A = ROOT / "journal_experiments/experiment_A_model_sweep"

RUNNER = ROOT / (
    "experiment-2b/implementation/e3_v3/"
    "run_experiment2b_e3v3_r7_dev_v1.py"
)

DATASET = ROOT / (
    "experiment-2b/reproducibility/"
    "e3v3_heldout_benchmark_v1.0/"
    "e3v3_heldout_benchmark_120_v1.csv"
)

PREREG = (
    EXP_A /
    "preregistration/"
    "experiment_A_extension_A7_A10_preregistration_v1.json"
)

A8_ANALYSIS = (
    EXP_A /
    "manifests/A8_analysis_manifest_v1.json"
)

RESULT = (
    EXP_A /
    "results/raw/"
    "A9_DeepSeek-V4.1-Flash_results.csv"
)

LOG = (
    EXP_A /
    "logs/"
    "A9_DeepSeek-V4.1-Flash_execution.log"
)

EXEC_MANIFEST = (
    EXP_A /
    "manifests/"
    "A9_execution_manifest_v1.json"
)

MODEL = "deepseek-ai/DeepSeek-V4.1-Flash"

EXPECTED = {
    "runner":
        "40c035cb864978c718a72fcfdfbb164d262529960b0ae78242d5194147cd8257",

    "dataset":
        "84576e7d99a2721e61ded5a14bdafeb96707b068397243bbed32aa735496d4de",

    "preregistration":
        "5568bf925a7b1de1e561820673bb15d89e818d08f2c2662f4c577fb9aa626f90",

    "A8_analysis":
        "402cd349c2537c9dc6f3891e93bd5635dd71fc28dab8ab9d873ff1d538dcdd4e",
}


def sha256(path):
    h = hashlib.sha256()

    with open(path, "rb") as f:
        for block in iter(lambda: f.read(1024 * 1024), b""):
            h.update(block)

    return h.hexdigest()


def bool_count(df, column, value=True):

    if column not in df.columns:
        return None

    s = (
        df[column]
        .astype(str)
        .str.strip()
        .str.lower()
    )

    target = "true" if value else "false"

    return int((s == target).sum())


print("=" * 110)
print("STEP A36 — EXECUTE A9 ON FROZEN 120-CASE BENCHMARK")
print("=" * 110)

# ------------------------------------------------------------------
# 1. Verify frozen artifacts
# ------------------------------------------------------------------

checks = [
    (
        "Frozen runner",
        RUNNER,
        EXPECTED["runner"]
    ),
    (
        "Frozen benchmark",
        DATASET,
        EXPECTED["dataset"]
    ),
    (
        "A7-A10 preregistration",
        PREREG,
        EXPECTED["preregistration"]
    ),
    (
        "Frozen A8 analysis manifest",
        A8_ANALYSIS,
        EXPECTED["A8_analysis"]
    ),
]

for label, path, expected_hash in checks:

    assert path.exists(), (
        f"STOP: missing required artifact:\n{path}"
    )

    actual = sha256(path)

    assert actual == expected_hash, (
        f"STOP: {label} changed.\n"
        f"Expected: {expected_hash}\n"
        f"Actual:   {actual}"
    )

    print(f"✓ {label} verified")


# ------------------------------------------------------------------
# 2. Verify preregistered A9 identity/order
# ------------------------------------------------------------------

prereg = json.loads(
    PREREG.read_text(encoding="utf-8")
)

arm_map = {
    arm["arm"]: arm["model"]
    for arm in prereg["arms"]
}

assert arm_map["A9"] == MODEL, (
    "STOP: A9 model does not match preregistration."
)

assert prereg["execution_order"] == [
    "A7",
    "A8",
    "A9",
    "A10",
], "STOP: frozen execution order changed."

print("✓ A9 exact model ID verified")
print("✓ Frozen execution order verified")


# ------------------------------------------------------------------
# 3. Output/API safety
# ------------------------------------------------------------------

assert os.environ.get("TOGETHER_API_KEY"), (
    "STOP: TOGETHER_API_KEY missing."
)

assert not RESULT.exists(), (
    f"STOP: A9 result already exists:\n{RESULT}\n"
    "Do not rerun A9."
)

assert not LOG.exists(), (
    f"STOP: A9 log already exists:\n{LOG}\n"
    "Do not rerun A9."
)

assert not EXEC_MANIFEST.exists(), (
    f"STOP: A9 execution manifest already exists:\n"
    f"{EXEC_MANIFEST}"
)

RESULT.parent.mkdir(
    parents=True,
    exist_ok=True
)

LOG.parent.mkdir(
    parents=True,
    exist_ok=True
)

print("✓ API key present")
print("✓ A9 output paths clean")


# ------------------------------------------------------------------
# 4. Exact frozen command
# ------------------------------------------------------------------

cmd = [
    "python",
    str(RUNNER),

    "--dataset",
    str(DATASET),

    "--out",
    str(RESULT),

    "--model",
    MODEL,
]

print("\n" + "-" * 110)
print("A9 FROZEN EXECUTION")
print("-" * 110)

print("Arm   : A9")
print("Model :", MODEL)

print("\nCommand:")
print(" ".join(cmd))

print("\nPolicy:")
print("  ✓ frozen 120-case benchmark")
print("  ✓ no --limit")
print("  ✓ only semantic-parser model changes")
print("  ✓ no prompt/gate/threshold changes")
print("  ✓ no automatic rerun")


# ------------------------------------------------------------------
# 5. Execute exactly once
# ------------------------------------------------------------------

started_utc = datetime.now(
    timezone.utc
).isoformat()

start = time.time()

proc = subprocess.run(
    cmd,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    env=os.environ.copy(),
)

elapsed = time.time() - start

finished_utc = datetime.now(
    timezone.utc
).isoformat()

LOG.write_text(
    proc.stdout or "",
    encoding="utf-8"
)

print("\n" + "-" * 110)
print("RUNNER OUTPUT")
print("-" * 110)

print(proc.stdout)

print(
    "\nProcess return code :",
    proc.returncode
)

print(
    "Wall-clock seconds :",
    round(elapsed, 2)
)


# ------------------------------------------------------------------
# 6. No automatic rerun
# ------------------------------------------------------------------

if proc.returncode != 0:

    print("\n" + "=" * 110)
    print("✗ A9 PROCESS RETURNED NON-ZERO")
    print("✗ DO NOT RERUN")
    print("✗ SEND OUTPUT FOR DIAGNOSIS")
    print("=" * 110)

    raise SystemExit


# ------------------------------------------------------------------
# 7. Verify result
# ------------------------------------------------------------------

assert RESULT.exists(), (
    "STOP: A9 result CSV missing."
)

df = pd.read_csv(RESULT)

assert len(df) == 120, (
    f"STOP: expected 120 rows; "
    f"found {len(df)}."
)

assert "id" in df.columns, (
    "STOP: id column missing."
)

unique_cases = df["id"].nunique()

assert unique_cases == 120, (
    f"STOP: expected 120 unique cases; "
    f"found {unique_cases}."
)


# ------------------------------------------------------------------
# 8. Lightweight diagnostics
# ------------------------------------------------------------------

llm_invoked = bool_count(
    df,
    "llm_invoked",
    True
)

parse_ok = bool_count(
    df,
    "parse_ok",
    True
)

accepted = bool_count(
    df,
    "accepted",
    True
)

exception_rows = 0

if "rejection_stage" in df.columns:

    stage_norm = (
        df["rejection_stage"]
        .fillna("")
        .astype(str)
        .str.strip()
        .str.lower()
    )

    exception_rows = int(
        stage_norm.eq("exception").sum()
    )

    rejection_stages = (
        df["rejection_stage"]
        .fillna("<none>")
        .value_counts()
        .to_dict()
    )

else:

    rejection_stages = {}


# ------------------------------------------------------------------
# 9. Freeze execution manifest
# ------------------------------------------------------------------

execution_record = {
    "manifest_id":
        "experiment-A-A9-execution-v1",

    "created_utc":
        datetime.now(timezone.utc).isoformat(),

    "arm":
        "A9",

    "model":
        MODEL,

    "started_utc":
        started_utc,

    "finished_utc":
        finished_utc,

    "wall_clock_seconds":
        elapsed,

    "return_code":
        proc.returncode,

    "command":
        cmd,

    "frozen_inputs": {
        "runner": {
            "path":
                str(RUNNER.relative_to(ROOT)),

            "sha256":
                sha256(RUNNER),
        },

        "benchmark": {
            "path":
                str(DATASET.relative_to(ROOT)),

            "sha256":
                sha256(DATASET),
        },

        "preregistration": {
            "path":
                str(PREREG.relative_to(ROOT)),

            "sha256":
                sha256(PREREG),
        },

        "prior_A8_analysis_manifest": {
            "path":
                str(A8_ANALYSIS.relative_to(ROOT)),

            "sha256":
                sha256(A8_ANALYSIS),
        },
    },

    "outputs": {
        "result": {
            "path":
                str(RESULT.relative_to(ROOT)),

            "sha256":
                sha256(RESULT),

            "rows":
                len(df),

            "unique_cases":
                unique_cases,
        },

        "log": {
            "path":
                str(LOG.relative_to(ROOT)),

            "sha256":
                sha256(LOG),
        },
    },

    "lightweight_diagnostics": {
        "llm_invoked":
            llm_invoked,

        "parse_ok":
            parse_ok,

        "accepted":
            accepted,

        "exception_rows":
            exception_rows,

        "rejection_stages":
            rejection_stages,
    },

    "performance_metrics_calculated":
        False,

    "automatic_rerun":
        False,
}

EXEC_MANIFEST.write_text(
    json.dumps(
        execution_record,
        indent=2
    ) + "\n",
    encoding="utf-8"
)


# ------------------------------------------------------------------
# 10. Report
# ------------------------------------------------------------------

print("\n" + "-" * 110)
print("A9 RAW EXECUTION ARTIFACTS")
print("-" * 110)

print("Result:")
print(RESULT)

print("Rows:", len(df))
print("Unique cases:", unique_cases)

print("SHA-256:")
print(sha256(RESULT))

print("\nLog:")
print(LOG)

print("SHA-256:")
print(sha256(LOG))

print("\nExecution manifest:")
print(EXEC_MANIFEST)

print("SHA-256:")
print(sha256(EXEC_MANIFEST))


print("\n" + "-" * 110)
print("LIGHTWEIGHT DIAGNOSTIC")
print("-" * 110)

print("LLM invoked   :", llm_invoked)
print("Parse OK      :", parse_ok)
print("Accepted      :", accepted)
print("Exception rows:", exception_rows)

print("\nRejection stages:")

for stage, count in rejection_stages.items():
    print(f"  {stage}: {count}")


print("\n" + "=" * 110)
print("✓ A9 EXECUTED EXACTLY ONCE")
print("✓ 120 RESULT ROWS VERIFIED")
print("✓ RAW RESULT FROZEN")
print("✓ EXECUTION LOG FROZEN")
print("✓ NO PERFORMANCE METRICS CALCULATED YET")
print("✓ NO AUTOMATIC RERUN")
print("✓ DO NOT RUN A10 YET")
print("✓ SEND THIS OUTPUT FOR A9 ANALYSIS")
print("=" * 110)

STEP A36 — EXECUTE A9 ON FROZEN 120-CASE BENCHMARK
✓ Frozen runner verified
✓ Frozen benchmark verified
✓ A7-A10 preregistration verified
✓ Frozen A8 analysis manifest verified
✓ A9 exact model ID verified
✓ Frozen execution order verified
✓ API key present
✓ A9 output paths clean

--------------------------------------------------------------------------------------------------------------
A9 FROZEN EXECUTION
--------------------------------------------------------------------------------------------------------------
Arm   : A9
Model : deepseek-ai/DeepSeek-V4.1-Flash

Command:
python /content/deterministic-intent-translation-journal/experiment-2b/implementation/e3_v3/run_experiment2b_e3v3_r7_dev_v1.py --dataset /content/deterministic-intent-translation-journal/experiment-2b/reproducibility/e3v3_heldout_benchmark_v1.0/e3v3_heldout_benchmark_120_v1.csv --out /content/deterministic-intent-translation-journal/journal_experiments/experiment_A_model_sweep/results/raw/A9_DeepSeek-V4.1-Flash

In [43]:
from pathlib import Path
from datetime import datetime, timezone
import hashlib
import json

import pandas as pd
from scipy.stats import beta


ROOT = Path("/content/deterministic-intent-translation-journal")
EXP_A = ROOT / "journal_experiments/experiment_A_model_sweep"

A9_RESULT = (
    EXP_A /
    "results/raw/A9_DeepSeek-V4.1-Flash_results.csv"
)

A9_LOG = (
    EXP_A /
    "logs/A9_DeepSeek-V4.1-Flash_execution.log"
)

A9_EXEC = (
    EXP_A /
    "manifests/A9_execution_manifest_v1.json"
)

PREREG = (
    EXP_A /
    "preregistration/"
    "experiment_A_extension_A7_A10_preregistration_v1.json"
)

A8_ANALYSIS = (
    EXP_A /
    "manifests/A8_analysis_manifest_v1.json"
)

# Original frozen A1 result
A1_RESULT = (
    ROOT /
    "experiment-2b/reproducibility/"
    "e3v3_heldout_benchmark_v1.0/"
    "e3v3_heldout_results_v1.csv"
)

OUT_METRICS = (
    EXP_A /
    "results/metrics/A9_metrics_v1.json"
)

OUT_ANALYSIS = (
    EXP_A /
    "results/analysis/A9_vs_A1_analysis_v1.json"
)

OUT_MANIFEST = (
    EXP_A /
    "manifests/A9_analysis_manifest_v1.json"
)


EXPECTED = {
    "A9_result":
        "b4b02483db7021c43e2e406d0ea4811f7f22f72258957613bbfcd96b11ec12aa",

    "A9_log":
        "07b6a12f48a03cec5948c701c5b510f52a10110e1ab6e3ac4cc67950aea84672",

    "A9_execution":
        "99f0140440e2645064fbe69bd713f35d7063f64cf6e38c0faa0810772938cffb",

    "preregistration":
        "5568bf925a7b1de1e561820673bb15d89e818d08f2c2662f4c577fb9aa626f90",

    "A8_analysis":
        "402cd349c2537c9dc6f3891e93bd5635dd71fc28dab8ab9d873ff1d538dcdd4e",
}


def sha256(path):
    h = hashlib.sha256()

    with open(path, "rb") as f:
        for block in iter(lambda: f.read(1024 * 1024), b""):
            h.update(block)

    return h.hexdigest()


def as_bool(series):
    return (
        series.astype(str)
        .str.strip()
        .str.lower()
        .map({
            "true": True,
            "false": False,
            "1": True,
            "0": False,
        })
    )


def cp_interval(k, n, alpha=0.05):

    if n == 0:
        return [None, None]

    lower = (
        0.0
        if k == 0
        else beta.ppf(
            alpha / 2,
            k,
            n - k + 1
        )
    )

    upper = (
        1.0
        if k == n
        else beta.ppf(
            1 - alpha / 2,
            k + 1,
            n - k
        )
    )

    return [
        float(lower),
        float(upper),
    ]


def descriptor_present(value):

    if pd.isna(value):
        return False

    text = str(value).strip()

    return (
        text != ""
        and text.lower()
        not in {
            "none",
            "null",
            "nan",
            "{}",
            "[]",
        }
    )


print("=" * 110)
print("STEP A37 — ANALYZE AND FREEZE A9 METRICS")
print("=" * 110)

# ------------------------------------------------------------------
# 1. Verify frozen artifacts
# ------------------------------------------------------------------

checks = [
    (
        "A9 raw result",
        A9_RESULT,
        EXPECTED["A9_result"]
    ),
    (
        "A9 execution log",
        A9_LOG,
        EXPECTED["A9_log"]
    ),
    (
        "A9 execution manifest",
        A9_EXEC,
        EXPECTED["A9_execution"]
    ),
    (
        "A7-A10 preregistration",
        PREREG,
        EXPECTED["preregistration"]
    ),
    (
        "A8 analysis manifest",
        A8_ANALYSIS,
        EXPECTED["A8_analysis"]
    ),
]

for label, path, expected_hash in checks:

    assert path.exists(), (
        f"STOP: missing {label}:\n{path}"
    )

    actual = sha256(path)

    assert actual == expected_hash, (
        f"STOP: {label} changed.\n"
        f"Expected: {expected_hash}\n"
        f"Actual:   {actual}"
    )

    print(f"✓ {label} verified")


for path in [
    OUT_METRICS,
    OUT_ANALYSIS,
    OUT_MANIFEST,
]:
    assert not path.exists(), (
        f"STOP: output already exists:\n{path}"
    )


# ------------------------------------------------------------------
# 2. Load A9
# ------------------------------------------------------------------

df = pd.read_csv(A9_RESULT)

assert len(df) == 120
assert df["id"].nunique() == 120

print("✓ 120 rows / 120 unique cases verified")


# ------------------------------------------------------------------
# 3. Ground truth / decisions
# ------------------------------------------------------------------

expected_valid = as_bool(
    df["expected_valid"]
)

accepted = as_bool(
    df["accepted"]
)

llm_invoked_mask = as_bool(
    df["llm_invoked"]
)

parse_ok = as_bool(
    df["parse_ok"]
)

assert expected_valid.notna().all()
assert accepted.notna().all()
assert llm_invoked_mask.notna().all()

valid_n = int(expected_valid.sum())
invalid_n = int((~expected_valid).sum())

assert valid_n == 58
assert invalid_n == 62

llm_invoked = int(
    llm_invoked_mask.sum()
)

assert llm_invoked == 77


# ------------------------------------------------------------------
# 4. Parse status
# ------------------------------------------------------------------

parse_success = int(
    (
        llm_invoked_mask
        & (parse_ok == True)
    ).sum()
)

parse_failure = int(
    (
        llm_invoked_mask
        & (parse_ok == False)
    ).sum()
)

parse_unknown = int(
    (
        llm_invoked_mask
        & parse_ok.isna()
    ).sum()
)


# ------------------------------------------------------------------
# 5. Confusion matrix
#
# Invalid/rejection = positive class
# ------------------------------------------------------------------

valid_accepted = int(
    (
        expected_valid
        & accepted
    ).sum()
)

valid_rejected = int(
    (
        expected_valid
        & ~accepted
    ).sum()
)

invalid_rejected = int(
    (
        ~expected_valid
        & ~accepted
    ).sum()
)

invalid_accepted = int(
    (
        ~expected_valid
        & accepted
    ).sum()
)

TP = invalid_rejected
FP = valid_rejected
FN = invalid_accepted
TN = valid_accepted

correct = TP + TN
incorrect = FP + FN

assert correct + incorrect == 120


# ------------------------------------------------------------------
# 6. Primary metrics
# ------------------------------------------------------------------

accuracy = correct / 120
VAR = valid_accepted / 58
IRR = invalid_rejected / 62

accuracy_ci = cp_interval(
    correct,
    120
)

VAR_ci = cp_interval(
    valid_accepted,
    58
)

IRR_ci = cp_interval(
    invalid_rejected,
    62
)


# ------------------------------------------------------------------
# 7. Secondary metrics
# ------------------------------------------------------------------

precision = (
    TP / (TP + FP)
    if (TP + FP)
    else None
)

recall = (
    TP / (TP + FN)
    if (TP + FN)
    else None
)

f1 = (
    2 * precision * recall /
    (precision + recall)

    if (
        precision is not None
        and recall is not None
        and (precision + recall) > 0
    )

    else None
)


# ------------------------------------------------------------------
# 8. Infrastructure checks
# ------------------------------------------------------------------

stage_norm = (
    df["rejection_stage"]
    .fillna("")
    .astype(str)
    .str.strip()
    .str.lower()
)

exception_rows = int(
    stage_norm.eq("exception").sum()
)

combined_errors = (
    df[
        [
            c for c in [
                "exception",
                "error_codes_json"
            ]
            if c in df.columns
        ]
    ]
    .fillna("")
    .astype(str)
    .agg(" ".join, axis=1)
    .str.lower()
)

provider_failure_rows = int(
    combined_errors.str.contains(
        (
            "model_not_available|"
            "non-serverless|"
            "dedicated endpoint"
        ),
        regex=True
    ).sum()
)


# ------------------------------------------------------------------
# 9. Containment
# ------------------------------------------------------------------

descriptor_mask = (
    df["descriptor_json"]
    .map(descriptor_present)
)

accepted_count = int(
    accepted.sum()
)

rejected_count = int(
    (~accepted).sum()
)

descriptor_count = int(
    descriptor_mask.sum()
)

containment_violations = int(
    (
        ~accepted
        & descriptor_mask
    ).sum()
)

accepted_without_descriptor = int(
    (
        accepted
        & ~descriptor_mask
    ).sum()
)


# ------------------------------------------------------------------
# 10. Rejection stages
# ------------------------------------------------------------------

rejection_stages = (
    df["rejection_stage"]
    .fillna("<none>")
    .value_counts()
    .to_dict()
)


# ------------------------------------------------------------------
# 11. Evaluability
# ------------------------------------------------------------------

evaluable = (
    llm_invoked == 77
    and parse_success == 77
    and parse_failure == 0
    and parse_unknown == 0
    and exception_rows == 0
    and provider_failure_rows == 0
)

status = (
    "EVALUABLE"
    if evaluable
    else "REQUIRES_DIAGNOSIS"
)


# ------------------------------------------------------------------
# 12. Frozen A1 reference metrics
# ------------------------------------------------------------------

A1_METRICS = {
    "accuracy": 101 / 120,
    "VAR": 51 / 58,
    "IRR": 50 / 62,
}

deltas = {
    "accuracy":
        accuracy - A1_METRICS["accuracy"],

    "VAR":
        VAR - A1_METRICS["VAR"],

    "IRR":
        IRR - A1_METRICS["IRR"],
}


# ------------------------------------------------------------------
# 13. Case-level comparison with A1
#
# Use official A1 result if present. We do not require a particular
# path/hash here because the original frozen metrics remain authoritative.
# ------------------------------------------------------------------

case_comparison = {
    "performed": False
}

if A1_RESULT.exists():

    a1 = pd.read_csv(A1_RESULT)

    if (
        "id" in a1.columns
        and "accepted" in a1.columns
        and len(a1) == 120
        and a1["id"].nunique() == 120
    ):

        a1_small = a1[
            ["id", "accepted"]
        ].copy()

        a1_small["A1_accepted"] = as_bool(
            a1_small["accepted"]
        )

        a9_small = df[
            ["id", "accepted", "decision_correct"]
        ].copy()

        a9_small["A9_accepted"] = as_bool(
            a9_small["accepted"]
        )

        merged = a1_small[
            ["id", "A1_accepted"]
        ].merge(
            a9_small[
                [
                    "id",
                    "A9_accepted",
                    "decision_correct"
                ]
            ],
            on="id",
            how="inner"
        )

        acceptance_differences = int(
            (
                merged["A1_accepted"]
                != merged["A9_accepted"]
            ).sum()
        )

        case_comparison = {
            "performed": True,
            "matched_cases": len(merged),
            "acceptance_decision_differences":
                acceptance_differences,
        }


# ------------------------------------------------------------------
# 14. Write metrics
# ------------------------------------------------------------------

metrics_record = {
    "artifact_id":
        "experiment-A-A9-metrics-v1",

    "created_utc":
        datetime.now(timezone.utc).isoformat(),

    "arm":
        "A9",

    "model":
        "deepseek-ai/DeepSeek-V4.1-Flash",

    "status":
        status,

    "sample": {
        "total": 120,
        "valid": 58,
        "invalid": 62,
    },

    "confusion_matrix_invalid_rejection_positive": {
        "TP_invalid_rejected": TP,
        "FP_valid_rejected": FP,
        "FN_invalid_accepted": FN,
        "TN_valid_accepted": TN,
    },

    "counts": {
        "correct": correct,
        "incorrect": incorrect,

        "valid_accepted": valid_accepted,
        "valid_rejected": valid_rejected,

        "invalid_rejected": invalid_rejected,
        "invalid_accepted": invalid_accepted,

        "accepted": accepted_count,
        "rejected": rejected_count,

        "descriptors": descriptor_count,

        "containment_violations":
            containment_violations,

        "accepted_without_descriptor":
            accepted_without_descriptor,
    },

    "metrics": {
        "accuracy": {
            "numerator": correct,
            "denominator": 120,
            "value": accuracy,
            "cp95": accuracy_ci,
        },

        "VAR": {
            "numerator": valid_accepted,
            "denominator": 58,
            "value": VAR,
            "cp95": VAR_ci,
        },

        "IRR": {
            "numerator": invalid_rejected,
            "denominator": 62,
            "value": IRR,
            "cp95": IRR_ci,
        },

        "rejection_precision":
            precision,

        "rejection_recall":
            recall,

        "rejection_f1":
            f1,
    },

    "llm_execution": {
        "invoked": llm_invoked,
        "parse_success": parse_success,
        "parse_failure": parse_failure,
        "parse_unknown": parse_unknown,
        "exception_rows": exception_rows,

        "provider_endpoint_failure_rows":
            provider_failure_rows,
    },

    "containment": {
        "definition":
            "accepted == False AND descriptor_json present",

        "violations":
            containment_violations,

        "accepted_without_descriptor":
            accepted_without_descriptor,
    },

    "rejection_stages":
        rejection_stages,

    "case_level_A1_comparison":
        case_comparison,

    "source_result": {
        "path":
            str(A9_RESULT.relative_to(ROOT)),

        "sha256":
            sha256(A9_RESULT),
    },
}

OUT_METRICS.write_text(
    json.dumps(
        metrics_record,
        indent=2
    ) + "\n",
    encoding="utf-8"
)


# ------------------------------------------------------------------
# 15. A9 vs A1 comparison
# ------------------------------------------------------------------

comparison_record = {
    "artifact_id":
        "experiment-A-A9-vs-A1-analysis-v1",

    "created_utc":
        datetime.now(timezone.utc).isoformat(),

    "A1": {
        "model":
            "Qwen/Qwen3.5-9B",

        "accuracy":
            A1_METRICS["accuracy"],

        "VAR":
            A1_METRICS["VAR"],

        "IRR":
            A1_METRICS["IRR"],

        "containment_violations":
            0,
    },

    "A9": {
        "model":
            "deepseek-ai/DeepSeek-V4.1-Flash",

        "accuracy":
            accuracy,

        "VAR":
            VAR,

        "IRR":
            IRR,

        "containment_violations":
            containment_violations,
    },

    "A9_minus_A1": {
        "accuracy":
            deltas["accuracy"],

        "VAR":
            deltas["VAR"],

        "IRR":
            deltas["IRR"],
    },

    "case_level_comparison":
        case_comparison,

    "interpretation_policy": (
        "Matching aggregate counts do not establish matching "
        "case-level behavior. Case-level acceptance decisions are "
        "compared separately when the frozen A1 result is available."
    ),
}

OUT_ANALYSIS.write_text(
    json.dumps(
        comparison_record,
        indent=2
    ) + "\n",
    encoding="utf-8"
)


# ------------------------------------------------------------------
# 16. Freeze analysis manifest
# ------------------------------------------------------------------

manifest = {
    "manifest_id":
        "experiment-A-A9-analysis-manifest-v1",

    "created_utc":
        datetime.now(timezone.utc).isoformat(),

    "arm":
        "A9",

    "status":
        status,

    "raw_result": {
        "path":
            str(A9_RESULT.relative_to(ROOT)),

        "sha256":
            sha256(A9_RESULT),
    },

    "metrics": {
        "path":
            str(OUT_METRICS.relative_to(ROOT)),

        "sha256":
            sha256(OUT_METRICS),
    },

    "comparison": {
        "path":
            str(OUT_ANALYSIS.relative_to(ROOT)),

        "sha256":
            sha256(OUT_ANALYSIS),
    },

    "no_model_call":
        True,

    "raw_result_modified":
        False,
}

OUT_MANIFEST.write_text(
    json.dumps(
        manifest,
        indent=2
    ) + "\n",
    encoding="utf-8"
)


# ------------------------------------------------------------------
# 17. Report
# ------------------------------------------------------------------

print("\n" + "=" * 110)
print("A9 EVALUABILITY")
print("=" * 110)

print("Status                         :", status)
print("LLM invoked                    :", llm_invoked)
print("Parse success                  :", parse_success)
print("Parse failure                  :", parse_failure)
print("Parse unknown                  :", parse_unknown)
print("Exception rows                 :", exception_rows)
print("Provider endpoint failure rows :", provider_failure_rows)


print("\n" + "=" * 110)
print("A9 PRIMARY RESULTS")
print("=" * 110)

print(
    f"Accuracy : {correct}/120 = "
    f"{accuracy * 100:.2f}% "
    f"[{accuracy_ci[0] * 100:.2f}, "
    f"{accuracy_ci[1] * 100:.2f}]"
)

print(
    f"VAR      : {valid_accepted}/58 = "
    f"{VAR * 100:.2f}% "
    f"[{VAR_ci[0] * 100:.2f}, "
    f"{VAR_ci[1] * 100:.2f}]"
)

print(
    f"IRR      : {invalid_rejected}/62 = "
    f"{IRR * 100:.2f}% "
    f"[{IRR_ci[0] * 100:.2f}, "
    f"{IRR_ci[1] * 100:.2f}]"
)

print("\nConfusion matrix:")
print("  TP invalid rejected :", TP)
print("  FP valid rejected   :", FP)
print("  FN invalid accepted :", FN)
print("  TN valid accepted   :", TN)

print("\nSecondary:")
print(
    "  Rejection precision:",
    None if precision is None
    else f"{precision:.6f}"
)

print(
    "  Rejection recall   :",
    None if recall is None
    else f"{recall:.6f}"
)

print(
    "  Rejection F1       :",
    None if f1 is None
    else f"{f1:.6f}"
)


print("\n" + "=" * 110)
print("A9 CONTAINMENT")
print("=" * 110)

print("Accepted                    :", accepted_count)
print("Rejected                    :", rejected_count)
print("Descriptors                 :", descriptor_count)
print("Containment violations      :", containment_violations)
print("Accepted without descriptor :", accepted_without_descriptor)


print("\n" + "=" * 110)
print("A9 vs A1")
print("=" * 110)

print(
    f"Accuracy delta : "
    f"{deltas['accuracy'] * 100:+.2f} percentage points"
)

print(
    f"VAR delta      : "
    f"{deltas['VAR'] * 100:+.2f} percentage points"
)

print(
    f"IRR delta      : "
    f"{deltas['IRR'] * 100:+.2f} percentage points"
)

print("\nCase-level comparison:")
print(
    json.dumps(
        case_comparison,
        indent=2
    )
)

print("\nRejection stages:")

for stage, count in rejection_stages.items():
    print(f"  {stage}: {count}")


print("\n" + "-" * 110)
print("FROZEN ANALYSIS ARTIFACTS")
print("-" * 110)

print("Metrics:")
print(OUT_METRICS)
print("SHA-256:")
print(sha256(OUT_METRICS))

print("\nA9 vs A1:")
print(OUT_ANALYSIS)
print("SHA-256:")
print(sha256(OUT_ANALYSIS))

print("\nAnalysis manifest:")
print(OUT_MANIFEST)
print("SHA-256:")
print(sha256(OUT_MANIFEST))


print("\n" + "=" * 110)

if status == "EVALUABLE":

    print("✓ A9 CLASSIFIED AS EVALUABLE")
    print("✓ ALL 77 LLM INVOCATIONS PARSED SUCCESSFULLY")
    print("✓ NO PROVIDER ENDPOINT FAILURE DETECTED")
    print("✓ PRIMARY METRICS FROZEN")
    print("✓ EXACT 95% CLOPPER-PEARSON INTERVALS FROZEN")
    print("✓ CONTAINMENT RESULT FROZEN")
    print("✓ A9 vs A1 COMPARISON FROZEN")
    print("✓ NO MODEL/API CALL MADE")
    print("✓ RAW RESULT NOT MODIFIED")
    print("✓ DO NOT RUN A10 YET")
    print("✓ SEND THIS OUTPUT BEFORE CONTINUING")

else:

    print("✗ A9 REQUIRES DIAGNOSIS")
    print("✗ DO NOT INTERPRET PERFORMANCE YET")
    print("✗ DO NOT RUN A10")
    print("✓ SEND THIS OUTPUT FOR DIAGNOSIS")

print("=" * 110)

STEP A37 — ANALYZE AND FREEZE A9 METRICS
✓ A9 raw result verified
✓ A9 execution log verified
✓ A9 execution manifest verified
✓ A7-A10 preregistration verified
✓ A8 analysis manifest verified
✓ 120 rows / 120 unique cases verified

A9 EVALUABILITY
Status                         : EVALUABLE
LLM invoked                    : 77
Parse success                  : 77
Parse failure                  : 0
Parse unknown                  : 0
Exception rows                 : 0
Provider endpoint failure rows : 0

A9 PRIMARY RESULTS
Accuracy : 101/120 = 84.17% [76.38, 90.19]
VAR      : 51/58 = 87.93% [76.70, 95.01]
IRR      : 50/62 = 80.65% [68.63, 89.58]

Confusion matrix:
  TP invalid rejected : 50
  FP valid rejected   : 7
  FN invalid accepted : 12
  TN valid accepted   : 51

Secondary:
  Rejection precision: 0.877193
  Rejection recall   : 0.806452
  Rejection F1       : 0.840336

A9 CONTAINMENT
Accepted                    : 63
Rejected                    : 57
Descriptors                 : 63
Co

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import subprocess
import hashlib
import os
import json
import pandas as pd
import time

ROOT = Path("/content/deterministic-intent-translation-journal")
EXP_A = ROOT / "journal_experiments/experiment_A_model_sweep"

RUNNER = ROOT / (
    "experiment-2b/implementation/e3_v3/"
    "run_experiment2b_e3v3_r7_dev_v1.py"
)

DATASET = ROOT / (
    "experiment-2b/reproducibility/"
    "e3v3_heldout_benchmark_v1.0/"
    "e3v3_heldout_benchmark_120_v1.csv"
)

PREREG = (
    EXP_A /
    "preregistration/"
    "experiment_A_extension_A7_A10_preregistration_v1.json"
)

A9_ANALYSIS = (
    EXP_A /
    "manifests/A9_analysis_manifest_v1.json"
)

RESULT = (
    EXP_A /
    "results/raw/"
    "A10_gpt-oss-120b_results.csv"
)

LOG = (
    EXP_A /
    "logs/"
    "A10_gpt-oss-120b_execution.log"
)

EXEC_MANIFEST = (
    EXP_A /
    "manifests/"
    "A10_execution_manifest_v1.json"
)

MODEL = "openai/gpt-oss-120b"

EXPECTED = {
    "runner":
        "40c035cb864978c718a72fcfdfbb164d262529960b0ae78242d5194147cd8257",

    "dataset":
        "84576e7d99a2721e61ded5a14bdafeb96707b068397243bbed32aa735496d4de",

    "preregistration":
        "5568bf925a7b1de1e561820673bb15d89e818d08f2c2662f4c577fb9aa626f90",

    "A9_analysis":
        "f115a5520362b11119057bbcc31a3b026d75afdf416d44514210235e0307f067",
}


def sha256(path):
    h = hashlib.sha256()

    with open(path, "rb") as f:
        for block in iter(lambda: f.read(1024 * 1024), b""):
            h.update(block)

    return h.hexdigest()


def bool_count(df, column, value=True):

    if column not in df.columns:
        return None

    s = (
        df[column]
        .astype(str)
        .str.strip()
        .str.lower()
    )

    target = "true" if value else "false"

    return int((s == target).sum())


print("=" * 110)
print("STEP A38 — EXECUTE A10 ON FROZEN 120-CASE BENCHMARK")
print("=" * 110)

# ------------------------------------------------------------------
# 1. Verify frozen artifacts
# ------------------------------------------------------------------

checks = [
    (
        "Frozen runner",
        RUNNER,
        EXPECTED["runner"]
    ),
    (
        "Frozen benchmark",
        DATASET,
        EXPECTED["dataset"]
    ),
    (
        "A7-A10 preregistration",
        PREREG,
        EXPECTED["preregistration"]
    ),
    (
        "Frozen A9 analysis manifest",
        A9_ANALYSIS,
        EXPECTED["A9_analysis"]
    ),
]

for label, path, expected_hash in checks:

    assert path.exists(), (
        f"STOP: missing required artifact:\n{path}"
    )

    actual = sha256(path)

    assert actual == expected_hash, (
        f"STOP: {label} changed.\n"
        f"Expected: {expected_hash}\n"
        f"Actual:   {actual}"
    )

    print(f"✓ {label} verified")


# ------------------------------------------------------------------
# 2. Verify A10 preregistration
# ------------------------------------------------------------------

prereg = json.loads(
    PREREG.read_text(encoding="utf-8")
)

arm_map = {
    arm["arm"]: arm["model"]
    for arm in prereg["arms"]
}

assert arm_map["A10"] == MODEL, (
    "STOP: A10 model does not match preregistration."
)

assert prereg["execution_order"] == [
    "A7",
    "A8",
    "A9",
    "A10",
], "STOP: frozen execution order changed."

print("✓ A10 exact model ID verified")
print("✓ Frozen execution order verified")


# ------------------------------------------------------------------
# 3. Safety checks
# ------------------------------------------------------------------

assert os.environ.get("TOGETHER_API_KEY"), (
    "STOP: TOGETHER_API_KEY missing."
)

assert not RESULT.exists(), (
    f"STOP: A10 result already exists:\n{RESULT}\n"
    "Do not rerun A10."
)

assert not LOG.exists(), (
    f"STOP: A10 log already exists:\n{LOG}\n"
    "Do not rerun A10."
)

assert not EXEC_MANIFEST.exists(), (
    f"STOP: A10 execution manifest already exists:\n"
    f"{EXEC_MANIFEST}"
)

RESULT.parent.mkdir(
    parents=True,
    exist_ok=True
)

LOG.parent.mkdir(
    parents=True,
    exist_ok=True
)

print("✓ API key present")
print("✓ A10 output paths clean")


# ------------------------------------------------------------------
# 4. Exact command
# ------------------------------------------------------------------

cmd = [
    "python",
    str(RUNNER),

    "--dataset",
    str(DATASET),

    "--out",
    str(RESULT),

    "--model",
    MODEL,
]

print("\n" + "-" * 110)
print("A10 FROZEN EXECUTION")
print("-" * 110)

print("Arm   : A10")
print("Model :", MODEL)

print("\nCommand:")
print(" ".join(cmd))

print("\nPolicy:")
print("  ✓ frozen 120-case benchmark")
print("  ✓ no --limit")
print("  ✓ only semantic-parser model changes")
print("  ✓ no prompt/gate/threshold changes")
print("  ✓ no automatic rerun")


# ------------------------------------------------------------------
# 5. Execute exactly once
# ------------------------------------------------------------------

started_utc = datetime.now(
    timezone.utc
).isoformat()

start = time.time()

proc = subprocess.run(
    cmd,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    env=os.environ.copy(),
)

elapsed = time.time() - start

finished_utc = datetime.now(
    timezone.utc
).isoformat()

LOG.write_text(
    proc.stdout or "",
    encoding="utf-8"
)

print("\n" + "-" * 110)
print("RUNNER OUTPUT")
print("-" * 110)

print(proc.stdout)

print(
    "\nProcess return code :",
    proc.returncode
)

print(
    "Wall-clock seconds :",
    round(elapsed, 2)
)


# ------------------------------------------------------------------
# 6. Never automatically rerun
# ------------------------------------------------------------------

if proc.returncode != 0:

    print("\n" + "=" * 110)
    print("✗ A10 PROCESS RETURNED NON-ZERO")
    print("✗ DO NOT RERUN")
    print("✗ SEND OUTPUT FOR DIAGNOSIS")
    print("=" * 110)

    raise SystemExit


# ------------------------------------------------------------------
# 7. Verify result
# ------------------------------------------------------------------

assert RESULT.exists(), (
    "STOP: A10 result CSV missing."
)

df = pd.read_csv(RESULT)

assert len(df) == 120, (
    f"STOP: expected 120 rows; found {len(df)}."
)

assert "id" in df.columns, (
    "STOP: id column missing."
)

unique_cases = df["id"].nunique()

assert unique_cases == 120, (
    f"STOP: expected 120 unique cases; "
    f"found {unique_cases}."
)


# ------------------------------------------------------------------
# 8. Lightweight diagnostics
# ------------------------------------------------------------------

llm_invoked = bool_count(
    df,
    "llm_invoked",
    True
)

parse_ok = bool_count(
    df,
    "parse_ok",
    True
)

parse_failed = bool_count(
    df,
    "parse_ok",
    False
)

accepted = bool_count(
    df,
    "accepted",
    True
)

exception_rows = 0

if "rejection_stage" in df.columns:

    stage_norm = (
        df["rejection_stage"]
        .fillna("")
        .astype(str)
        .str.strip()
        .str.lower()
    )

    exception_rows = int(
        stage_norm.eq("exception").sum()
    )

    rejection_stages = (
        df["rejection_stage"]
        .fillna("<none>")
        .value_counts()
        .to_dict()
    )

else:

    rejection_stages = {}


# ------------------------------------------------------------------
# 9. Freeze execution manifest
# ------------------------------------------------------------------

execution_record = {
    "manifest_id":
        "experiment-A-A10-execution-v1",

    "created_utc":
        datetime.now(timezone.utc).isoformat(),

    "arm":
        "A10",

    "model":
        MODEL,

    "started_utc":
        started_utc,

    "finished_utc":
        finished_utc,

    "wall_clock_seconds":
        elapsed,

    "return_code":
        proc.returncode,

    "command":
        cmd,

    "frozen_inputs": {
        "runner": {
            "path":
                str(RUNNER.relative_to(ROOT)),

            "sha256":
                sha256(RUNNER),
        },

        "benchmark": {
            "path":
                str(DATASET.relative_to(ROOT)),

            "sha256":
                sha256(DATASET),
        },

        "preregistration": {
            "path":
                str(PREREG.relative_to(ROOT)),

            "sha256":
                sha256(PREREG),
        },

        "prior_A9_analysis_manifest": {
            "path":
                str(A9_ANALYSIS.relative_to(ROOT)),

            "sha256":
                sha256(A9_ANALYSIS),
        },
    },

    "outputs": {
        "result": {
            "path":
                str(RESULT.relative_to(ROOT)),

            "sha256":
                sha256(RESULT),

            "rows":
                len(df),

            "unique_cases":
                unique_cases,
        },

        "log": {
            "path":
                str(LOG.relative_to(ROOT)),

            "sha256":
                sha256(LOG),
        },
    },

    "lightweight_diagnostics": {
        "llm_invoked":
            llm_invoked,

        "parse_ok":
            parse_ok,

        "parse_failed":
            parse_failed,

        "accepted":
            accepted,

        "exception_rows":
            exception_rows,

        "rejection_stages":
            rejection_stages,
    },

    "performance_metrics_calculated":
        False,

    "automatic_rerun":
        False,
}

EXEC_MANIFEST.write_text(
    json.dumps(
        execution_record,
        indent=2
    ) + "\n",
    encoding="utf-8"
)


# ------------------------------------------------------------------
# 10. Report
# ------------------------------------------------------------------

print("\n" + "-" * 110)
print("A10 RAW EXECUTION ARTIFACTS")
print("-" * 110)

print("Result:")
print(RESULT)

print("Rows:", len(df))
print("Unique cases:", unique_cases)

print("SHA-256:")
print(sha256(RESULT))

print("\nLog:")
print(LOG)

print("SHA-256:")
print(sha256(LOG))

print("\nExecution manifest:")
print(EXEC_MANIFEST)

print("SHA-256:")
print(sha256(EXEC_MANIFEST))


print("\n" + "-" * 110)
print("LIGHTWEIGHT DIAGNOSTIC")
print("-" * 110)

print("LLM invoked   :", llm_invoked)
print("Parse OK      :", parse_ok)
print("Parse failed  :", parse_failed)
print("Accepted      :", accepted)
print("Exception rows:", exception_rows)

print("\nRejection stages:")

for stage, count in rejection_stages.items():
    print(f"  {stage}: {count}")


print("\n" + "=" * 110)
print("✓ A10 EXECUTED EXACTLY ONCE")
print("✓ 120 RESULT ROWS VERIFIED")
print("✓ RAW RESULT FROZEN")
print("✓ EXECUTION LOG FROZEN")
print("✓ NO PERFORMANCE METRICS CALCULATED YET")
print("✓ NO AUTOMATIC RERUN")
print("✓ EXPERIMENT A MODEL EXECUTION PHASE COMPLETE")
print("✓ SEND THIS OUTPUT FOR A10 ANALYSIS")
print("=" * 110)